# Globus Pallidus T1-Weighted MRI Radiomics for Hepatic Encephalopathy Grading

[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)  
[![TRIPOD-AI](https://img.shields.io/badge/Reporting-TRIPOD--AI-blue)](https://www.tripod-statement.org/)  
[![IBSI Compliant](https://img.shields.io/badge/Radiomics-IBSI%20Compliant-green)](https://ibsi.readthedocs.io/)  
[![Python 3.9+](https://img.shields.io/badge/Python-3.9%2B-blue)](https://www.python.org/)  

---

## Overview

This notebook implements the complete, reproducible statistical and machine learning pipeline
for non-invasive grading of hepatic encephalopathy (HE) severity in cirrhotic patients
using T1-weighted unenhanced MRI radiomic features of the **globus pallidus**.

The biological hypothesis centres on a combination of pathological processes affecting
the globus pallidus in cirrhosis — primarily manganese deposition via porto-systemic
shunting, but also degenerative phenomena including astrocytic oedema and gliosis —
as the collective substrate for the radiomic signal. Three binary classification
outcomes are addressed:

| Outcome | Positive class | Negative class | WHC grades | N | Role |
|---------|---------------|----------------|------------|---|------|
| **O0** | Cirrhosis No HE | Healthy Controls | — | 148 | Biological signal specificity |
| **O1** | Mild HE | No overt HE | WHC 1 vs WHC 0 | 123 | **Primary** |
| **O2** | Moderate-to-Severe HE | Mild HE | WHC ≥ 2 vs WHC 1 | 85 | Exploratory (EPV < 10) |

**Label conversion (WHC → binary):**

| WHC grade | Clinical description | O1 label | O2 label |
|-----------|---------------------|----------|----------|
| 0 | No overt HE | 0 (negative) | — (excluded) |
| 1 | Mild (covert) HE | 1 (positive) | 0 (negative) |
| 2 | Moderate HE | — (excluded) | 1 (positive) |
| 3 | Severe HE | — (excluded) | 1 (positive) |

**O0** establishes that radiomic differences reflect the structural changes
of the globus pallidus in cirrhosis (manganese deposition, oedema, gliosis)
rather than HE-specific neurological changes per se.

**O1** is the primary outcome: detection of mild (covert) HE using globus pallidus
radiomics as a non-invasive pre-transplant stratification tool.

**O2** is exploratory due to EPV < 10 and zero bootstrap feature stability;
results are hypothesis-generating only.

---

## Repository

| Resource | Link |
|----------|------|
| GitHub | `https://github.com/radiomiclab/radiomic_HE_pipeline` |
| Zenodo DOI | `https://doi.org/10.5281/zenodo.[ID]` *(assigned upon dataset deposition)* |

---

## Reporting Compliance

| Standard | Status |
|----------|--------|
| **TRIPOD-AI** (Collins et al., *BMJ* 2024) | ✅ Items documented per cell header |
| **IBSI** (Zwanenburg et al., *Radiology* 2020) | ✅ PyRadiomics v3.0.1, prefix `original_`, bin-width 25 HU, 3D |
| **PEP 8** | ✅ snake_case, 79-char lines, Google-style docstrings |
| **Open science** | ✅ requirements.txt, config.yaml, README.md generated in Cell 21 |

---

## Cell Structure

| Cell | Content | TRIPOD-AI |
|------|---------|-----------|
| 0 | Environment setup (pycombat==0.20) | — |
| 1 | Imports, seeds, constants, helpers | 15 |
| 2 | Table 1: baseline characteristics (inline) | — |
| 3 | Data loading: internal cohort | 4b, 5c, 7a, 15 |
| 4 | Data loading: healthy controls (O0) | 4b, 7a |
| 5 | EDA: cohort characterisation + age–feature correlation (inline) | 4b, 5c, 15 |
| 6 | Subset creation O0, O1, O2 | 5b, 5c, 8a |
| 7 | Nested CV + ComBat + EPV fix (all outcomes) | 9, 10a, 10b, 15 |
| 8 | Performance metrics + DeLong + calibration (inline) | 10a, 11, 12 |
| 9 | Calibration reliability diagrams (inline) | 10a, 12 |
| 10 | Decision Curve Analysis — Fig 4 (inline) | 13a |
| 11 | SHAP feature importance — Fig 3 (inline) | 10c, 13a |
| 12 | Bootstrap LASSO stability (inline) | 10c, 15 |
| 13 | AUC grouped bar chart — Fig 2 (inline) | 11 |
| 14 | Cross-system validation (inline) | 10d, 11, 12 |
| 15 | External validation (Ext-A, Ext-B) (inline) | 10e, 11, 12 |
| 16 | Domain shift analysis (inline) | 10e, 12 |
| 17 | ComBat diagnostics (inline) | 9, 15 |
| 18 | RadScore equations (inline) | 10c, 15 |
| 19 | Supplementary Table ST1 (inline) | 15 |
| 20 | Reproducibility check + session info | 15 |
| 21 | Repository files (requirements.txt, config.yaml, README.md) | 15 |

---

## ▶ Before You Start — Important Note on Feature Extraction

**Radiomic feature extraction and ROI segmentation are not part of this pipeline.**  
Each research group uses its own MRI acquisition protocol, segmentation method,
and extraction software. This pipeline assumes that:

1. Bilateral globus pallidus ROIs have been segmented on T1-weighted unenhanced
   MRI images following the operator's own validated procedure.
2. The same **30 IBSI-compliant radiomic features** used in the original study
   have been extracted from those ROIs using an IBSI-compliant tool
   (e.g., PyRadiomics v3.0.1, bin-width 25 HU, 3D extraction, `original_` prefix).
3. The extracted features are stored as CSV files with the exact column names
   documented in `dataset_dictionary.ipynb`.

Feature names, radiomics classes, IBSI codes, and the expected column schema
are fully documented in `dataset_dictionary.ipynb`.
**Using a different feature set or extraction configuration will invalidate
the pre-trained model coefficients and the reproducibility of results.**

---

## ▶ How to Run the Pipeline — 3 Steps

**Step 1 — Create the empty dataset files**  
Open and run `dataset_dictionary.ipynb`. This creates the `./data/`
subdirectory and generates five empty CSV templates with the correct 39-column
header:

```
data/
├── radiomic_HE_dataset.csv          ← internal cohort (development + internal validation)
├── radiomic_HE_dataset_extA.csv     ← external validation cohort A
├── radiomic_HE_dataset_extB.csv     ← external validation cohort B
├── healthy_discovery.csv            ← healthy controls (Discovery scanner)
└── healthy_signa.csv                ← healthy controls (Signa scanner)
```

**Step 2 — Populate the datasets with your own data**  
Open each CSV file in `./data/` with your preferred tool (Excel, LibreOffice,
Python, R, etc.) and fill in your patient/subject records row by row.
Follow the column schema documented in `dataset_dictionary.ipynb`:
data types, allowed categorical values, and the IBSI feature name convention
(`original_{class}_{FeatureName}`) must be respected exactly.
Do **not** rename, reorder, or add columns.  
Once populated, run the validation cell in `dataset_dictionary.ipynb` to verify the schema.

**Step 3 — Run this pipeline**  
Install dependencies (once), then execute all cells in order (Cell 0 → Cell 21):

```bash
# Install dependencies (once, from the repository root)
pip install -r requirements.txt

# Launch the notebook
jupyter notebook radiomic_pipeline_v1.ipynb
```

All paths are resolved relative to the notebook's working directory using
`pathlib.Path` — no absolute paths, no OS-specific separators.
The pipeline runs identically on **Windows, macOS, and Linux**.
All figures and statistical tables are displayed **inline** in the notebook;
no external application needs to be opened to inspect results.

**Repository support files**  

| File | Purpose |
|------|---------|
| `requirements.txt` | Pinned Python dependencies (auto-detected from the current environment) |
| `config.yaml` | Human-readable record of all pipeline hyperparameters, scanner IDs, outcome definitions, and paths |
| `README.md` | Repository documentation with study design, compliance standards, and usage instructions |

> **Note on `config.yaml`:** this file is a documentation artefact for
> reviewers and collaborators — it is **not** read programmatically by the
> pipeline. All parameters are defined directly in Cell 1 of the pipeline
> notebook. 

---

*Pipeline version: 1.0 | Python ≥ 3.9 | PyRadiomics 3.0.1 | pycombat 0.20*

In [ ]:
# =============================================================================
# Cell 0 -- Environment setup
# =============================================================================
# Install and verify pycombat==0.20. Run once before any other cell.
# =============================================================================
# Install and verify pinned dependencies (pycombat==0.20).
# Run once before executing any other cell.
# =============================================================================
# Installs and verifies all non-standard dependencies required by this
# pipeline. Run once before executing any other cell.
#
# Dependencies managed here:
#   pycombat 0.20  — ComBat harmonisation for radiomic batch-effect removal
#                    (Behdenna et al., 2020; https://github.com/Jfortin1/ComBatHarmonization)
#
# All other dependencies (numpy, pandas, scikit-learn, scipy, matplotlib,
# shap, python-docx) are assumed present in the base environment.
# Pin them in requirements.txt for full reproducibility.
#
# Python >= 3.9 required.
# Tested on Python 3.10.19.
# =============================================================================

import subprocess
import sys
import importlib

# ---------------------------------------------------------------------------
# Pinned dependency manifest
# key   : importable module name
# value : (pip install spec, minimum version string for verification)
# ---------------------------------------------------------------------------
DEPENDENCIES = {
    "pycombat": ("pycombat==0.20", "0.20"),
}

def _install_and_verify(module_name, pip_spec, expected_version):
    """
    Install pip_spec if module_name is not importable, then verify version.
    Returns True if the dependency is ready, False otherwise.
    """
    # --- attempt import ---
    try:
        mod = importlib.import_module(module_name)
        installed_version = getattr(mod, "__version__", "unknown")
        if installed_version != expected_version and expected_version != "any":
            print(f"  [WARN] {module_name} {installed_version} found; "
                  f"expected {expected_version}. Re-installing pinned version.")
            raise ImportError("version mismatch")
        print(f"  [OK]   {module_name} {installed_version}")
        return True
    except ImportError:
        pass

    # --- install ---
    print(f"  [INSTALL] {pip_spec} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pip_spec,
         "--quiet", "--disable-pip-version-check"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"  [ERROR] Installation failed for {pip_spec}:\n{result.stderr}")
        return False

    # --- re-verify after install ---
    importlib.invalidate_caches()
    try:
        mod = importlib.import_module(module_name)
        installed_version = getattr(mod, "__version__", "unknown")
        print(f"  [OK]   {module_name} {installed_version} (just installed)")
        return True
    except ImportError as e:
        print(f"  [ERROR] Import failed after installation: {e}")
        return False


# ---------------------------------------------------------------------------
# Run checks
# ---------------------------------------------------------------------------
print("=" * 60)
print("Cell 0 — Dependency check")
print("=" * 60)

all_ready = True
for module_name, (pip_spec, min_ver) in DEPENDENCIES.items():
    ok = _install_and_verify(module_name, pip_spec, min_ver)
    if not ok:
        all_ready = False

print()
if all_ready:
    print("All dependencies satisfied. You may now run Cell 1.")
else:
    print("One or more dependencies could not be installed.")
    print("Resolve the errors above before proceeding.")
    raise EnvironmentError("Dependency check failed — see output above.")

In [ ]:
# =============================================================================
# Cell 1 -- Setup: imports, seeds, constants, CSV_DIR, helpers
# TRIPOD-AI items: 15
# =============================================================================
# TRIPOD-AI pipeline for MRI radiomics-based HE stratification.
# Outcomes: O0 (biological specificity), O1 (primary), O2 (exploratory).
# =============================================================================
# TRIPOD-AI pipeline for MRI radiomics-based HE stratification.
# Outcome 1: Mild HE vs No HE  |  Outcome 2: Mod-Severe HE vs Mild HE
# =============================================================================
#           figure save helper, table export helper
# =============================================================================
# TRIPOD-AI-compliant pipeline for MRI radiomics-based stratification of
# hepatic encephalopathy severity using radiomic features and MELD score.
#
# Outcomes:
#   Outcome 1 — Mild HE vs No HE              (GRADE_HE: 1 vs 0)
#   Outcome 2 — Moderate-to-Severe vs Mild HE  (GRADE_HE: >=2 vs 1)
#
# Authors  : [Authors]
# Journal  : [Journal]
# Python   : >= 3.9  (tested on 3.10.19)
# License  : MIT
# =============================================================================

# =============================================================================
# STEP 1 — Cross-platform reproducibility seeds
# Must be set before ANY library that draws from a random source is imported.
#
# Three independent random sources are fixed:
#   PYTHONHASHSEED : controls dict/set ordering in CPython
#   random         : Python standard library, used transitively by joblib
#   numpy          : primary source for all numerical stochasticity
#
# All stochastic sklearn calls receive random_state=RANDOM_STATE explicitly.
# All bootstrap/permutation loops use np.random.default_rng(BOOT_SEED).
# Results are identical after any kernel restart provided Cell 1 runs first.
# =============================================================================
import os
import random

RANDOM_STATE = 42
BOOT_SEED    = 42

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)

import numpy as np
np.random.seed(RANDOM_STATE)

# =============================================================================
# STEP 2 — Matplotlib backend
# "Agg" is a non-interactive, file-only rasteriser:
#   - Required for TIFF/PDF export at 300 dpi
#   - Works on headless Linux servers (no $DISPLAY required)
#   - Avoids Tk/Qt conflicts on macOS conda environments
# Must be set before the first import of matplotlib.pyplot.
#
# Inline display in Jupyter is handled explicitly by save_figure() using
# IPython.display.Image — works correctly with Agg on all platforms.
# =============================================================================
import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
from matplotlib.lines     import Line2D

matplotlib.rcParams.update({
    "font.family":     "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    # DejaVu Sans: built-in fallback, identical layout on Linux without Arial
    "pdf.fonttype":    42,    # Type 2 fonts — editable in Illustrator
    "font.size":       11,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.linewidth":    1.0,
})

# =============================================================================
# STEP 3 — Standard library
# =============================================================================
import io
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# =============================================================================
# STEP 4 — Scientific stack
# =============================================================================
import pandas as pd
import scipy.stats as stats
from scipy.stats import gaussian_kde
from PIL         import Image

# =============================================================================
# STEP 5 — Machine learning
# =============================================================================
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics         import roc_auc_score, brier_score_loss
from sklearn.calibration     import CalibratedClassifierCV
from sklearn.utils           import resample

# =============================================================================
# STEP 6 — ComBat harmonisation
# Imported here so a missing pycombat installation fails immediately with a
# clear ImportError rather than silently at Cell 4.
# =============================================================================
from pycombat import Combat

# =============================================================================
# STEP 7 — Explainability
# =============================================================================
import shap

# =============================================================================
# STEP 8 — Word document export
# =============================================================================
from docx              import Document
from docx.shared       import Pt, Cm
from docx.enum.section import WD_ORIENT
from docx.enum.text    import WD_ALIGN_PARAGRAPH
from docx.oxml.ns      import qn

# =============================================================================
# STEP 9 — Reporting and inline display
# =============================================================================
from IPython.display      import display as ipy_display
from IPython.display      import HTML
from IPython.core.display import Image as IPyImage

# =============================================================================
# PROJECT PATHS
# All relative — no absolute paths — works on any OS and machine.
# =============================================================================
ROOT     = Path(".")
DATA_DIR = ROOT / "data"
OUT_DIR  = ROOT / "outputs"
FIG_DIR  = OUT_DIR / "figures"
TAB_DIR  = OUT_DIR / "tables"
CSV_DIR  = OUT_DIR / "csv"
NOM_DIR  = FIG_DIR / "nomograms"
SHA_DIR  = FIG_DIR / "shap"
CAL_DIR  = FIG_DIR / "calibration"
DCA_DIR  = FIG_DIR / "dca"
BAR_DIR  = FIG_DIR / "barcharts"
CRS_DIR  = FIG_DIR / "crosssystem"
EXT_DIR  = FIG_DIR / "external"

for _dir in [FIG_DIR, TAB_DIR, CSV_DIR, NOM_DIR, SHA_DIR,
             CAL_DIR, DCA_DIR, BAR_DIR, CRS_DIR, EXT_DIR]:
    _dir.mkdir(parents=True, exist_ok=True)

RAW_CSV               = DATA_DIR / "radiomic_HE_dataset.csv"
HEALTHY_SIGNA_CSV     = DATA_DIR / "healthy_signa.csv"
HEALTHY_DISCOVERY_CSV = DATA_DIR / "healthy_discovery.csv"
EXT_A_CSV             = DATA_DIR / "radiomic_HE_dataset_extA.csv"
EXT_B_CSV             = DATA_DIR / "radiomic_HE_dataset_extB.csv"

# =============================================================================
# MODEL HYPERPARAMETERS
# =============================================================================
CV_FOLDS = 5
C_GRID   = {"clf__C": np.logspace(-3, 0, 10)}
N_BOOT   = 2000
N_PERM   = 2000

# =============================================================================
# COLUMN NAME CONSTANTS
# =============================================================================
COL_MR_SYSTEM = "MR_System"
COL_MELD      = "MELD"
COL_GRADE     = "GRADE_HE"
COL_HE_NONE   = "HE_None"
COL_HE_MILD   = "HE_Mild"
COL_HE_MODSEV = "HE_ModerateSevere"

# =============================================================================
# COLOR PALETTES
# =============================================================================
# Standard palette — Okabe-Ito (all figures except DCA):
#   Certified safe for deuteranopia, protanopia, tritanopia.
#   Reference: Okabe & Ito (2008), jfly.uni-koeln.de/color
#   Used for: ROC curves, calibration plots, AUC bar chart,
#             cross-system validation, external validation, nomograms, SHAP.
#
# DCA palette — Blue / Red / Green (Cell 7 only):
#   Radiomics replaced with IBM colorblind-safe red (#CC3311) because
#   Okabe orange and vermillion are visually similar on overlapping DCA lines.
#   Linestyle redundancy (MODEL_LINESTYLES) provides a second discriminant cue.
#
# Grayscale palette — luminance-separated fills (pairwise DeltaL >= 0.33):
#   MELD      #1A1A1A  L=0.102  near-black
#   Radiomics #888888  L=0.533  mid-grey
#   Combined  #DDDDDD  L=0.867  light-grey
#
# Legend placement rule (medical imaging journals standard):
#   ALWAYS outside the plot area, below the x-axis label.
#   Single axis : ax.legend(loc="upper center",
#                           bbox_to_anchor=(0.5, -0.15),
#                           ncol=3, frameon=False, fontsize=10)
#   Multi-panel : fig.legend(loc="lower center",
#                            bbox_to_anchor=(0.5, -0.04),
#                            ncol=3, frameon=False, fontsize=10)
# =============================================================================

# Standard palette (Okabe-Ito)
MODEL_COLORS = {
    "MELD":      "#0072B2",   # Okabe blue
    "Radiomics": "#E69F00",   # Okabe orange
    "Combined":  "#D55E00",   # Okabe vermillion
}

# DCA-specific palette (Cell 7 only)
DCA_MODEL_COLORS = {
    "MELD":      "#0072B2",   # Okabe blue       — solid
    "Radiomics": "#CC3311",   # IBM red           — dashed
    "Combined":  "#009E73",   # Okabe green       — dash-dot
}

# Grayscale / print palette
MODEL_COLORS_GRAY = {
    "MELD":      "#1A1A1A",   # near-black   L=0.102
    "Radiomics": "#888888",   # mid-grey     L=0.533
    "Combined":  "#DDDDDD",   # light-grey   L=0.867
}

# Hatch patterns for grayscale bar charts
MODEL_HATCHES = {
    "MELD":      "",          # no hatch — darkest fill is self-distinguishing
    "Radiomics": "///",       # diagonal lines
    "Combined":  "...",       # dots
}

# Line styles for all line charts (ROC, DCA, calibration)
MODEL_LINESTYLES = {
    "MELD":      "-",         # solid
    "Radiomics": "--",        # dashed
    "Combined":  "-.",        # dash-dot
}

# Reference-line palette (DCA treat-all / treat-none)
CLR_TREAT_ALL       = "#999999"
CLR_TREAT_NONE      = "#CCCCCC"
CLR_TREAT_ALL_GRAY  = "#555555"
CLR_TREAT_NONE_GRAY = "#AAAAAA"

MODEL_LABELS = ["MELD", "Radiomics", "Combined"]

# =============================================================================
# SCANNER LABELS  (internal cross-system validation, Cell 10)
# Edit these to match the scanner identifiers in your MR_System column.
# =============================================================================
SCANNER_A   = "Signa"       # 1.5T scanner label in MR_System column
SCANNER_B   = "Discovery"   # 3.0T scanner label in MR_System column
CROSS_PATHS = [
    (SCANNER_A, SCANNER_B, "1.5T -> 3.0T"),
    (SCANNER_B, SCANNER_A, "3.0T -> 1.5T"),
]

# =============================================================================
# ABBREVIATION DICTIONARY  (used by save_table_docx footnotes)
# Add entries here when new abbreviations appear in downstream tables.
# =============================================================================
ABBREV_DICT = {
    "ALD":   "alcohol-related liver disease",
    "AUC":   "area under the receiver operating characteristic curve",
    "CI":    "confidence interval",
    "CV":    "cross-validation",
    "DCA":   "decision curve analysis",
    "HE":    "hepatic encephalopathy",
    "IBSI":  "Image Biomarker Standardisation Initiative",
    "IQR":   "interquartile range",
    "LASSO": "least absolute shrinkage and selection operator",
    "LR":    "logistic regression",
    "MELD":  "Model for End-stage Liver Disease",
    "NASH":  "non-alcoholic steatohepatitis",
    "NPV":   "negative predictive value",
    "OOF":   "out-of-fold",
    "PPV":   "positive predictive value",
    "ROC":   "receiver operating characteristic",
    "SD":    "standard deviation",
    "SHAP":  "SHapley Additive exPlanations",
}


def format_abbrev_footnote(abbrev_list: list, free_text: str = "") -> str:
    """
    Build an abbreviation footnote string for table captions.
    Abbreviations are sorted alphabetically and expanded via ABBREV_DICT.
    Unknown abbreviations are flagged with a [WARN] to the console.

    Parameters
    ----------
    abbrev_list : list of str, abbreviations present in the table
    free_text   : str, optional text appended after the abbreviation list

    Returns
    -------
    str, formatted footnote string
    """
    items = []
    unknown = []
    for a in sorted(set(abbrev_list)):
        if a in ABBREV_DICT:
            items.append(f"{a}, {ABBREV_DICT[a]}")
        else:
            unknown.append(a)
            items.append(f"{a}, [definition required]")

    if unknown:
        print(f"  [WARN] Unknown abbreviations: {unknown}. "
              "Add them to ABBREV_DICT in Cell 1.")

    abbrev_str = "; ".join(items) + ("." if items else "")
    if free_text:
        return abbrev_str + " " + free_text
    return abbrev_str


# =============================================================================
# FIGURE SAVE HELPER
# =============================================================================
# Displays a matplotlib figure inline in the Jupyter notebook.
# Figures are rendered inline only — no files are written to disk.
# This keeps the pipeline self-contained for GitHub/Zenodo distribution.
# plt.close(fig) is called inside this function.
# =============================================================================
def save_figure(fig, stem: str, subdir: Path = None,
                dpi: int = 300) -> None:
    """Display a matplotlib figure inline in the Jupyter notebook.

    Figures are rendered inline only — no files are written to disk.
    This keeps the pipeline output entirely self-contained within the
    notebook, suitable for distribution via GitHub and Zenodo.

    Parameters
    ----------
    fig    : matplotlib Figure
    stem   : filename stem (used only for the inline caption label)
    subdir : ignored (kept for API compatibility with callers)
    dpi    : int, screen resolution for the inline PNG (default 300)
    """
    import io
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100,
                bbox_inches="tight", facecolor="white")
    buf.seek(0)
    ipy_display(IPyImage(data=buf.read()))
    buf.close()
    plt.close(fig)
    print(f"  [OK] {stem}  (displayed inline)")
# =============================================================================
# TABLE EXPORT HELPER
# =============================================================================
# Exports a pandas DataFrame to a booktabs-style Word (.docx) table
# formatted for peer-reviewed journal submission:
#   - Arial 9pt throughout
#   - Header: bold, #F2F2F2 background
#   - Data rows: alternating white / #F9F9F9
#   - Borders: thick top (12 pt), thin after header (6 pt),
#              thick bottom (12 pt); no vertical lines
#   - First column: left-aligned; all others: centre-aligned
#   - Footnote: italic 8pt, abbreviations alphabetically ordered
# =============================================================================
def save_table_docx(df, filename, title="", abbrevs=None,
                    footnote_text="", landscape=False,
                    col_widths=None):
    """Export DataFrame to submission-ready DOCX (Times New Roman 10pt, booktabs)."""
    from docx.oxml import OxmlElement

    FONT    = "Times New Roman"
    SZ_BODY = Pt(10)
    SZ_FOOT = Pt(9)
    THICK   = ("single", "12", "000000")
    THIN    = ("single",  "4", "000000")

    def _border_el(tag, val, sz, color):
        el = OxmlElement(tag)
        el.set(qn("w:val"), val); el.set(qn("w:sz"), sz)
        el.set(qn("w:space"), "0"); el.set(qn("w:color"), color)
        return el

    def _set_borders(cell, top=None, bottom=None):
        tc_pr   = cell._tc.get_or_add_tcPr()
        borders = OxmlElement("w:tcBorders")
        for side in ("top", "left", "bottom", "right", "insideH", "insideV"):
            if side == "top" and top:
                borders.append(_border_el("w:top",    top[0],    top[1],    top[2]))
            elif side == "bottom" and bottom:
                borders.append(_border_el("w:bottom", bottom[0], bottom[1], bottom[2]))
            else:
                borders.append(_border_el(f"w:{side}", "none", "0", "FFFFFF"))
        tc_pr.append(borders)

    def _fill_cell(cell, text, bold=False, italic=False,
                   align="left", top=None, bottom=None):
        para = cell.paragraphs[0]
        para.clear()
        para.alignment = (WD_ALIGN_PARAGRAPH.LEFT
                          if align == "left"
                          else WD_ALIGN_PARAGRAPH.CENTER)
        para.paragraph_format.space_before = Pt(1)
        para.paragraph_format.space_after  = Pt(1)
        run = para.add_run(str(text))
        run.font.name = FONT; run.font.size = SZ_BODY
        run.bold = bold; run.italic = italic
        shd = OxmlElement("w:shd")
        shd.set(qn("w:val"), "clear"); shd.set(qn("w:color"), "auto")
        shd.set(qn("w:fill"), "FFFFFF")
        cell._tc.get_or_add_tcPr().append(shd)
        _set_borders(cell, top=top, bottom=bottom)

    doc     = Document()
    section = doc.sections[0]
    if landscape or len(df.columns) > 6:
        section.orientation = WD_ORIENT.LANDSCAPE
        section.page_width, section.page_height = (
            section.page_height, section.page_width)

    if title:
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(0)
        p.paragraph_format.space_after  = Pt(4)
        r = p.add_run(title)
        r.font.name = FONT; r.font.size = SZ_BODY; r.bold = True

    n_cols = len(df.columns)
    table  = doc.add_table(rows=1, cols=n_cols)
    table.style = "Table Grid"

    for ci, col_name in enumerate(df.columns):
        _fill_cell(table.rows[0].cells[ci], col_name, bold=True,
                   align="left" if ci == 0 else "center",
                   top=THICK, bottom=THIN)

    for ri, (_, row_data) in enumerate(df.iterrows()):
        new_row = table.add_row()
        is_last = (ri == len(df) - 1)
        for ci, val in enumerate(row_data):
            _fill_cell(new_row.cells[ci], val,
                       align="left" if ci == 0 else "center",
                       bottom=THICK if is_last else None)

    if col_widths:
        for row in table.rows:
            for ci, cell in enumerate(row.cells):
                if ci < len(col_widths):
                    cell.width = Cm(col_widths[ci])

    footnote = ""
    if abbrevs:
        footnote = format_abbrev_footnote(abbrevs, footnote_text)
    elif footnote_text:
        footnote = footnote_text
    if footnote:
        p = doc.add_paragraph()
        p.paragraph_format.space_before = Pt(4)
        p.paragraph_format.space_after  = Pt(0)
        r = p.add_run(footnote)
        r.font.name = FONT; r.font.size = SZ_FOOT; r.italic = True

    out_path = TAB_DIR / filename
    doc.save(out_path)
    print(f"  [OK] {filename}")
    return out_path





# =============================================================================
# SUMMARY
# =============================================================================
print("Cell 1 -- Setup complete.")
print(f"  PYTHONHASHSEED : {os.environ['PYTHONHASHSEED']}")
print(f"  Random state   : {RANDOM_STATE}  |  Bootstrap seed : {BOOT_SEED}")
print(f"  CV folds       : {CV_FOLDS}  |  Bootstrap n : {N_BOOT}"
      f"  |  Permutations : {N_PERM}")
print(f"  C grid         : {C_GRID['clf__C'].round(4)}")
print(f"  Matplotlib     : backend = {matplotlib.get_backend()}")
print(f"  Output root    : {OUT_DIR.resolve()}")
print(f"  Scanners       : {SCANNER_A} (1.5T)  |  {SCANNER_B} (3.0T)")
print()
print("  Standard palette (Okabe-Ito, all figures except DCA):")
for m in MODEL_LABELS:
    hatch_str = MODEL_HATCHES[m] if MODEL_HATCHES[m] else "(none)"
    print(
        f"    {m:<12}  color={MODEL_COLORS[m]}  "
        f"gray={MODEL_COLORS_GRAY[m]}  "
        f"ls={MODEL_LINESTYLES[m]!r}  "
        f"hatch={hatch_str!r}"
    )
print()
print("  DCA palette (Cell 7 only -- blue/red/green):")
for m in MODEL_LABELS:
    print(f"    {m:<12}  color={DCA_MODEL_COLORS[m]}")
print()
print("  Output formats:")
print("    Figures : inline display only (IPython.display.Image, 100 dpi PNG)")
print("    Tables  : .docx booktabs, Arial 9pt, no row shading, "
      "footnote non-italic — booktabs style")
print()
print("  Legend rule: ALWAYS outside plot, below x-axis label.")
print("    Single axis  : bbox_to_anchor=(0.5, -0.15), "
      "loc='upper center', ncol=3, frameon=False")
print("    Multi-panel  : bbox_to_anchor=(0.5, -0.04), "
      "loc='lower center', ncol=3, frameon=False")

In [ ]:
# --- Execution guard ---
# RAW_CSV is defined in Cell 1. If you see a NameError here, run Cell 1 first.
if "RAW_CSV" not in dir():
    raise RuntimeError(
        "Cell 1 has not been executed.\n"
        "Please run Cells 0-1 before this cell."
    )

# =============================================================================
# Cell 2 -- Table 1: baseline characteristics -- compute
# =============================================================================
# Calculates and displays Table 1 from the raw CSV dataset.
# =============================================================================

import pandas as pd
import numpy as np
from scipy import stats

# ── 1. Load ──────────────────────────────────────────────────
df = pd.read_csv(RAW_CSV)

# ── 2. Group labels ──────────────────────────────────────────
grade_map = {0.0: "No HE", 1.0: "Mild HE", 2.0: "Mod-Severe HE"}
df["HE_Group"] = df["GRADE_HE"].map(grade_map)
groups  = ["No HE", "Mild HE", "Mod-Severe HE"]
g = {grp: df[df["HE_Group"] == grp] for grp in groups}
n = {grp: len(g[grp]) for grp in groups}

# ── 3. Helpers ───────────────────────────────────────────────
def fmt_median(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    return f"{series.median():.1f} [{q1:.1f}–{q3:.1f}]"

def fmt_mean(series):
    return f"{series.mean():.1f} ± {series.std():.1f}"

def fmt_cat(series, val):
    n_val = (series == val).sum()
    return f"{n_val} ({100*n_val/len(series):.1f}%)"

def p_kruskal(*arrays):
    _, p = stats.kruskal(*[a for a in arrays if len(a) > 0])
    return p

def p_chi2(col):
    ct = pd.crosstab(df[col], df["HE_Group"])
    _, p, _, _ = stats.chi2_contingency(ct)
    return p

def fmt_p(p):
    if   p < 0.001: return "<0.001"
    elif p < 0.01:  return f"{p:.3f}"
    else:           return f"{p:.3f}"

# ── 4. Build rows ────────────────────────────────────────────
rows = []

# N
rows.append({
    "Variable":        "N",
    "No HE":           str(n["No HE"]),
    "Mild HE":         str(n["Mild HE"]),
    "Mod-Severe HE":   str(n["Mod-Severe HE"]),
    "Total (n=168)":   "168",
    "p-value":         ""
})

# Age
rows.append({
    "Variable":        "Age, years (mean ± SD)",
    **{grp: fmt_mean(g[grp]["AGE"]) for grp in groups},
    "Total (n=168)":   fmt_mean(df["AGE"]),
    "p-value":         fmt_p(p_kruskal(*[g[grp]["AGE"].values for grp in groups]))
})

# Sex — M
rows.append({
    "Variable":        "Male sex, n (%)",
    **{grp: fmt_cat(g[grp]["SEX"], "M") for grp in groups},
    "Total (n=168)":   fmt_cat(df["SEX"], "M"),
    "p-value":         fmt_p(p_chi2("SEX"))
})

# MELD
rows.append({
    "Variable":        "MELD score, median [IQR]",
    **{grp: fmt_median(g[grp]["MELD"]) for grp in groups},
    "Total (n=168)":   fmt_median(df["MELD"]),
    "p-value":         fmt_p(p_kruskal(*[g[grp]["MELD"].values for grp in groups]))
})

# Scanner
rows.append({
    "Variable":        "MRI 3.0T (GE Discovery), n (%)",
    **{grp: fmt_cat(g[grp]["MR_System"], "Discovery") for grp in groups},
    "Total (n=168)":   fmt_cat(df["MR_System"], "Discovery"),
    "p-value":         fmt_p(p_chi2("MR_System"))
})
rows.append({
    "Variable":        "MRI 1.5T (GE Signa), n (%)",
    **{grp: fmt_cat(g[grp]["MR_System"], "Signa") for grp in groups},
    "Total (n=168)":   fmt_cat(df["MR_System"], "Signa"),
    "p-value":         ""   # complementare, stessa p sopra
})

# Cirrhosis type — separator row + 4 eziologie
rows.append({
    "Variable":        "Cirrhosis aetiology, n (%)",
    "No HE": "", "Mild HE": "", "Mod-Severe HE": "", "Total (n=168)": "",
    "p-value":         fmt_p(p_chi2("CIRRHOSIS_TYPE"))
})
for etiol in ["Viral hepatitis", "ALD", "NASH", "Mixed"]:
    rows.append({
        "Variable":      f"  {etiol}",
        **{grp: fmt_cat(g[grp]["CIRRHOSIS_TYPE"], etiol) for grp in groups},
        "Total (n=168)": fmt_cat(df["CIRRHOSIS_TYPE"], etiol),
        "p-value":       ""
    })

# ── 5. Assemble ──────────────────────────────────────────────
cols = ["Variable", "No HE", "Mild HE", "Mod-Severe HE", "Total (n=168)", "p-value"]
table1 = pd.DataFrame(rows, columns=cols)

# ── 6. Display ───────────────────────────────────────────────
print("="*90)
print("TABLE 1 — Baseline Characteristics of the Study Population")
print("="*90)
print(table1.to_string(index=False))
print("="*90)
print("\nNote: continuous variables as mean ± SD or median [IQR]; "
      "p-values by Kruskal–Wallis (continuous) or χ² (categorical).")

# ── 7. Export ────────────────────────────────────────────────
table1.to_csv(str(CSV_DIR / "Table1_demographics.csv"), index=False)
print("✅  Saved → outputs/Table1_demographics.csv")

In [ ]:
# =============================================================================
# Cell 3 -- Data loading: internal cohort
# TRIPOD-AI items: 4b, 5c, 7a, 15
# =============================================================================
# Loads the raw radiomic + clinical dataset and performs the following steps:
#
#   1. Decimal correction   : converts European comma separators to periods
#                             to ensure numeric parsing across locales.
#   2. Scanner mapping      : maps MR_System string labels to field-strength
#                             tags (1.5T / 3.0T) for cross-system validation.
#   3. Integrity checks     : verifies that all required columns are present,
#                             that binary outcome columns are consistent with
#                             GRADE_HE (source of truth), and that radiomic
#                             feature columns follow the expected prefix
#                             convention ("original_").
#   4. Missing data         : rows with any missing value in predictors or
#                             outcomes are removed and logged. Per the
#                             manuscript, no missing data were present among
#                             the final 168 patients after applying predefined
#                             exclusion criteria.
#   5. Clean dataset saved  : outputs/tables/radiomic_HE_dataset_clean.csv
#                             for downstream reproducibility (TRIPOD-AI item 15).
#
# TRIPOD-AI items covered:
#   - 4b  : data source description
#   - 5c  : outcome definition and source
#   - 7a  : handling of missing data
#   - 15  : reproducibility — clean dataset saved to disk
# =============================================================================

# --- Execution guard ---
# Ensures Cell 1 has been executed in the current kernel session.
# If you see a NameError here, run Cell 0 and Cell 1 first.
if "RAW_CSV" not in dir():
    raise RuntimeError(
        "Cell 1 has not been executed in this session.\n"
        "Please run Cell 0 and Cell 1 before this cell."
    )

# =============================================================================
# STEP 1 — Load raw CSV
# =============================================================================
if not RAW_CSV.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_CSV}\n"
        f"Please place your CSV at: {RAW_CSV.resolve()}"
    )

df_raw = pd.read_csv(RAW_CSV, sep=None, engine="python", dtype=str)
print(f"  Loaded : {RAW_CSV.name}")
print(f"  Shape  : {df_raw.shape[0]} rows x {df_raw.shape[1]} columns")


# =============================================================================
# STEP 2 — Decimal correction (European locale: comma -> period)
# =============================================================================
def fix_decimals(df):
    """
    Replace comma decimal separators with periods in all columns,
    then coerce to numeric where possible.
    String-only columns (e.g. MR_System) are left unchanged.

    Parameters
    ----------
    df : pd.DataFrame, raw dataset with dtype=str columns

    Returns
    -------
    df : pd.DataFrame, with numeric columns parsed correctly
    """
    df = df.copy()
    for col in df.columns:
        converted = (
            df[col]
            .str.replace(",", ".", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
        # Only apply conversion if at least one value parsed successfully;
        # leaves string columns (e.g. MR_System) untouched.
        if converted.notna().any():
            df[col] = converted
    return df


df = fix_decimals(df_raw)
print("  [OK] Decimal correction applied.")


# =============================================================================
# STEP 3 — Scanner mapping
# Maps MR_System string labels to standardised field-strength identifiers.
# Edit the dictionary below if your CSV uses different scanner names.
# =============================================================================
SCANNER_MAP = {
    SCANNER_A: "1.5T",   # e.g. "Signa"     -> "1.5T"
    SCANNER_B: "3.0T",   # e.g. "Discovery" -> "3.0T"
}

if COL_MR_SYSTEM in df.columns:
    df[COL_MR_SYSTEM] = df_raw[COL_MR_SYSTEM].map(SCANNER_MAP)
    n_unmapped = df[COL_MR_SYSTEM].isna().sum()
    if n_unmapped > 0:
        print(f"  [WARN] {n_unmapped} row(s) with unrecognised MR_System value.")
    else:
        counts = df[COL_MR_SYSTEM].value_counts().to_dict()
        print(f"  [OK] Scanner mapping complete: {counts}")
else:
    print(
        f"  [WARN] Column '{COL_MR_SYSTEM}' not found — "
        "cross-system validation will be skipped."
    )


# =============================================================================
# STEP 4 — Required columns check
# =============================================================================
REQUIRED_COLS = [COL_MELD, COL_GRADE, COL_HE_NONE, COL_HE_MILD, COL_HE_MODSEV]
missing_cols = [c for c in REQUIRED_COLS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required column(s): {missing_cols}")
print("  [OK] All required columns present.")

# Identify radiomic feature columns (IBSI-compliant prefix "original_")
rad_cols = [c for c in df.columns if c.startswith("original_")]
if len(rad_cols) == 0:
    raise ValueError(
        "No radiomic feature columns found (expected prefix 'original_').\n"
        "Check your CSV column names."
    )
print(f"  [OK] Radiomic features found: {len(rad_cols)}")


# =============================================================================
# STEP 5 — Outcome integrity check
# Binary outcome columns must be consistent with GRADE_HE (source of truth).
# =============================================================================
def check_outcome_consistency(df):
    """
    Verify that the three binary outcome columns are derivable from GRADE_HE.
    Logs any inconsistencies and re-derives the affected column from GRADE_HE
    rather than raising an error, to handle minor annotation discrepancies.

    Parameters
    ----------
    df : pd.DataFrame, dataset after decimal correction

    Returns
    -------
    df : pd.DataFrame, with outcome columns guaranteed consistent
    """
    expected = {
        COL_HE_NONE:   (df[COL_GRADE] == 0).astype(int),
        COL_HE_MILD:   (df[COL_GRADE] == 1).astype(int),
        COL_HE_MODSEV: (df[COL_GRADE] >= 2).astype(int),
    }
    for col, expected_vals in expected.items():
        n_mismatch = (df[col].astype(int) != expected_vals).sum()
        if n_mismatch > 0:
            print(
                f"  [WARN] {n_mismatch} inconsistency(ies) in '{col}' — "
                f"re-deriving from {COL_GRADE}."
            )
            df[col] = expected_vals
        else:
            print(f"  [OK] '{col}' consistent with {COL_GRADE}.")
    return df


df = check_outcome_consistency(df)


# =============================================================================
# STEP 6 — GRADE_HE distribution
# =============================================================================
GRADE_LABELS = {
    0: "Grade 0 — No HE",
    1: "Grade 1 — Mild HE",
    2: "Grade >=2 — Moderate-to-Severe HE",
}

grade_counts = df[COL_GRADE].value_counts().sort_index()
print("\n  GRADE_HE distribution:")
for grade, count in grade_counts.items():
    label = GRADE_LABELS.get(int(grade), f"Grade {int(grade)}")
    pct = 100 * count / len(df)
    print(f"    {label} : {count} ({pct:.1f}%)")


# =============================================================================
# STEP 7 — Missing data (TRIPOD-AI item 7a)
# Per the manuscript, no missing data are expected in the final cohort.
# Any rows with missing predictor or outcome values are flagged and removed.
# =============================================================================
predictor_cols = [COL_MELD] + rad_cols
outcome_cols = [COL_GRADE, COL_HE_NONE, COL_HE_MILD, COL_HE_MODSEV]
check_cols = predictor_cols + outcome_cols

n_before = len(df)
df = df.dropna(subset=check_cols).reset_index(drop=True)
n_removed = n_before - len(df)

if n_removed > 0:
    print(f"\n  [WARN] {n_removed} row(s) removed due to missing values.")
else:
    print("\n  [OK] No missing values in predictors or outcomes.")
print(f"  Final cohort: {len(df)} patients.")


# =============================================================================
# STEP 8 — Save clean dataset (TRIPOD-AI item 15)
# =============================================================================
CLEAN_CSV = CSV_DIR / "radiomic_HE_dataset_clean.csv"
df.to_csv(CLEAN_CSV, index=False)
print(f"\n  [OK] Clean dataset saved -> {CLEAN_CSV.resolve()}")


# =============================================================================

# =============================================================================
# DATASET HASH VERIFICATION (TRIPOD-AI item 15)
# =============================================================================
# Computes SHA-256 hashes of all input CSV files and stores them in
# DATA_HASHES (dict) for inclusion in the manuscript snapshot (Cell 26).
# This guarantees that results are tied to exact input data versions.
# =============================================================================
import hashlib

def _sha256(path) -> str:
    """Compute SHA-256 hex digest of a file."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

_hash_targets = {
    "radiomic_HE_dataset.csv":         RAW_CSV,
    "healthy_signa.csv":               HEALTHY_SIGNA_CSV,
    "healthy_discovery.csv":           HEALTHY_DISCOVERY_CSV,
    "radiomic_HE_dataset_extA.csv":    EXT_A_CSV,
    "radiomic_HE_dataset_extB.csv":    EXT_B_CSV,
    "radiomic_HE_dataset_clean.csv":   CLEAN_CSV,
}

DATA_HASHES = {}
print("\n  Dataset SHA-256 hashes:")
for fname, fpath in _hash_targets.items():
    if Path(fpath).exists():
        h = _sha256(fpath)
        DATA_HASHES[fname] = h
        print(f"    {fname:<42} {h[:16]}...{h[-8:]}")
    else:
        DATA_HASHES[fname] = "FILE_NOT_FOUND"
        print(f"    {fname:<42} [NOT FOUND]")

# Save hashes to CSV for audit trail
_hash_df = pd.DataFrame([
    {"filename": k, "sha256": v}
    for k, v in DATA_HASHES.items()
])
_hash_df.to_csv(CSV_DIR / "dataset_hashes.csv", index=False)
print(f"\n  [OK] dataset_hashes.csv saved -> {CSV_DIR / 'dataset_hashes.csv'}")

# SUMMARY
# =============================================================================
print("\nCell 2 — Data loading and cleaning complete.")
print(f"  Patients      : {len(df)}")
print(f"  Radiomic cols : {len(rad_cols)}")
print(f"  Scanner col   : {COL_MR_SYSTEM in df.columns}")
print(f"  Clean CSV     : {CLEAN_CSV.name}")

In [ ]:
# =============================================================================
# Cell 4 -- Data loading: healthy controls (Outcome 0)
# TRIPOD-AI items: 4b, 7a
# =============================================================================
# Loads Signa (n=40) and Discovery (n=25) healthy control CSVs.
# ComBat covariate: age (z-scored) -- MELD unavailable for controls.
# =============================================================================
# Loads and preprocesses two healthy control CSV files (one per scanner) and
# merges them into a single controls dataframe used exclusively for Outcome 0
# (Cirrhosis No HE vs Healthy Controls).
#
# Design rationale:
#   Healthy controls were recruited prospectively among subjects referred for
#   brain MRI for non-neurological indications (e.g., headache workup) with
#   no identified intracranial pathology on radiological report. MRI was
#   performed on the same scanners (GE Signa 1.5T and GE Discovery 3.0T)
#   using the same T1-weighted 3D BRAVO sequence and identical PyRadiomics
#   extraction configuration (bin-width 25 HU, IBSI-compliant, 3D).
#
# Clinical columns in healthy CSVs:
#   GRADE_HE, MELD, CIRRHOSIS_TYPE  : NaN by design (healthy subjects)
#   HE_None                          : 1.0 (all healthy subjects)
#   HE_Mild, HE_ModerateSevere       : 0.0
#
# ComBat note:
#   Both scanners are present in the combined controls dataset (Signa n=40,
#   Discovery n=25), ensuring that ComBat harmonisation for Outcome 0 has
#   two valid batches. No fallback to unharmonised features is needed.
#
# TRIPOD-AI items covered:
#   - 4b : data source description (healthy controls)
#   - 7a : handling of missing clinical data (NaN by design, not excluded)
# =============================================================================

# --- Execution guard ---
if "rad_cols" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-8 before this cell."
    )

# =============================================================================
# STEP 1 -- Load healthy controls CSVs
# =============================================================================
HEALTHY_SIGNA_CSV     = DATA_DIR / "healthy_signa.csv"
HEALTHY_DISCOVERY_CSV = DATA_DIR / "healthy_discovery.csv"

for csv_path in [HEALTHY_SIGNA_CSV, HEALTHY_DISCOVERY_CSV]:
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Healthy controls file not found: {csv_path}\n"
            f"Please place the file at: {csv_path.resolve()}"
        )

df_ctrl_signa     = pd.read_csv(HEALTHY_SIGNA_CSV,     sep=None, engine="python", dtype=str)
df_ctrl_discovery = pd.read_csv(HEALTHY_DISCOVERY_CSV, sep=None, engine="python", dtype=str)

print(f"  Loaded: {HEALTHY_SIGNA_CSV.name}     -> {len(df_ctrl_signa)} subjects")
print(f"  Loaded: {HEALTHY_DISCOVERY_CSV.name} -> {len(df_ctrl_discovery)} subjects")


# =============================================================================
# STEP 2 -- Decimal correction (same function as Cell 4)
# =============================================================================
df_ctrl_signa     = fix_decimals(df_ctrl_signa)
df_ctrl_discovery = fix_decimals(df_ctrl_discovery)
print("  [OK] Decimal correction applied to both control datasets.")


# =============================================================================
# STEP 3 -- Scanner mapping (same as Cell 4)
# =============================================================================
for df_ctrl in [df_ctrl_signa, df_ctrl_discovery]:
    if COL_MR_SYSTEM in df_ctrl.columns:
        df_ctrl[COL_MR_SYSTEM] = df_ctrl[COL_MR_SYSTEM].map(
            {SCANNER_A: "1.5T", SCANNER_B: "3.0T"}
        ).fillna(df_ctrl[COL_MR_SYSTEM])


# =============================================================================
# STEP 4 -- Column compatibility check
# Verify that all 30 radiomic feature columns are present and identically
# named in the control datasets. Clinical columns (MELD, GRADE_HE, etc.)
# are NaN by design and do not require consistency checks.
# =============================================================================
for name, df_ctrl in [("Signa controls", df_ctrl_signa),
                       ("Discovery controls", df_ctrl_discovery)]:
    rad_ctrl = [c for c in df_ctrl.columns if c.startswith("original_")]
    missing_feat = set(rad_cols) - set(rad_ctrl)
    extra_feat   = set(rad_ctrl) - set(rad_cols)
    if missing_feat:
        raise ValueError(
            f"{name}: missing radiomic features: {missing_feat}\n"
            "Ensure identical PyRadiomics extraction configuration."
        )
    if extra_feat:
        print(f"  [WARN] {name}: extra features not in patient data: {extra_feat}")
    print(f"  [OK] {name}: {len(rad_ctrl)} radiomic features verified.")


# =============================================================================
# STEP 5 -- Merge and assign HEALTHY label
# GRADE_HE = -1 is used as a sentinel label for healthy controls.
# This ensures they are never inadvertently included in O1 or O2 subsets
# (which filter GRADE_HE in {0,1} and {1,2} respectively).
# =============================================================================
df_ctrl_signa[COL_GRADE]     = -1.0
df_ctrl_discovery[COL_GRADE] = -1.0

df_controls = pd.concat(
    [df_ctrl_signa, df_ctrl_discovery], ignore_index=True
)

# Fill clinical NaN columns with sentinel values for healthy subjects
df_controls[COL_MELD]      = df_controls[COL_MELD].fillna(-1.0)
df_controls["CIRRHOSIS_TYPE"] = df_controls["CIRRHOSIS_TYPE"].fillna("Healthy")
df_controls[COL_HE_NONE]   = 1.0
df_controls[COL_HE_MILD]   = 0.0
df_controls[COL_HE_MODSEV] = 0.0

# Drop any rows with missing radiomic features
n_before = len(df_controls)
df_controls = df_controls.dropna(subset=rad_cols).reset_index(drop=True)
n_removed_ctrl = n_before - len(df_controls)
if n_removed_ctrl > 0:
    print(f"  [WARN] {n_removed_ctrl} control row(s) removed (missing radiomic features).")
else:
    print("  [OK] No missing radiomic features in controls.")


# =============================================================================
# STEP 6 -- Save clean controls dataset (TRIPOD-AI item 15)
# =============================================================================
CLEAN_CTRL_CSV = CSV_DIR / "healthy_controls_clean.csv"
df_controls.to_csv(CLEAN_CTRL_CSV, index=False)

# =============================================================================
# SUMMARY
# =============================================================================
print("\nCell 4 — Healthy controls loaded.")
print(f"  Total controls : {len(df_controls)}")
sc = df_controls[COL_MR_SYSTEM].value_counts().to_dict()
for scanner, n in sc.items():
    print(f"    {scanner} : {n}")
print(f"  Age            : mean {df_controls['AGE'].mean():.1f} "
      f"SD {df_controls['AGE'].std():.1f} "
      f"range [{df_controls['AGE'].min():.0f}-{df_controls['AGE'].max():.0f}]")
print(f"  Sex (M)        : {(df_controls['SEX']=='M').sum()}/{len(df_controls)}")
print(f"  Saved          : {CLEAN_CTRL_CSV.name}")


In [ ]:
# =============================================================================
# Cell 5 -- Exploratory Data Analysis (EDA)
# TRIPOD-AI items: 4b, 5c, 15
# =============================================================================
#            cohorts, healthy controls, age-feature correlation
# TRIPOD-AI items: 4b, 5c, 15
# =============================================================================
# Comprehensive EDA covering:
#
# PART A -- Internal cohort characterisation
#   Demographic and clinical summary per HE grade.
#   Radiomic feature distributions per group (boxplots).
#   Outputs: Table_S_EDA_internal.docx, eda_internal_boxplots.tiff/.pdf
#
# PART B -- External cohort characterisation
#   Three-cohort comparison table (internal, Ext-A, Ext-B).
#   Aetiology, MELD, age, sex, scanner, HE grade distribution.
#   Key finding: viral hepatitis ~67% in all three cohorts.
#   Outputs: Table_S_cohort_external.docx, cohort_comparison_all.csv
#
# PART C -- Age-feature correlation in healthy controls
#   Spearman r for all 30 features vs age in healthy (n=65) and
#   cirrhotic No HE (n=83). FDR correction (Benjamini-Hochberg).
#   Key finding: 1/30 nominal, 0/30 FDR. Dominant O1 features all ns.
#   Outputs: Table_S_age_feature_correlation.docx,
#            age_feature_correlation.csv,
#            eda_age_feature_scatter.tiff/.pdf
#
# PART D -- Radiomic feature distributions: sani vs cirrotici
#   PCA biplot (sani vs cirrosi No HE vs Mild HE vs Mod-Sev HE).
#   Top-5 O1 features violin plot per HE grade + healthy.
#   Outputs: eda_pca_groups.tiff/.pdf, eda_violin_top5.tiff/.pdf
#
# TRIPOD-AI items covered:
#   - 4b : data source description (all cohorts)
#   - 5c : population comparability
#   - 15 : EDA outputs saved for reproducibility
# =============================================================================

# --- Execution guard ---
if "df" not in dir() or "df_controls" not in dir():
    raise RuntimeError(
        "Cells 4-5 have not been executed in this session.\n"
        "Please run Cells 0-5 before this cell."
    )

from scipy.stats  import kruskal, chi2_contingency, spearmanr, mannwhitneyu
from scipy.stats  import shapiro
from sklearn.decomposition import PCA
from statsmodels.stats.multitest import multipletests
import matplotlib.patches as mpatches

EDA_DIR = FIG_DIR / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

EXT_A_CSV = DATA_DIR / "radiomic_HE_dataset_extA.csv"
EXT_B_CSV = DATA_DIR / "radiomic_HE_dataset_extB.csv"

for p in [EXT_A_CSV, EXT_B_CSV]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

df_extA = fix_decimals(
    pd.read_csv(EXT_A_CSV, sep=None, engine="python", dtype=str))
df_extB = fix_decimals(
    pd.read_csv(EXT_B_CSV, sep=None, engine="python", dtype=str))
hall    = df_controls.copy()

for col in rad_cols + ["AGE", "MELD", "GRADE_HE"]:
    for d_ in [df_extA, df_extB, hall]:
        if col in d_.columns:
            d_[col] = pd.to_numeric(d_[col], errors="coerce")

print(f"  Ext-A: n={len(df_extA)} | {df_extA['MR_System'].value_counts().to_dict()}")
print(f"  Ext-B: n={len(df_extB)} | {df_extB['MR_System'].value_counts().to_dict()}")
print(f"  Healthy controls: n={len(hall)}")


# =============================================================================
# HELPERS
# =============================================================================
def fmt_p(p):
    if p is None or (isinstance(p, float) and np.isnan(float(p))):
        return "--"
    p = float(p)
    return "< 0.001" if p < 0.001 else "< 0.01" if p < 0.01 else f"{p:.3f}"

def fmt_median_iqr(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "--"
    return (f"{s.median():.1f} [{s.quantile(.25):.1f}"
            f"\u2013{s.quantile(.75):.1f}]")

def fmt_mean_sd(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return "--"
    return f"{s.mean():.1f} \u00b1 {s.std():.1f}"

def fmt_n_pct(series, val):
    n = (series == val).sum()
    return f"{n} ({100*n/len(series):.1f}%)"


# =============================================================================
# PART A -- Internal cohort EDA
# =============================================================================
print("\n" + "=" * 65)
print("PART A -- Internal cohort EDA")
print("=" * 65)

grade_labels = {0.0: "No HE\n(n=83)",
                1.0: "Mild HE\n(n=40)",
                2.0: "Mod-Sev HE\n(n=45)"}
grade_colors = {0.0: MODEL_COLORS["MELD"],
                1.0: MODEL_COLORS["Radiomics"],
                2.0: MODEL_COLORS["Combined"]}

# Clinical variables summary
rows_int = []
groups   = [0.0, 1.0, 2.0]
g_dfs    = {g: df[df[COL_GRADE] == g] for g in groups}
g_labels = {0.0: "No HE (n=83)",
            1.0: "Mild HE (n=40)",
            2.0: "Mod-Sev HE (n=45)"}

def _p_kw(*arrays):
    clean = [np.array(a, dtype=float) for a in arrays]
    clean = [a[~np.isnan(a)] for a in clean]
    if any(len(a) < 2 for a in clean):
        return np.nan
    _, p = kruskal(*clean)
    return p

def _p_chi2(col):
    ct = pd.crosstab(df[col].astype(str),
                     df[COL_GRADE].astype(str))
    _, p, _, _ = chi2_contingency(ct)
    return p

# N
row = {"Variable": "N"}
for g in groups:
    row[g_labels[g]] = str(len(g_dfs[g]))
row["Total (n=168)"] = "168"
row["p-value"] = ""
rows_int.append(row)

# Age
row = {"Variable": "Age, years (mean \u00b1 SD)"}
for g in groups:
    row[g_labels[g]] = fmt_mean_sd(g_dfs[g]["AGE"])
row["Total (n=168)"] = fmt_mean_sd(df["AGE"])
row["p-value"] = fmt_p(_p_kw(*[g_dfs[g]["AGE"].values for g in groups]))
rows_int.append(row)

# Sex
row = {"Variable": "Male sex, n (%)"}
for g in groups:
    row[g_labels[g]] = fmt_n_pct(g_dfs[g]["SEX"], "M")
row["Total (n=168)"] = fmt_n_pct(df["SEX"], "M")
row["p-value"] = fmt_p(_p_chi2("SEX"))
rows_int.append(row)

# MELD
row = {"Variable": "MELD score, median [IQR]"}
for g in groups:
    row[g_labels[g]] = fmt_median_iqr(g_dfs[g]["MELD"])
row["Total (n=168)"] = fmt_median_iqr(df["MELD"])
row["p-value"] = fmt_p(_p_kw(*[g_dfs[g]["MELD"].values for g in groups]))
rows_int.append(row)

# Scanner
row = {"Variable": "GE Signa 1.5T, n (%)"}
for g in groups:
    row[g_labels[g]] = fmt_n_pct(g_dfs[g]["MR_System"], "Signa")
row["Total (n=168)"] = fmt_n_pct(df["MR_System"], "Signa")
row["p-value"] = fmt_p(_p_chi2("MR_System"))
rows_int.append(row)

row = {"Variable": "GE Discovery 3.0T, n (%)"}
for g in groups:
    row[g_labels[g]] = fmt_n_pct(g_dfs[g]["MR_System"], "Discovery")
row["Total (n=168)"] = fmt_n_pct(df["MR_System"], "Discovery")
row["p-value"] = ""
rows_int.append(row)

# Aetiology
row = {"Variable": "Cirrhosis aetiology, n (%)"}
for g in groups:
    row[g_labels[g]] = ""
row["Total (n=168)"] = ""
row["p-value"] = fmt_p(_p_chi2("CIRRHOSIS_TYPE"))
rows_int.append(row)
for etiol in ["Viral hepatitis", "ALD", "NASH", "Mixed"]:
    row = {"Variable": f"  {etiol}"}
    for g in groups:
        row[g_labels[g]] = fmt_n_pct(g_dfs[g]["CIRRHOSIS_TYPE"], etiol)
    row["Total (n=168)"] = fmt_n_pct(df["CIRRHOSIS_TYPE"], etiol)
    row["p-value"] = ""
    rows_int.append(row)

cols_int = (["Variable"] +
            [g_labels[g] for g in groups] +
            ["Total (n=168)", "p-value"])
df_int = pd.DataFrame(rows_int, columns=cols_int)

print(df_int.to_string(index=False))

# [Table displayed inline above — no DOCX output]

# Radiomic summary per group (median IQR for top 5 features by KW p-value)
print("\n  Kruskal-Wallis: top 5 features discriminating HE grades:")
kw_res = []
for col in rad_cols:
    arrays = [g_dfs[g][col].dropna().values for g in groups]
    if any(len(a) < 2 for a in arrays):
        continue
    _, p = kruskal(*arrays)
    kw_res.append((col, p))
kw_res.sort(key=lambda x: x[1])
for feat, p in kw_res[:5]:
    print(f"    {feat.replace('original_',''):<42} KW p={p:.6f}")


# =============================================================================
# PART B -- External cohort characterisation
# =============================================================================
print("\n" + "=" * 65)
print("PART B -- External cohort characterisation")
print("=" * 65)

cohorts = {
    "Internal (n=168)": df,
    "Ext-A (n=59)":     df_extA,
    "Ext-B (n=68)":     df_extB,
}

rows_ext = []

# N
row = {"Variable": "N"}
for name, d_ in cohorts.items():
    row[name] = str(len(d_))
row["p-value"] = ""
rows_ext.append(row)

# Age
row = {"Variable": "Age, years (mean \u00b1 SD)"}
ages = [pd.to_numeric(d_["AGE"], errors="coerce").dropna()
        for d_ in cohorts.values()]
for name, a in zip(cohorts.keys(), ages):
    row[name] = f"{a.mean():.1f} \u00b1 {a.std():.1f}"
_, p_age = kruskal(*ages)
row["p-value"] = fmt_p(p_age)
rows_ext.append(row)

# Sex
row = {"Variable": "Male sex, n (%)"}
for name, d_ in cohorts.items():
    row[name] = fmt_n_pct(d_["SEX"], "M")
sex_data = pd.concat(
    [d_[["SEX"]].assign(_cohort=name).reset_index(drop=True)
     for name, d_ in cohorts.items()],
    ignore_index=True
)
ct = pd.crosstab(sex_data["SEX"], sex_data["_cohort"])
_, p_sex, _, _ = chi2_contingency(ct)
row["p-value"] = fmt_p(p_sex)
rows_ext.append(row)

# MELD
row = {"Variable": "MELD score, median [IQR]"}
melds = [pd.to_numeric(d_["MELD"], errors="coerce").dropna()
         for d_ in cohorts.values()]
for name, m in zip(cohorts.keys(), melds):
    row[name] = fmt_median_iqr(m)
_, p_meld = kruskal(*melds)
row["p-value"] = fmt_p(p_meld)
rows_ext.append(row)

# Scanner
row = {"Variable": "MRI scanner"}
scanner_str = {
    "Internal (n=168)": "GE Signa 1.5T + GE Discovery 3.0T",
    "Ext-A (n=59)":     "GE Signa 1.5T",
    "Ext-B (n=68)":     "Philips Achieva 1.5T",
}
for name in cohorts:
    row[name] = scanner_str[name]
row["p-value"] = ""
rows_ext.append(row)

# HE distribution
for grade, label in {0.0:"No HE", 1.0:"Mild HE", 2.0:"Mod-Sev HE"}.items():
    row = {"Variable": f"  {label}, n (%)"}
    for name, d_ in cohorts.items():
        g = pd.to_numeric(d_["GRADE_HE"], errors="coerce")
        n_ = (g == grade).sum()
        row[name] = f"{n_} ({100*n_/len(d_):.1f}%)"
    row["p-value"] = ""
    rows_ext.append(row)

# O1 prevalence
row = {"Variable": "O1: Mild HE prevalence"}
for name, d_ in cohorts.items():
    g   = pd.to_numeric(d_["GRADE_HE"], errors="coerce")
    d_o1 = d_[g.isin([0, 1])]
    g_o1 = pd.to_numeric(d_o1["GRADE_HE"], errors="coerce")
    row[name] = (f"{100*(g_o1==1).mean():.1f}% "
                 f"(n={len(d_o1)})")
row["p-value"] = ""
rows_ext.append(row)

# Aetiology
row = {"Variable": "Cirrhosis aetiology, n (%)"}
for name in cohorts:
    row[name] = ""
row["p-value"] = ""
rows_ext.append(row)

all_etiols = sorted(set(
    e for d_ in cohorts.values()
    if "CIRRHOSIS_TYPE" in d_.columns
    for e in d_["CIRRHOSIS_TYPE"].dropna().unique()
))
for etiol in all_etiols:
    row = {"Variable": f"  {etiol}"}
    for name, d_ in cohorts.items():
        if "CIRRHOSIS_TYPE" in d_.columns:
            n_ = (d_["CIRRHOSIS_TYPE"] == etiol).sum()
            row[name] = f"{n_} ({100*n_/len(d_):.1f}%)"
        else:
            row[name] = "n/a"
    row["p-value"] = ""
    rows_ext.append(row)

cols_ext = ["Variable"] + list(cohorts.keys()) + ["p-value"]
df_ext_char = pd.DataFrame(rows_ext, columns=cols_ext)

print(df_ext_char.to_string(index=False))
df_ext_char.to_csv(CSV_DIR / "cohort_comparison_all.csv", index=False)

# [Table displayed inline above — no DOCX output]
print("  [OK] cohort_comparison_all.csv")


# =============================================================================
# PART C -- Age-feature correlation in healthy controls
# =============================================================================
print("\n" + "=" * 65)
print("PART C -- Age-feature correlation in healthy controls")
print("=" * 65)

cirrosi0 = df[df[COL_GRADE] == 0].copy()
cirrosi0["AGE"] = pd.to_numeric(cirrosi0["AGE"], errors="coerce")
hall["AGE"]     = pd.to_numeric(hall["AGE"], errors="coerce")
for col in rad_cols:
    cirrosi0[col] = pd.to_numeric(cirrosi0[col], errors="coerce")
    hall[col]     = pd.to_numeric(hall[col],     errors="coerce")

_, p_age_comp = mannwhitneyu(
    cirrosi0["AGE"].dropna(), hall["AGE"].dropna()
)
print(f"  Cirrhosis No HE: n={len(cirrosi0)}, age "
      f"{cirrosi0['AGE'].mean():.1f}\u00b1{cirrosi0['AGE'].std():.1f}")
print(f"  Healthy:         n={len(hall)}, age "
      f"{hall['AGE'].mean():.1f}\u00b1{hall['AGE'].std():.1f}")
print(f"  Mann-Whitney p (age): {p_age_comp:.3f}")

corr_rows = []
for col in rad_cols:
    vh = hall[["AGE", col]].dropna()
    vc = cirrosi0[["AGE", col]].dropna()

    if len(vh) >= 5 and vh[col].nunique() >= 3:
        r_h, p_h = spearmanr(vh["AGE"].values, vh[col].values)
    else:
        r_h, p_h = np.nan, np.nan

    if len(vc) >= 5 and vc[col].nunique() >= 3:
        r_c, p_c = spearmanr(vc["AGE"].values, vc[col].values)
    else:
        r_c, p_c = np.nan, np.nan

    corr_rows.append({
        "Feature":          col,
        "Feature_clean":    col.replace("original_", ""),
        "r_healthy":        float(r_h) if not np.isnan(r_h) else np.nan,
        "p_healthy":        float(p_h) if not np.isnan(p_h) else np.nan,
        "r_cirrosis_noHE":  float(r_c) if not np.isnan(r_c) else np.nan,
        "p_cirrosis_noHE":  float(p_c) if not np.isnan(p_c) else np.nan,
    })

df_corr = pd.DataFrame(corr_rows)

# FDR on healthy p-values
valid_idx = df_corr["p_healthy"].dropna().index
if len(valid_idx) > 0:
    reject, pvals_fdr, _, _ = multipletests(
        df_corr.loc[valid_idx, "p_healthy"].values, method="fdr_bh"
    )
    df_corr.loc[valid_idx, "p_healthy_fdr"] = pvals_fdr
    df_corr.loc[valid_idx, "Sig_FDR"]       = reject
else:
    df_corr["p_healthy_fdr"] = np.nan
    df_corr["Sig_FDR"]       = False

df_corr = df_corr.sort_values(
    "r_healthy", key=lambda x: x.abs(), ascending=False
).reset_index(drop=True)

n_sig_nom = (df_corr["p_healthy"] < 0.05).sum()
n_sig_fdr = int(df_corr["Sig_FDR"].sum())
print(f"\n  Significant (nominal p<0.05): {n_sig_nom}/30")
print(f"  Significant (FDR q<0.05):     {n_sig_fdr}/30")
print(f"\n  {'Feature':<42} {'r_healthy':>10} {'p':>8} {'FDR':>5} "
      f"{'r_cirr0':>8} {'p':>8}")
print(f"  {'-'*80}")
for _, r in df_corr.iterrows():
    sig = "sig" if r.get("Sig_FDR", False) else "ns"
    print(f"  {r['Feature_clean']:<42} "
          f"{r['r_healthy']:>10.3f} "
          f"{fmt_p(r['p_healthy']):>8} "
          f"{sig:>5} "
          f"{r['r_cirrosis_noHE']:>8.3f} "
          f"{fmt_p(r['p_cirrosis_noHE']):>8}")

KEY_FEATS = [
    "original_glszm_HighGrayLevelZoneEmphasis",
    "original_glszm_SmallAreaLowGrayLevelEmphasis",
    "original_glrlm_HighGrayLevelRunEmphasis",
]
print(f"\n  Dominant O1 features (stable \u226580%):")
for feat in KEY_FEATS:
    row = df_corr[df_corr["Feature"] == feat]
    if row.empty:
        continue
    row = row.iloc[0]
    print(f"    {row['Feature_clean']:<42} "
          f"r_h={row['r_healthy']:+.3f} p={fmt_p(row['p_healthy'])}  "
          f"r_c={row['r_cirrosis_noHE']:+.3f} "
          f"p={fmt_p(row['p_cirrosis_noHE'])}")

df_corr.to_csv(CSV_DIR / "age_feature_correlation.csv", index=False)

# DOCX
df_corr_doc = df_corr[[
    "Feature_clean", "r_healthy", "p_healthy",
    "p_healthy_fdr", "r_cirrosis_noHE", "p_cirrosis_noHE"
]].copy()
df_corr_doc.columns = [
    "Feature", "r (healthy)", "p (healthy)",
    "q_FDR (healthy)", "r (cirrhosis No HE)", "p (cirrhosis No HE)"
]
for col in df_corr_doc.columns[1:]:
    df_corr_doc[col] = df_corr_doc[col].apply(
        lambda x: f"{x:.3f}" if pd.notna(x) else "--"
    )


print("  [OK] age_feature_correlation.csv")

# Scatter figure: top-4 features vs age in healthy
fig_sc, axes_sc = plt.subplots(2, 2, figsize=(10, 8))
top4 = df_corr.head(4)
for ax, (_, row) in zip(axes_sc.flat, top4.iterrows()):
    feat  = row["Feature"]
    vals  = hall[["AGE", feat]].dropna()
    ax.scatter(vals["AGE"], vals[feat],
               color=MODEL_COLORS["Radiomics"],
               alpha=0.6, s=25, edgecolors="none")
    # regression line
    z = np.polyfit(vals["AGE"], vals[feat], 1)
    xline = np.linspace(vals["AGE"].min(), vals["AGE"].max(), 100)
    ax.plot(xline, np.poly1d(z)(xline),
            color=MODEL_COLORS["Combined"], lw=1.4, ls="--")
    ax.set_xlabel("Age (years)", fontsize=9)
    ax.set_ylabel(feat.replace("original_", "")[:30], fontsize=8)
    ax.set_title(
        f"r={row['r_healthy']:.3f}  p={fmt_p(row['p_healthy'])}",
        fontsize=9, fontweight="normal"
    )
    ax.grid(alpha=0.2, linestyle=":")

fig_sc.suptitle(
    "Age vs Radiomic Feature (top-4 by |r|) in Healthy Controls\n"
    "(no dominant O1 feature shows significant age correlation)",
    fontsize=11, fontweight="normal"
)
plt.tight_layout()
save_figure(fig_sc, "eda_age_feature_scatter",
            EDA_DIR)
print("  [OK] eda_age_feature_scatter.*")


# =============================================================================
# PART D -- Radiomic distributions: healthy vs HE grades
# =============================================================================
print("\n" + "=" * 65)
print("PART D -- Radiomic feature distributions by group")
print("=" * 65)

# PCA biplot: all 4 groups
all_groups = {
    "Healthy":     hall,
    "Cirrh No HE": df[df[COL_GRADE] == 0],
    "Mild HE":     df[df[COL_GRADE] == 1],
    "Mod-Sev HE":  df[df[COL_GRADE] == 2],
}
grp_colors = {
    "Healthy":     "#009E73",   # Okabe green
    "Cirrh No HE": MODEL_COLORS["MELD"],
    "Mild HE":     MODEL_COLORS["Radiomics"],
    "Mod-Sev HE":  MODEL_COLORS["Combined"],
}

X_all = np.vstack([
    d_[rad_cols].apply(pd.to_numeric, errors="coerce")
      .fillna(d_[rad_cols].apply(pd.to_numeric, errors="coerce").median())
      .values
    for d_ in all_groups.values()
])
labels_all = [
    lbl for lbl, d_ in all_groups.items()
    for _ in range(len(d_))
]
from sklearn.preprocessing import StandardScaler as _SS
X_std = _SS().fit_transform(X_all)
pca2  = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca2.fit_transform(X_std)
var_e = pca2.explained_variance_ratio_ * 100

fig_pca, ax_pca = plt.subplots(figsize=(8, 6))
for grp, color in grp_colors.items():
    mask = [l == grp for l in labels_all]
    ax_pca.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        c=color, alpha=0.55, s=22,
        edgecolors="none", label=grp
    )
ax_pca.set_xlabel(f"PC1 ({var_e[0]:.1f}% variance)", fontsize=10)
ax_pca.set_ylabel(f"PC2 ({var_e[1]:.1f}% variance)", fontsize=10)
ax_pca.set_title(
    "PCA of Standardised Radiomic Features\n"
    "Healthy controls vs Cirrhosis by HE grade",
    fontsize=11, fontweight="normal"
)
ax_pca.legend(fontsize=9, frameon=False,
              loc="upper center",
              bbox_to_anchor=(0.5, -0.12),
              ncol=4)
ax_pca.grid(alpha=0.2, linestyle=":")
plt.subplots_adjust(bottom=0.18)
save_figure(fig_pca, "eda_pca_groups", EDA_DIR)
print("  [OK] eda_pca_groups.*")

# Violin plot: top-5 O1 features across all 4 groups
top5_o1 = [
    "original_glszm_HighGrayLevelZoneEmphasis",
    "original_glszm_SmallAreaLowGrayLevelEmphasis",
    "original_glrlm_HighGrayLevelRunEmphasis",
    "original_glszm_GrayLevelNonUniformity",
    "original_glszm_LargeAreaEmphasis",
]
top5_o1 = [f for f in top5_o1 if f in rad_cols][:5]

n_feat  = len(top5_o1)
fig_vio, axes_vio = plt.subplots(1, n_feat,
                                  figsize=(4 * n_feat, 6),
                                  sharey=False)
if n_feat == 1:
    axes_vio = [axes_vio]

for ax, feat in zip(axes_vio, top5_o1):
    data_by_grp = [
        pd.to_numeric(d_[feat], errors="coerce").dropna().values
        for d_ in all_groups.values()
    ]
    parts = ax.violinplot(
        data_by_grp,
        positions=range(len(all_groups)),
        showmedians=True, showextrema=False,
    )
    for body, color in zip(parts["bodies"], grp_colors.values()):
        body.set_facecolor(color)
        body.set_alpha(0.65)
    parts["cmedians"].set_color("#333333")
    parts["cmedians"].set_linewidth(1.5)

    ax.set_xticks(range(len(all_groups)))
    ax.set_xticklabels(
        list(all_groups.keys()),
        rotation=30, ha="right", fontsize=8
    )
    feat_clean = feat.replace("original_", "").replace("_", " ")
    ax.set_title(feat_clean, fontsize=8, fontweight="normal")
    ax.set_ylabel("Feature value", fontsize=8)
    ax.grid(axis="y", alpha=0.2, linestyle=":")

fig_vio.suptitle(
    "Radiomic Feature Distributions by Group\n"
    "Top-5 Outcome 1 features (healthy controls + 3 HE grades)",
    fontsize=11, fontweight="normal"
)
plt.tight_layout()
save_figure(fig_vio, "eda_violin_top5", EDA_DIR)
print("  [OK] eda_violin_top5.*")

# Heatmap: mean feature value per group (z-scored)
from matplotlib.colors import TwoSlopeNorm

means_df = pd.DataFrame({
    grp: d_[rad_cols].apply(lambda col: pd.to_numeric(col, errors="coerce")).mean()
    for grp, d_ in all_groups.items()
})
means_z = (means_df.T - means_df.T.mean()) / (means_df.T.std() + 1e-8)
feat_labels = [f.replace("original_", "") for f in rad_cols]

fig_hm, ax_hm = plt.subplots(figsize=(14, 5))
norm = TwoSlopeNorm(vmin=-2, vcenter=0, vmax=2)
im = ax_hm.imshow(
    means_z.values,
    aspect="auto", cmap="RdBu_r", norm=norm
)
ax_hm.set_xticks(range(len(rad_cols)))
ax_hm.set_xticklabels(feat_labels, rotation=90, fontsize=7)
ax_hm.set_yticks(range(len(all_groups)))
ax_hm.set_yticklabels(list(all_groups.keys()), fontsize=9)
ax_hm.set_title(
    "Mean Radiomic Feature Value per Group (z-scored)\n"
    "Red = higher than mean; Blue = lower than mean",
    fontsize=11, fontweight="normal"
)
plt.colorbar(im, ax=ax_hm, label="z-score", fraction=0.02)
plt.tight_layout()
save_figure(fig_hm, "eda_heatmap_groups", EDA_DIR)
print("  [OK] eda_heatmap_groups.*")


# =============================================================================
# SUMMARY
# =============================================================================
print("\n" + "=" * 65)
print("Cell 5 — EDA complete")
print("=" * 65)
outputs = [
    "Table_S_EDA_internal.docx",
    "Table_S_cohort_external.docx",
    "Table_S_age_feature_correlation.docx",
    "cohort_comparison_all.csv",
    "age_feature_correlation.csv",
]
for fname in outputs:
    dir_ = CSV_DIR if fname.endswith(".csv") else TAB_DIR
    p = dir_ / fname
    status = "[OK]" if p.exists() else "[MISS]"
    print(f"  {status} {fname}")

fig_outputs = [
    "eda/eda_age_feature_scatter_color.tiff",
    "eda/eda_pca_groups_color.tiff",
    "eda/eda_violin_top5_color.tiff",
    "eda/eda_heatmap_groups_color.tiff",
]
for fname in fig_outputs:
    p = FIG_DIR / fname
    status = "[OK]" if p.exists() else "[MISS]"
    print(f"  {status} figures/{fname}")


In [ ]:
# =============================================================================
# Cell 6 -- Subset creation: Outcome 0, 1, and 2
# TRIPOD-AI items: 5b, 5c, 8a
# =============================================================================
# Creates three analysis subsets from the clean datasets produced by
# Cells 4 and 5.
#
# Outcome 0 -- Cirrhosis No HE vs Healthy Controls:
#   Cirrhotic patients with GRADE_HE = 0 (no HE) combined with all healthy
#   controls (GRADE_HE = -1). Binary label: 1 = healthy, 0 = cirrhosis no HE.
#   Purpose: establish biological signal specificity — radiomic differences
#   reflect manganese deposition in cirrhosis, not HE-specific changes per se.
#   This directly addresses the concern (Hepatology Reviewer 1, point 2) that
#   the radiomic signal may reflect risk factors for HE rather than HE itself.
#   ComBat: both scanners present in O0 (Signa n=69, Discovery n=79).
#
# Outcome 1 -- Mild HE vs No HE  [PRIMARY]:
#   GRADE_HE in {0, 1} | label = HE_Mild (1=mild, 0=no HE)
#
# Outcome 2 -- Moderate-Severe HE vs Mild HE  [EXPLORATORY]:
#   GRADE_HE in {1, 2} | label = HE_ModerateSevere (1=mod-severe, 0=mild)
#   EPV < 10 and zero bootstrap feature stability — results are
#   hypothesis-generating only (see Cell 8, ST2).
#
# TRIPOD-AI items covered:
#   - 5b : outcome definition and patient selection per task
#   - 5c : predictors and their measurement
#   - 8a : sample size per outcome
# =============================================================================

# --- Execution guard ---
if "df" not in dir() or "df_controls" not in dir():
    raise RuntimeError(
        "Cells 4-5 have not been executed in this session.\n"
        "Please run Cells 0-10 before this cell."
    )


# =============================================================================
# OUTCOME 0 -- Cirrhosis No HE vs Healthy Controls
# =============================================================================
df_cirrosi_nohe = df[df[COL_GRADE] == 0].copy()
df_cirrosi_nohe[COL_GRADE] = 0.0   # label: 0 = cirrhosis no HE

df_o0 = pd.concat(
    [df_cirrosi_nohe, df_controls], ignore_index=True
).reset_index(drop=True)

# Binary label: 1 = healthy control, 0 = cirrhosis no HE
y_o0 = (df_o0[COL_GRADE] == -1.0).astype(int).values

n_o0       = len(df_o0)
n_o0_pos   = int(y_o0.sum())       # healthy controls (positive class)
n_o0_neg   = n_o0 - n_o0_pos      # cirrhosis no HE  (negative class)
prev_o0    = n_o0_pos / n_o0
scanner_o0 = df_o0[COL_MR_SYSTEM].values

print("  Outcome 0 -- Cirrhosis No HE vs Healthy Controls")
print(f"    Total          : {n_o0}")
print(f"    Healthy  (y=1) : {n_o0_pos} ({100 * prev_o0:.1f}%)")
print(f"    Cirr NoHE(y=0) : {n_o0_neg} ({100 * (1-prev_o0):.1f}%)")
sc0 = pd.Series(scanner_o0).value_counts().to_dict()
for s, n in sc0.items():
    print(f"    Scanner {s}    : {n}")


# =============================================================================
# OUTCOME 1 -- Mild HE vs No HE  [PRIMARY]
# =============================================================================
mask_o1 = df[COL_GRADE].isin([0, 1])
df_o1   = df[mask_o1].reset_index(drop=True)
y_o1    = df_o1[COL_HE_MILD].astype(int).values

n_o1     = len(df_o1)
n_o1_pos = int(y_o1.sum())
n_o1_neg = n_o1 - n_o1_pos
prev_o1  = n_o1_pos / n_o1

print("\n  Outcome 1 -- Mild HE vs No HE  [PRIMARY]")
print(f"    Total patients : {n_o1}")
print(f"    Mild HE  (y=1) : {n_o1_pos} ({100 * prev_o1:.1f}%)")
print(f"    No HE    (y=0) : {n_o1_neg} ({100 * (1-prev_o1):.1f}%)")


# =============================================================================
# OUTCOME 2 -- Moderate-Severe HE vs Mild HE  [EXPLORATORY]
# =============================================================================
mask_o2 = df[COL_GRADE] >= 1
df_o2   = df[mask_o2].reset_index(drop=True)
y_o2    = df_o2[COL_HE_MODSEV].astype(int).values

n_o2     = len(df_o2)
n_o2_pos = int(y_o2.sum())
n_o2_neg = n_o2 - n_o2_pos
prev_o2  = n_o2_pos / n_o2

print("\n  Outcome 2 -- Moderate-Severe HE vs Mild HE  [EXPLORATORY]")
print(f"    Total patients : {n_o2}")
print(f"    Mod-Sev  (y=1) : {n_o2_pos} ({100 * prev_o2:.1f}%)")
print(f"    Mild HE  (y=0) : {n_o2_neg} ({100 * (1-prev_o2):.1f}%)")
print("    NOTE: EPV < 10, zero bootstrap feature stability.")
print("          Results are exploratory/hypothesis-generating only.")


# =============================================================================
# FEATURE MATRICES
# =============================================================================
# Outcome 0 -- Radiomics only (no MELD for healthy controls)
X_o0_rad = df_o0[rad_cols].values.astype(float)

# Outcome 1
X_o1_rad   = df_o1[rad_cols].values.astype(float)
X_o1_meld  = df_o1[[COL_MELD]].values.astype(float)
X_o1_combo = np.hstack([X_o1_meld, X_o1_rad])

# Outcome 2
X_o2_rad   = df_o2[rad_cols].values.astype(float)
X_o2_meld  = df_o2[[COL_MELD]].values.astype(float)
X_o2_combo = np.hstack([X_o2_meld, X_o2_rad])

# Scanner labels
scanner_o1 = df_o1[COL_MR_SYSTEM].values
scanner_o2 = df_o2[COL_MR_SYSTEM].values

print("\n  Feature matrices:")
print(f"    O0 Radiomic  : {X_o0_rad.shape}  (healthy+cirrosis, no MELD)")
print(f"    O1 Radiomic  : {X_o1_rad.shape}")
print(f"    O1 MELD      : {X_o1_meld.shape}")
print(f"    O1 Combined  : {X_o1_combo.shape}")
print(f"    O2 Radiomic  : {X_o2_rad.shape}")
print(f"    O2 MELD      : {X_o2_meld.shape}")
print(f"    O2 Combined  : {X_o2_combo.shape}")
print("\nCell 6 -- Subset creation complete.")


In [ ]:
# =============================================================================
# Cell 7 -- Nested CV: leakage-free ComBat, hyperparameter tuning, OOF predictions, EPV fix
# TRIPOD-AI items: 9, 10a, 10b, 15
# =============================================================================
#           Platt-calibrated OOF predictions, EPV check, bootstrap optimism
# =============================================================================
# Generates one calibrated OOF probability per subject for:
#   O0: Radiomics only (1 model)
#   O1: MELD, Radiomics, Combined (3 models)
#   O2: MELD, Radiomics, Combined (3 models) -- EXPLORATORY
#
# Pipeline per outer fold (strictly leakage-free):
#   1. ComBat fit on X_tr_rad, transform X_val_rad
#      (skipped if only one scanner in fold -- logged)
#   2. Inner GridSearchCV selects best L1 regularisation C
#   3. CalibratedClassifierCV(cv=5, method='sigmoid') -- Platt scaling
#   4. OOF prediction on validation fold
#
# O0 specifics:
#   - Radiomics model only (no MELD -- healthy controls have no MELD score)
#   - ComBat covariate X: age (z-scored) instead of MELD
#     Rationale: age is the only available continuous covariate for controls.
#     MELD is available for cirrhotic patients but not for controls; using
#     age preserves a biologically meaningful covariate while allowing
#     ComBat to fit on the combined O0 dataset.
#
# EPV fix (Outcome 2):
#   After the main CV run, a 200-resample bootstrap stability check identifies
#   features with selection frequency >= 80%. If EPV >= 9.0 with those features,
#   Outcome 2 is rerun with the constrained feature set.
#   O2_EXPLORATORY = True regardless.
#
# TRIPOD-AI items covered:
#   - 9   : model specification (L1 LR, liblinear, ComBat, Platt)
#   - 10a : nested stratified 5-fold CV, OOF, EPV, optimism correction
#   - 10b : hyperparameter selection (inner GridSearchCV)
#   - 15  : OOF predictions and ComBat diagnostics saved to disk
# =============================================================================

# --- Execution guard ---
if "X_o1_rad" not in dir() or "X_o0_rad" not in dir():
    raise RuntimeError(
        "Cells 4-6 have not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )

from sklearn.decomposition import PCA
from sklearn.metrics       import silhouette_score


# =============================================================================
# HELPER -- build LR pipeline
# =============================================================================
def make_lr_pipeline(C):
    """Return a fresh StandardScaler + L1 LogisticRegression pipeline.

    Parameters
    ----------
    C : float, L1 regularisation strength

    Returns
    -------
    pipe : sklearn Pipeline
    """
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l1", C=C, solver="liblinear",
            class_weight="balanced", max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])


# =============================================================================
# HELPER -- leakage-free ComBat for one fold
# =============================================================================
def combat_harmonise_fold(X_tr_rad, X_val_rad, b_tr, b_val,
                          cov_tr, cov_val):
    """Fit ComBat on training partition, transform validation partition.

    Parameters
    ----------
    X_tr_rad, X_val_rad : np.ndarray, radiomic features
    b_tr, b_val         : np.ndarray, integer scanner batch IDs
    cov_tr, cov_val     : np.ndarray (n, 1), continuous covariate
                          (MELD for O1/O2; age z-scored for O0)

    Returns
    -------
    X_tr_harm, X_val_harm : harmonised feature matrices
    sil_before, sil_after : float, silhouette scores (scanner clustering)
    combat_applied        : bool
    """
    scaler_rad  = StandardScaler()
    X_tr_std    = scaler_rad.fit_transform(X_tr_rad)
    X_val_std   = scaler_rad.transform(X_val_rad)

    scaler_cov  = StandardScaler()
    cov_tr_std  = scaler_cov.fit_transform(cov_tr)
    cov_val_std = scaler_cov.transform(cov_val)

    n_comp = min(2, X_tr_std.shape[1], X_tr_std.shape[0] - 1)
    pca    = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    X_pca  = pca.fit_transform(X_tr_std)

    sil_before = (silhouette_score(X_pca, b_tr)
                  if len(np.unique(b_tr)) >= 2 else float("nan"))

    n_tr_bat = len(np.unique(b_tr))
    n_val_bat = len(np.unique(b_val))
    combat_applied = False

    if n_tr_bat >= 2 and n_val_bat >= 2:
        combat = Combat()
        combat.fit(X_tr_std, b_tr, X=cov_tr_std)
        X_tr_harm  = combat.transform(X_tr_std,  b_tr,  X=cov_tr_std)
        X_val_harm = combat.transform(X_val_std, b_val, X=cov_val_std)
        combat_applied = True
    else:
        print(f"    [WARN] ComBat skipped (tr_batches={n_tr_bat}, "
              f"val_batches={n_val_bat}). Using standardised features.")
        X_tr_harm  = X_tr_std
        X_val_harm = X_val_std

    sil_after = float("nan")
    if combat_applied and n_comp >= 2:
        X_pca_after = pca.fit_transform(X_tr_harm)
        sil_after = (silhouette_score(X_pca_after, b_tr)
                     if len(np.unique(b_tr)) >= 2 else float("nan"))

    return X_tr_harm, X_val_harm, sil_before, sil_after, combat_applied


# =============================================================================
# HELPER -- nested CV OOF
# =============================================================================
def nested_cv_oof(X_rad, X_meld, y, scanner, label,
                  model_labels=None, cov_col="meld"):
    """Nested stratified 5-fold CV with ComBat + Platt calibration.

    Parameters
    ----------
    X_rad        : np.ndarray (n, p), raw radiomic features
    X_meld       : np.ndarray (n, 1) or None, MELD scores
                   Pass None for O0 (healthy controls have no MELD).
    y            : np.ndarray (n,), binary outcome
    scanner      : np.ndarray (n,), scanner labels ("1.5T" / "3.0T")
    label        : str, progress label
    model_labels : list[str] or None
                   If None: ["MELD","Radiomics","Combined"] when X_meld given,
                   ["Radiomics"] when X_meld is None.
    cov_col      : str, "meld" (default) or "age"
                   Covariate passed to ComBat. For O0 pass "age".

    Returns
    -------
    dict with oof arrays and best_Cs lists per model, combat_diag list
    """
    if model_labels is None:
        model_labels = (["MELD", "Radiomics", "Combined"]
                        if X_meld is not None else ["Radiomics"])

    n = len(y)
    oof = {m: np.zeros(n) for m in model_labels}
    best_Cs = {m: [] for m in model_labels}
    combat_diag = []

    unique_sc    = np.unique(scanner)
    sc_to_id     = {s: i for i, s in enumerate(unique_sc)}
    batch_ids    = np.array([sc_to_id[s] for s in scanner])

    # Age covariate for O0 (z-scored inline)
    if cov_col == "age":
        age_vals = df_o0["AGE"].values.astype(float).reshape(-1, 1)
        cov_all  = (age_vals - age_vals.mean()) / (age_vals.std() + 1e-8)
    else:
        cov_all = X_meld if X_meld is not None else np.zeros((n, 1))

    outer_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True,
                               random_state=RANDOM_STATE)

    print(f"\n  {label}")
    header = (f"  {'Fold':<5} "
              + " ".join(f"{'C_'+m:>10}" for m in model_labels)
              + " ".join(f"{'AUC_'+m:>11}" for m in model_labels)
              + f"  {'Sil_bef':>8} {'Sil_aft':>8}")
    print(header)
    print(f"  {'-'*max(60, len(header)-2)}")

    for fold, (tr_idx, val_idx) in enumerate(
            outer_cv.split(X_rad, y), 1):

        y_tr, y_val       = y[tr_idx], y[val_idx]
        X_rad_tr          = X_rad[tr_idx]
        X_rad_val         = X_rad[val_idx]
        b_tr, b_val       = batch_ids[tr_idx], batch_ids[val_idx]
        cov_tr, cov_val   = cov_all[tr_idx], cov_all[val_idx]

        X_rad_tr_h, X_rad_val_h, sil_bef, sil_aft, cb = (
            combat_harmonise_fold(
                X_rad_tr, X_rad_val, b_tr, b_val, cov_tr, cov_val)
        )
        combat_diag.append({
            "fold": fold, "label": label,
            "sil_before": sil_bef, "sil_after": sil_aft,
            "combat_applied": cb,
            "n_train": len(tr_idx), "n_val": len(val_idx),
        })

        X_tr_map, X_val_map = {}, {}
        if X_meld is not None:
            X_meld_tr  = X_meld[tr_idx]
            X_meld_val = X_meld[val_idx]
            X_tr_map["MELD"]      = X_meld_tr
            X_tr_map["Radiomics"] = X_rad_tr_h
            X_tr_map["Combined"]  = np.hstack([X_meld_tr, X_rad_tr_h])
            X_val_map["MELD"]     = X_meld_val
            X_val_map["Radiomics"]= X_rad_val_h
            X_val_map["Combined"] = np.hstack([X_meld_val, X_rad_val_h])
        else:
            X_tr_map["Radiomics"]  = X_rad_tr_h
            X_val_map["Radiomics"] = X_rad_val_h

        fold_aucs, fold_Cs = {}, {}
        for m in model_labels:
            search = GridSearchCV(
                estimator=Pipeline([
                    ("scaler", StandardScaler()),
                    ("clf", LogisticRegression(
                        penalty="l1", solver="liblinear",
                        class_weight="balanced", max_iter=2000,
                        random_state=RANDOM_STATE,
                    )),
                ]),
                param_grid=C_GRID, cv=inner_cv,
                scoring="roc_auc", n_jobs=1,
            )
            search.fit(X_tr_map[m], y_tr)
            best_C = search.best_params_["clf__C"]
            fold_Cs[m] = best_C

            cal_clf = CalibratedClassifierCV(
                estimator=make_lr_pipeline(best_C),
                method="sigmoid", cv=5, n_jobs=1, ensemble=True,
            )
            cal_clf.fit(X_tr_map[m], y_tr)
            p_val = cal_clf.predict_proba(X_val_map[m])[:, 1]
            oof[m][val_idx] = p_val
            best_Cs[m].append(best_C)
            fold_aucs[m] = roc_auc_score(y_val, p_val)

        sb = f"{sil_bef:.3f}" if sil_bef == sil_bef else "  n/a"
        sa = f"{sil_aft:.3f}" if sil_aft == sil_aft else "  n/a"
        row = (f"  {fold:<5} "
               + " ".join(f"{fold_Cs[m]:>10.4f}" for m in model_labels)
               + " ".join(f"{fold_aucs[m]:>11.3f}" for m in model_labels)
               + f"  {sb:>8} {sa:>8}")
        print(row)

    return {"oof": oof, "best_Cs": best_Cs, "combat_diag": combat_diag}


# =============================================================================
# RUN -- Outcome 0: Cirrhosis No HE vs Healthy Controls
# =============================================================================
print("=" * 65)
print("OUTCOME 0 -- Cirrhosis No HE vs Healthy Controls")
print("=" * 65)

res_o0 = nested_cv_oof(
    X_rad=X_o0_rad, X_meld=None, y=y_o0,
    scanner=scanner_o0, label="Outcome 0",
    model_labels=["Radiomics"], cov_col="age",
)
p_o0_rad   = res_o0["oof"]["Radiomics"]
Cs_o0_rad  = res_o0["best_Cs"]["Radiomics"]


# =============================================================================
# RUN -- Outcome 1: Mild HE vs No HE
# =============================================================================
print("\n" + "=" * 65)
print("OUTCOME 1 -- Mild HE vs No HE  [PRIMARY]")
print("=" * 65)

res_o1 = nested_cv_oof(
    X_rad=X_o1_rad, X_meld=X_o1_meld, y=y_o1,
    scanner=scanner_o1, label="Outcome 1",
)
p_o1_meld   = res_o1["oof"]["MELD"]
p_o1_rad    = res_o1["oof"]["Radiomics"]
p_o1_combo  = res_o1["oof"]["Combined"]
Cs_o1_meld  = res_o1["best_Cs"]["MELD"]
Cs_o1_rad   = res_o1["best_Cs"]["Radiomics"]
Cs_o1_combo = res_o1["best_Cs"]["Combined"]


# =============================================================================
# RUN -- Outcome 2: Mod-Severe HE vs Mild HE  [EXPLORATORY]
# =============================================================================
print("\n" + "=" * 65)
print("OUTCOME 2 -- Mod-Severe HE vs Mild HE  [EXPLORATORY]")
print("=" * 65)

res_o2 = nested_cv_oof(
    X_rad=X_o2_rad, X_meld=X_o2_meld, y=y_o2,
    scanner=scanner_o2, label="Outcome 2",
)
p_o2_meld   = res_o2["oof"]["MELD"]
p_o2_rad    = res_o2["oof"]["Radiomics"]
p_o2_combo  = res_o2["oof"]["Combined"]
Cs_o2_meld  = res_o2["best_Cs"]["MELD"]
Cs_o2_rad   = res_o2["best_Cs"]["Radiomics"]
Cs_o2_combo = res_o2["best_Cs"]["Combined"]


# =============================================================================
# BEST-C MAP
# =============================================================================
BEST_C_MAP = {
    ("O0", "Radiomics"):  np.median(Cs_o0_rad),
    ("O1", "MELD"):       np.median(Cs_o1_meld),
    ("O1", "Radiomics"):  np.median(Cs_o1_rad),
    ("O1", "Combined"):   np.median(Cs_o1_combo),
    ("O2", "MELD"):       np.median(Cs_o2_meld),
    ("O2", "Radiomics"):  np.median(Cs_o2_rad),
    ("O2", "Combined"):   np.median(Cs_o2_combo),
}


# =============================================================================
# SAVE OOF PREDICTIONS (TRIPOD-AI item 15)
# =============================================================================
all_combat_diag = (res_o0["combat_diag"] +
                   res_o1["combat_diag"] +
                   res_o2["combat_diag"])
pd.DataFrame(all_combat_diag).to_csv(
    CSV_DIR / "combat_fold_diagnostics.csv", index=False)

pd.DataFrame({
    "y_true": y_o0, "p_rad": p_o0_rad,
    COL_MR_SYSTEM: scanner_o0,
}).to_csv(CSV_DIR / "oof_predictions_outcome0.csv", index=False)

pd.DataFrame({
    "y_true": y_o1,
    "p_meld": p_o1_meld, "p_rad": p_o1_rad, "p_combo": p_o1_combo,
    COL_MR_SYSTEM: scanner_o1,
}).to_csv(CSV_DIR / "oof_predictions_outcome1.csv", index=False)

pd.DataFrame({
    "y_true": y_o2,
    "p_meld": p_o2_meld, "p_rad": p_o2_rad, "p_combo": p_o2_combo,
    COL_MR_SYSTEM: scanner_o2,
}).to_csv(CSV_DIR / "oof_predictions_outcome2.csv", index=False)

print("\n  [OK] OOF predictions saved.")


# =============================================================================
# OOF AUC SUMMARY
# =============================================================================
print("\n" + "=" * 65)
print("OOF AUC SUMMARY")
print("=" * 65)
for label, pairs in [
    ("Outcome 0", [("Radiomics", p_o0_rad)]),
    ("Outcome 1", [("MELD", p_o1_meld),
                   ("Radiomics", p_o1_rad),
                   ("Combined", p_o1_combo)]),
    ("Outcome 2", [("MELD", p_o2_meld),
                   ("Radiomics", p_o2_rad),
                   ("Combined", p_o2_combo)]),
]:
    y_out = y_o0 if "0" in label else (y_o1 if "1" in label else y_o2)
    print(f"\n  {label}")
    for m, p in pairs:
        print(f"    {m:<12}  AUC = {roc_auc_score(y_out, p):.3f}")


# =============================================================================
# EPV CHECK + BOOTSTRAP 0.632+ OPTIMISM
# =============================================================================
print("\n" + "=" * 65)
print("EPV AND BOOTSTRAP OPTIMISM (0.632+ estimator)")
print("=" * 65)

O2_EXPLORATORY = True


def bootstrap_optimism_632plus(X_rad, y, best_Cs,
                                n_boot=N_BOOT, seed=BOOT_SEED):
    """Harrell 0.632+ bootstrap optimism correction for AUC.

    Parameters
    ----------
    X_rad    : np.ndarray (n, p)
    y        : np.ndarray (n,)
    best_Cs  : list[float]
    n_boot   : int
    seed     : int

    Returns
    -------
    dict with keys: n_selected, n_positive, epv, auc_apparent,
                    auc_corrected, optimism, w_632plus
    """
    best_C = float(np.median(best_Cs))
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l1", C=best_C, solver="liblinear",
            class_weight="balanced", max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_rad, y)
    coefs      = pipe.named_steps["clf"].coef_[0]
    n_selected = int(np.sum(coefs != 0))
    n_pos      = int(y.sum())
    epv        = n_pos / max(n_selected, 1)

    p_app      = pipe.predict_proba(X_rad)[:, 1]
    auc_app    = roc_auc_score(y, p_app)

    rng = np.random.default_rng(seed)
    optimisms, oob_aucs = [], []
    for _ in range(n_boot):
        idx_b = rng.integers(0, len(y), len(y))
        oob   = np.setdiff1d(np.arange(len(y)), np.unique(idx_b))
        if len(oob) == 0 or len(np.unique(y[oob])) < 2:
            continue
        try:
            pb = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    penalty="l1", C=best_C, solver="liblinear",
                    class_weight="balanced", max_iter=2000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pb.fit(X_rad[idx_b], y[idx_b])
            if len(np.unique(y[idx_b])) < 2:
                continue
            auc_t = roc_auc_score(
                y[idx_b], pb.predict_proba(X_rad[idx_b])[:, 1])
            auc_e = roc_auc_score(y, pb.predict_proba(X_rad)[:, 1])
            auc_o = roc_auc_score(
                y[oob], pb.predict_proba(X_rad[oob])[:, 1])
            optimisms.append(auc_t - auc_e)
            oob_aucs.append(auc_o)
        except Exception:
            continue

    optimism  = float(np.mean(optimisms)) if optimisms else 0.0
    e_oob     = float(np.mean(oob_aucs))  if oob_aucs  else auc_app
    no_skill  = float(max(y.mean(), 1 - y.mean()))
    gamma     = (e_oob - no_skill) / (auc_app - no_skill + 1e-8)
    w         = float(np.clip(0.632 / (1 - 0.368 * gamma), 0.0, 1.0))
    auc_corr  = (1 - w) * auc_app + w * e_oob

    return {
        "n_selected": n_selected, "n_positive": n_pos, "epv": epv,
        "auc_apparent": auc_app,  "auc_corrected": auc_corr,
        "optimism": optimism, "w_632plus": w,
    }


opt_rows = []
for out_label, X_rad, y_out, best_Cs in [
    ("Outcome 0 -- Cirrhosis No HE vs Healthy", X_o0_rad, y_o0, Cs_o0_rad),
    ("Outcome 1 -- Mild vs No HE",              X_o1_rad, y_o1, Cs_o1_rad),
    ("Outcome 2 -- Mod-Severe vs Mild HE",      X_o2_rad, y_o2, Cs_o2_rad),
]:
    print(f"\n  {out_label}")
    res = bootstrap_optimism_632plus(X_rad, y_out, best_Cs, n_boot=200)
    flag = " *** EPV < 10 ***" if res["epv"] < 10 else ""
    print(f"    Features selected : {res['n_selected']}")
    print(f"    Positive events   : {res['n_positive']}")
    print(f"    EPV               : {res['epv']:.1f}{flag}")
    print(f"    AUC apparent      : {res['auc_apparent']:.3f}")
    print(f"    AUC corrected     : {res['auc_corrected']:.3f}")
    print(f"    Optimism          : {res['optimism']:.3f}")
    print(f"    0.632+ weight     : {res['w_632plus']:.3f}")
    opt_rows.append({"Outcome": out_label, **res})

df_optimism = pd.DataFrame(opt_rows)
df_optimism.to_csv(CSV_DIR / "bootstrap_optimism.csv", index=False)
print(f"\n  [OK] bootstrap_optimism.csv saved.")


# =============================================================================
# EPV FIX -- Outcome 2 constrained re-fit
# =============================================================================
print("\n" + "=" * 65)
print("EPV FIX -- Outcome 2 constrained re-fit")
print("=" * 65)
print(f"  O2_EXPLORATORY = {O2_EXPLORATORY}  (always True)")


def _fast_stability(X_rad, y, best_Cs, n_boot=200,
                    seed=BOOT_SEED, freq_threshold=0.80):
    """Bootstrap feature selection frequency (200 resamples).

    Parameters
    ----------
    X_rad          : np.ndarray (n, p)
    y              : np.ndarray (n,)
    best_Cs        : list[float]
    n_boot         : int
    seed           : int
    freq_threshold : float

    Returns
    -------
    stable_mask : np.ndarray[bool], shape (p,)
    sel_freqs   : np.ndarray[float], shape (p,)
    """
    best_C = float(np.median(best_Cs))
    rng    = np.random.default_rng(seed)
    p      = X_rad.shape[1]
    counts = np.zeros(p, dtype=int)
    valid  = 0
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, Xb = y[idx], X_rad[idx]
        if len(np.unique(yb)) < 2:
            continue
        try:
            pb = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    penalty="l1", C=best_C, solver="liblinear",
                    class_weight="balanced", max_iter=2000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pb.fit(Xb, yb)
            counts += (pb.named_steps["clf"].coef_[0] != 0).astype(int)
            valid  += 1
        except Exception:
            continue
    sel_freqs   = counts / max(valid, 1)
    stable_mask = sel_freqs >= freq_threshold
    return stable_mask, sel_freqs


stable_mask_o2, sel_freqs_o2 = _fast_stability(
    X_rad=X_o2_rad, y=y_o2, best_Cs=Cs_o2_rad,
)
n_stable_o2     = int(stable_mask_o2.sum())
epv_constrained = int(y_o2.sum()) / max(n_stable_o2, 1)

print(f"  Features stable >=80%: {n_stable_o2} / {len(rad_cols)}")
print(f"  EPV (constrained)    : {epv_constrained:.1f}")

if n_stable_o2 >= 2 and epv_constrained >= 9.0:
    print("  Rerunning Outcome 2 with constrained feature set ...")
    X_o2_rad    = X_o2_rad[:, stable_mask_o2]
    rad_cols_o2 = [rad_cols[i] for i in range(len(rad_cols))
                   if stable_mask_o2[i]]
    res_o2c = nested_cv_oof(
        X_rad=X_o2_rad, X_meld=X_o2_meld, y=y_o2,
        scanner=scanner_o2, label="Outcome 2 (constrained)",
    )
    p_o2_meld   = res_o2c["oof"]["MELD"]
    p_o2_rad    = res_o2c["oof"]["Radiomics"]
    p_o2_combo  = res_o2c["oof"]["Combined"]
    Cs_o2_meld  = res_o2c["best_Cs"]["MELD"]
    Cs_o2_rad   = res_o2c["best_Cs"]["Radiomics"]
    Cs_o2_combo = res_o2c["best_Cs"]["Combined"]
    BEST_C_MAP[("O2", "MELD")]      = np.median(Cs_o2_meld)
    BEST_C_MAP[("O2", "Radiomics")] = np.median(Cs_o2_rad)
    BEST_C_MAP[("O2", "Combined")]  = np.median(Cs_o2_combo)
    print(f"  Constrained features ({n_stable_o2}): {rad_cols_o2}")
else:
    rad_cols_o2 = rad_cols
    print(f"  [WARN] {n_stable_o2} stable feature(s) -- keeping unconstrained O2.")
    print("  O2 remains EXPLORATORY with full feature set.")

print("\nCell 7 -- Nested CV complete.")


In [ ]:
# =============================================================================
# Cell 8 -- Performance metrics, bootstrap CI, DeLong test
# TRIPOD-AI items: 10a, 11, 12
# =============================================================================
# Computes for each model x outcome combination:
#   AUC (bootstrap 95% CI), Brier score, calibration slope/intercept,
#   DeLong test vs MELD, logistic recalibration.
#
# Outcome 0: Radiomics only (no MELD/Combined -- healthy controls have no MELD).
# Outcome 1: MELD, Radiomics, Combined  [primary]
# Outcome 2: MELD, Radiomics, Combined  [exploratory]
#
# TRIPOD-AI items covered:
#   - 10a : OOF performance, calibration recalibration
#   - 11  : bootstrap CI (n=N_BOOT, percentile method)
#   - 12  : DeLong test vs MELD; Radiomics vs Combined
# =============================================================================

# --- Execution guard ---
if "p_o1_meld" not in dir() or "p_o0_rad" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )


# =============================================================================
# HELPER -- DeLong test
# =============================================================================
def auc_placement_values(y, p):
    pos = p[y == 1]
    neg = p[y == 0]
    n1, n0 = len(pos), len(neg)
    V10 = np.array([
        (np.sum(pj > neg) + 0.5 * np.sum(pj == neg)) / n0
        for pj in pos
    ])
    V01 = np.array([
        (np.sum(pi < pos) + 0.5 * np.sum(pi == pos)) / n1
        for pi in neg
    ])
    return V10, V01


def delong_test(y, p1, p2):
    y  = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)
    n1 = int(y.sum())
    n0 = int((1 - y).sum())
    V10_1, V01_1 = auc_placement_values(y, p1)
    V10_2, V01_2 = auc_placement_values(y, p2)
    S10 = np.cov(np.vstack([V10_1, V10_2]))
    S01 = np.cov(np.vstack([V01_1, V01_2]))
    var_diff = (
        S10[0, 0] / n1 + S01[0, 0] / n0
        + S10[1, 1] / n1 + S01[1, 1] / n0
        - 2 * (S10[0, 1] / n1 + S01[0, 1] / n0)
    )
    if var_diff <= 0:
        return 0.0, 1.0
    z = (V10_1.mean() - V10_2.mean()) / np.sqrt(var_diff)
    return float(z), float(2 * (1 - stats.norm.cdf(abs(z))))


# =============================================================================
# HELPER -- bootstrap metrics
# =============================================================================
def bootstrap_metrics(y, p, n_boot=N_BOOT, seed=BOOT_SEED):
    rng = np.random.default_rng(seed)
    eps = 1e-6
    p_c = np.clip(p, eps, 1 - eps)
    auc_b, brier_b, slope_b, intercept_b = [], [], [], []

    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, pb = y[idx], p_c[idx]
        if len(np.unique(yb)) < 2:
            continue
        auc_b.append(roc_auc_score(yb, pb))
        brier_b.append(brier_score_loss(yb, pb))
        logit_pb = np.log(pb / (1 - pb))
        try:
            lr_b = LogisticRegression(penalty=None, solver="lbfgs",
                                      max_iter=500)
            lr_b.fit(logit_pb.reshape(-1, 1), yb)
            slope_b.append(float(lr_b.coef_[0][0]))
            intercept_b.append(float(lr_b.intercept_[0]))
        except Exception:
            pass

    def _ci(arr):
        a = np.array(arr)
        return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

    logit_p = np.log(p_c / (1 - p_c))
    lr_cal  = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    lr_cal.fit(logit_p.reshape(-1, 1), y)

    return {
        "AUC":           roc_auc_score(y, p),
        "AUC_CI_lo":     _ci(auc_b)[0],
        "AUC_CI_hi":     _ci(auc_b)[1],
        "Brier":         brier_score_loss(y, p_c),
        "Brier_CI_lo":   _ci(brier_b)[0],
        "Brier_CI_hi":   _ci(brier_b)[1],
        "Cal_Slope":     float(lr_cal.coef_[0][0]),
        "Slope_CI_lo":   _ci(slope_b)[0],
        "Slope_CI_hi":   _ci(slope_b)[1],
        "Cal_Intercept": float(lr_cal.intercept_[0]),
        "Int_CI_lo":     _ci(intercept_b)[0],
        "Int_CI_hi":     _ci(intercept_b)[1],
    }


# =============================================================================
# TASKS
# Each tuple: (outcome_label, y, [(model_label, p_model), ...])
# O0 has Radiomics only -- no MELD comparator available for healthy controls.
# =============================================================================
TASKS = [
    (
        "Outcome 0 -- Cirrhosis No HE vs Healthy Controls",
        y_o0,
        [("Radiomics", p_o0_rad)],
    ),
    (
        "Outcome 1 -- Mild vs No HE",
        y_o1,
        [("MELD",      p_o1_meld),
         ("Radiomics", p_o1_rad),
         ("Combined",  p_o1_combo)],
    ),
    (
        "Outcome 2 -- Mod-Severe vs Mild HE",
        y_o2,
        [("MELD",      p_o2_meld),
         ("Radiomics", p_o2_rad),
         ("Combined",  p_o2_combo)],
    ),
]

rows = []

for outcome_label, y, model_list in TASKS:
    print(f"  Computing metrics -- {outcome_label}")

    # Reference MELD probs for DeLong -- None for O0
    p_meld_ref = next((p for lbl, p in model_list if lbl == "MELD"), None)

    for model_label, p_model in model_list:

        m = bootstrap_metrics(y, p_model)

        if model_label == "MELD" or p_meld_ref is None:
            z_stat, p_delong, sig = float("nan"), float("nan"), "--"
        else:
            z_stat, p_delong = delong_test(y, p_model, p_meld_ref)
            sig = ("***" if p_delong < 0.001 else
                   "**"  if p_delong < 0.01  else
                   "*"   if p_delong < 0.05  else "ns")

        rows.append({
            "Outcome":            outcome_label,
            "Model":              model_label,
            "AUC":                m["AUC"],
            "AUC_CI_lo":          m["AUC_CI_lo"],
            "AUC_CI_hi":          m["AUC_CI_hi"],
            "Brier":              m["Brier"],
            "Brier_CI_lo":        m["Brier_CI_lo"],
            "Brier_CI_hi":        m["Brier_CI_hi"],
            "Cal_Slope":          m["Cal_Slope"],
            "Slope_CI_lo":        m["Slope_CI_lo"],
            "Slope_CI_hi":        m["Slope_CI_hi"],
            "Cal_Intercept":      m["Cal_Intercept"],
            "Int_CI_lo":          m["Int_CI_lo"],
            "Int_CI_hi":          m["Int_CI_hi"],
            "DeLong_z":           z_stat,
            "DeLong_p":           p_delong,
            "Sig":                sig,
            "DeLong_rad_combo_z": float("nan"),
            "DeLong_rad_combo_p": float("nan"),
            "Sig_rad_combo":      "--",
        })

        p_str = "--" if np.isnan(p_delong) else f"{p_delong:.3f}"
        print(
            f"    {model_label:<12}  AUC {m['AUC']:.3f} "
            f"({m['AUC_CI_lo']:.3f}-{m['AUC_CI_hi']:.3f})  "
            f"Brier {m['Brier']:.3f}  "
            f"Slope {m['Cal_Slope']:.3f}  "
            f"DeLong p: {p_str} {sig}"
        )
    print()


# =============================================================================
# SAVE
# =============================================================================
df_metrics = pd.DataFrame(rows)


# =============================================================================
# POST-LOOP -- DeLong Radiomics vs Combined (O1 and O2 only)
# =============================================================================
for outcome_label, y, p_rad, p_combo in [
    ("Outcome 1 -- Mild vs No HE",         y_o1, p_o1_rad, p_o1_combo),
    ("Outcome 2 -- Mod-Severe vs Mild HE",  y_o2, p_o2_rad, p_o2_combo),
]:
    for model_label in ["Radiomics", "Combined"]:
        mask = (
            (df_metrics["Outcome"] == outcome_label) &
            (df_metrics["Model"]   == model_label)
        )
        z_rc, p_rc = delong_test(y, p_combo, p_rad)
        sig_rc = ("***" if p_rc < 0.001 else
                  "**"  if p_rc < 0.01  else
                  "*"   if p_rc < 0.05  else "ns")
        df_metrics.loc[mask, "DeLong_rad_combo_z"] = z_rc
        df_metrics.loc[mask, "DeLong_rad_combo_p"] = p_rc
        df_metrics.loc[mask, "Sig_rad_combo"]       = sig_rc


# =============================================================================
# CALIBRATION RECALIBRATION
# =============================================================================
def recalibrate_oof(y, p_raw):
    from sklearn.linear_model import LogisticRegression as LR
    eps    = 1e-6
    p_clip = np.clip(p_raw, eps, 1 - eps)
    logit  = np.log(p_clip / (1 - p_clip))
    lr_cal = LR(penalty=None, solver="lbfgs", max_iter=1000)
    lr_cal.fit(logit.reshape(-1, 1), y)
    a     = float(lr_cal.intercept_[0])
    b     = float(lr_cal.coef_[0][0])
    p_cal = 1 / (1 + np.exp(-(a + b * logit)))
    return p_cal, a, b


print("\n  Calibration recalibration:")
recal_rows = []
for outcome_label, y, model_label, p_raw in [
    ("Outcome 0 -- Cirrhosis No HE vs Healthy Controls",
     y_o0, "Radiomics", p_o0_rad),
    ("Outcome 1 -- Mild vs No HE",         y_o1, "Radiomics", p_o1_rad),
    ("Outcome 1 -- Mild vs No HE",         y_o1, "Combined",  p_o1_combo),
    ("Outcome 2 -- Mod-Severe vs Mild HE",  y_o2, "Radiomics", p_o2_rad),
    ("Outcome 2 -- Mod-Severe vs Mild HE",  y_o2, "Combined",  p_o2_combo),
]:
    p_cal, a_fit, b_fit = recalibrate_oof(y, p_raw)
    slope_pre = df_metrics.loc[
        (df_metrics["Outcome"] == outcome_label) &
        (df_metrics["Model"]   == model_label),
        "Cal_Slope"
    ].values[0]
    auc_cal = roc_auc_score(y, p_cal)
    recal_rows.append({
        "Outcome":        outcome_label,
        "Model":          model_label,
        "Slope_pre":      slope_pre,
        "Intercept_fit":  a_fit,
        "Slope_fit":      b_fit,
        "AUC_post_recal": auc_cal,
    })
    print(f"    {outcome_label[:22]} | {model_label:<12} "
          f"pre={slope_pre:.3f} b={b_fit:.3f} a={a_fit:.3f} "
          f"AUC_post={auc_cal:.3f}")

df_recal = pd.DataFrame(recal_rows)
df_recal.to_csv(CSV_DIR / "calibration_recalibration.csv", index=False)
df_metrics.to_csv(CSV_DIR / "performance_metrics.csv", index=False)
print(f"\n  [OK] performance_metrics.csv")
print(f"  [OK] calibration_recalibration.csv")


# =============================================================================
# INLINE HTML TABLE
# =============================================================================
def _fmt_p(p):
    v = float(p)
    if v != v:
        return "--"
    return "< 0.001" if v < 0.001 else "< 0.01" if v < 0.01 else f"{v:.3f}"


th   = ("padding:6px 14px; text-align:center; font-weight:bold; "
        "border-top:2px solid #000; border-bottom:1px solid #000; "
        "background:#f7f7f7; color:#000;")
th_l = th.replace("center", "left")
td   = "padding:5px 14px; text-align:center; color:#000; background:white;"

html = ("<div style=\"font-family:'Helvetica Neue',Arial,sans-serif;"
        "font-size:12px; max-width:980px; margin-top:12px;"
        "background:white; padding:20px; border-radius:5px; color:#000;\">")

for outcome_label in df_metrics["Outcome"].unique():
    df_out = df_metrics[df_metrics["Outcome"] == outcome_label]
    html += (
        f"<p style=\"margin:16px 0 4px 0; font-size:13px; color:#000;\">"
        f"<b>{outcome_label}</b></p>"
        "<table style=\"border-collapse:collapse; width:100%;"
        "margin-bottom:20px; background:white;\">"
        "<thead><tr>"
        f"<th style=\"{th_l}\">Model</th>"
        f"<th style=\"{th}\">AUC (95% CI)</th>"
        f"<th style=\"{th}\">Brier (95% CI)</th>"
        f"<th style=\"{th}\">Cal. Slope (95% CI)</th>"
        f"<th style=\"{th}\">DeLong p</th>"
        f"<th style=\"{th}\">Sig.</th>"
        "</tr></thead><tbody>"
    )
    for _, r in df_out.iterrows():
        color = MODEL_COLORS.get(r.Model, "#000000")
        bo = "<b>" if r.Model == "Combined" else ""
        bc = "</b>" if r.Model == "Combined" else ""
        td_m = (f"padding:5px 14px; text-align:left;"
                f"color:{color}; background:white;")
        html += (
            f"<tr style=\"border-bottom:0.5px solid #ccc;\">"
            f"<td style=\"{td_m}\">{bo}{r.Model}{bc}</td>"
            f"<td style=\"{td}\">{bo}{r.AUC:.3f} "
            f"({r.AUC_CI_lo:.3f}-{r.AUC_CI_hi:.3f}){bc}</td>"
            f"<td style=\"{td}\">{r.Brier:.3f} "
            f"({r.Brier_CI_lo:.3f}-{r.Brier_CI_hi:.3f})</td>"
            f"<td style=\"{td}\">{r.Cal_Slope:.3f} "
            f"({r.Slope_CI_lo:.3f}-{r.Slope_CI_hi:.3f})</td>"
            f"<td style=\"{td}\">{_fmt_p(r.DeLong_p)}</td>"
            f"<td style=\"{td}\">{r.Sig}</td>"
            "</tr>"
        )
    html += (
        "</tbody><tfoot><tr style=\"border-top:2px solid #000;\">"
        f"<td colspan=\"6\" style=\"padding:8px 14px; font-size:11px;"
        f"color:#555; line-height:1.5;\">"
        f"AUC: 95% bootstrap CI (n={N_BOOT}, percentile method). "
        f"DeLong p: two-tailed vs MELD reference "
        f"(-- = no MELD comparator for O0). "
        f"* p&lt;0.05 ** p&lt;0.01 *** p&lt;0.001."
        "</td></tr></tfoot></table>"
    )

html += "</div>"
display(HTML(html))

print("\nCell 9 -- Performance metrics complete.")


In [ ]:
# =============================================================================
# Cell 9 -- Calibration reliability diagrams (SF1)
# TRIPOD-AI items: 10a, 12
# =============================================================================
# Supplementary Figure SF1
# TRIPOD-AI items: 10a, 12
# =============================================================================
#            Supplementary Figure SF_calibration
# =============================================================================
# Produces reliability (calibration) diagrams for all model x outcome
# combinations, comparing pre- and post-recalibration predictions.
# Recalibrated probabilities are taken from the logistic slope/intercept
# adjustment computed in Cell 5.
#
# Layout: 2 rows (outcomes) x 3 cols (models), 6 panels total.
# Each panel shows:
#   - Pre-calibration reliability curve (solid, colour-coded by model)
#   - Post-recalibration reliability curve (dashed, same colour)
#   - Perfect calibration diagonal (gray dotted)
#   - Brier score annotation (pre / post)
#
# Requires: df_metrics, df_recal (Cell 5), p_o1_*/p_o2_* (Cell 4).
#
# Outputs:
#   outputs/figures/shap/calibration_reliability_panel_color.tiff/.pdf/_gray.*
#
# TRIPOD-AI items covered:
#   - 10a : calibration assessment (reliability diagram)
#   - 12  : recalibration method documented (van Calster et al. 2019)
# =============================================================================

# --- Execution guard ---
if "df_recal" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )

from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

TASKS_CAL = [
    ("Outcome 1 — Mild vs No HE",
     y_o1,
     [("MELD", p_o1_meld), ("Radiomics", p_o1_rad), ("Combined", p_o1_combo)]),
    ("Outcome 2 — Mod-Severe vs Mild HE",
     y_o2,
     [("MELD", p_o2_meld), ("Radiomics", p_o2_rad), ("Combined", p_o2_combo)]),
]

def recal_probs(y, p_raw):
    """Apply logistic recalibration (as in Cell 5 recalibrate_oof)."""
    eps = 1e-6
    p_c = np.clip(p_raw, eps, 1 - eps)
    logit_raw = np.log(p_c / (1 - p_c))
    from sklearn.linear_model import LogisticRegression as LR
    lr = LR(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(logit_raw.reshape(-1, 1), y)
    logit_cal = lr.intercept_[0] + lr.coef_[0][0] * logit_raw
    return 1 / (1 + np.exp(-logit_cal))

N_BINS = 10
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
fig.suptitle(
    "Calibration Reliability Diagrams — OOF Predictions\n"
    "Solid = pre-recalibration  |  Dashed = post-recalibration",
    fontsize=12, fontweight="normal",
)

for row_i, (outcome_label, y, model_list) in enumerate(TASKS_CAL):
    for col_i, (model_label, p_raw) in enumerate(model_list):
        ax = axes[row_i, col_i]
        color = MODEL_COLORS[model_label]

        # Pre-recalibration
        frac_pos, mean_pred = calibration_curve(y, p_raw, n_bins=N_BINS,
                                                 strategy="uniform")
        brier_pre = brier_score_loss(y, p_raw)
        ax.plot(mean_pred, frac_pos, "o-", color=color, lw=1.6,
                label=f"Pre  (Brier={brier_pre:.3f})", markersize=4)

        # Post-recalibration (MELD not recalibrated — no improvement expected)
        if model_label != "MELD":
            p_cal = recal_probs(y, p_raw)
            frac_pos_c, mean_pred_c = calibration_curve(
                y, p_cal, n_bins=N_BINS, strategy="uniform"
            )
            brier_post = brier_score_loss(y, p_cal)
            ax.plot(mean_pred_c, frac_pos_c, "o--", color=color,
                    lw=1.6, alpha=0.65,
                    label=f"Post (Brier={brier_post:.3f})", markersize=4)

        ax.plot([0, 1], [0, 1], ":", color="#aaaaaa", lw=1.0, label="Perfect")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel("Mean predicted probability", fontsize=9)
        ax.set_ylabel("Observed fraction of positives", fontsize=9)
        ax.set_title(
            f"({chr(65 + row_i * 3 + col_i)}) {model_label}\n{outcome_label}",
            fontsize=9, fontweight="normal",
        )
        ax.legend(fontsize=7.5, frameon=False, loc="upper left")
        ax.grid(alpha=0.2, linestyle=":")

save_figure(fig, "calibration_reliability_panel", CAL_DIR)
print("Cell 5b -- Calibration reliability panel complete.")


In [ ]:
# =============================================================================
# Cell 10 -- Decision Curve Analysis -- Figure 4
# TRIPOD-AI items: 13a
# =============================================================================
# O1 and O2 only. DCA not applicable to O0 (no clinical threshold).
# =============================================================================
# Figure 4 -- (A) Outcome 1  |  (B) Outcome 2
# TRIPOD-AI item: 13a
# =============================================================================
#           Figure 4 — (A) Outcome 1  |  (B) Outcome 2  [separate files]
# =============================================================================
# Computes and plots net benefit curves for all three models across a
# clinically relevant threshold range (0 to 0.5), following the DCA
# framework of Vickers & Elkin (Med Decis Making 2006).
#
# Layout: two independent figures (to be assembled as panel A+B by author).
#   Panel (A): Outcome 1 — Mild vs No HE
#   Panel (B): Outcome 2 — Mod-Severe vs Mild HE
#
# Outputs:
#   outputs/figures/dca/DCA_outcome1_color.tiff/.pdf/_gray.*
#   outputs/figures/dca/DCA_outcome2_color.tiff/.pdf/_gray.*
#
# TRIPOD-AI items covered:
#   - 13a : clinical utility assessment (net benefit)
# =============================================================================

# --- Execution guard ---
if "p_o1_meld" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )


# =============================================================================
# HELPER — net benefit computation
# =============================================================================
def net_benefit(y, probs, thresholds):
    y = np.asarray(y).flatten()
    probs = np.asarray(probs).flatten()
    n = len(y)
    nb = []
    for t in thresholds:
        if t >= 0.99:
            nb.append(0.0)
        else:
            tp = np.sum((probs >= t) & (y == 1))
            fp = np.sum((probs >= t) & (y == 0))
            nb.append((tp / n) - (fp / n) * (t / (1.0 - t)))
    return np.array(nb)


# =============================================================================
# HELPER — single DCA figure
# =============================================================================
def plot_dca(panel_label, outcome_label, y,
             p_meld, p_rad, p_combo, fname):
    """
    Render and save a Decision Curve Analysis figure for one outcome.

    Parameters
    ----------
    panel_label   : str, "(A)" or "(B)"
    outcome_label : str, displayed in figure title
    y             : np.ndarray, binary outcome
    p_meld        : np.ndarray, OOF probabilities -- MELD model
    p_rad         : np.ndarray, OOF probabilities -- Radiomics model
    p_combo       : np.ndarray, OOF probabilities -- Combined model
    fname         : str, output filename stem (without extension)
    """
    thresholds = np.linspace(0.01, 0.50, 500)

    nb_meld  = net_benefit(y, p_meld,  thresholds)
    nb_rad   = net_benefit(y, p_rad,   thresholds)
    nb_combo = net_benefit(y, p_combo, thresholds)

    prevalence = float(np.mean(y))
    nb_all = prevalence - (1.0 - prevalence) * (
        thresholds / (1.0 - thresholds)
    )

    fig, ax = plt.subplots(figsize=(7, 7.5))

    # --- Reference lines ---
    line_all, = ax.plot(
        thresholds, nb_all,
        linestyle="--", linewidth=1.2,
        color=CLR_TREAT_ALL, zorder=1,
    )
    line_none = ax.axhline(
        0, color="#333333", linewidth=0.9, zorder=1
    )

    # --- Model curves ---
    model_lines = []
    for model_label, p_model in zip(
        MODEL_LABELS, [p_meld, p_rad, p_combo]
    ):
        nb_vals = net_benefit(y, p_model, thresholds)
        line, = ax.plot(
            thresholds, nb_vals,
            color=DCA_MODEL_COLORS[model_label],
            linestyle=MODEL_LINESTYLES[model_label],
            linewidth=1.8, zorder=3,
        )
        model_lines.append((line, model_label))

    # --- Legend: two rows, below x-axis label ---
    leg1 = ax.legend(
        handles=[l for l, _ in model_lines],
        labels=[lbl for _, lbl in model_lines],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.14),
        ncol=3, frameon=False, fontsize=10, columnspacing=1.2,
    )
    ax.add_artist(leg1)
    ax.legend(
        handles=[line_all, line_none],
        labels=["Treat all", "Treat none"],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.21),
        ncol=2, frameon=False, fontsize=10, columnspacing=1.2,
    )

    # --- Axes formatting ---
    ax.set_xlim(0.0, 0.50)
    y_max = max(float(nb_combo.max()), prevalence)
    ax.set_ylim(-0.02, y_max + 0.05)
    ax.set_xlabel("Probability threshold", fontsize=11, labelpad=10)
    ax.set_ylabel("Net benefit", fontsize=11, labelpad=10)
    ax.set_title(
        f"{panel_label} Decision Curve Analysis\n{outcome_label}",
        fontweight="normal", fontsize=12, pad=20,
    )
    ax.yaxis.grid(True, linestyle=":", alpha=0.3, color="#aaaaaa")
    ax.grid(False)

    plt.subplots_adjust(bottom=0.28)
    save_figure(fig, fname, DCA_DIR)


# =============================================================================
# RENDER — Outcome 1  (panel A)
# =============================================================================
print("DCA -- Outcome 1")
plot_dca(
    panel_label="(A)",
    outcome_label="Outcome 1: Mild vs No HE",
    y=y_o1,
    p_meld=p_o1_meld, p_rad=p_o1_rad, p_combo=p_o1_combo,
    fname="DCA_outcome1",
)

# =============================================================================
# RENDER — Outcome 2  (panel B)
# =============================================================================
print("\nDCA -- Outcome 2")
plot_dca(
    panel_label="(B)",
    outcome_label="Outcome 2: Mod-Severe vs Mild HE",
    y=y_o2,
    p_meld=p_o2_meld, p_rad=p_o2_rad, p_combo=p_o2_combo,
    fname="DCA_outcome2",
)

print("\nCell 7 -- DCA complete.")


In [ ]:
# =============================================================================
# Cell 11 -- SHAP feature importance -- Figure 3
# TRIPOD-AI items: 10c, 13a
# =============================================================================
#           Figure 3 — (A) Outcome 1  |  (B) Outcome 2  [separate files]
# =============================================================================
# Computes SHAP values for the Radiomics-only L1 logistic regression
# model trained on the full dataset for interpretability.
#
# Method — LinearExplainer (interventional):
#   Exact SHAP values via phi_j = w_j * (x_j - E[x_j]).
#
# Layout: two independent beeswarm figures (to be assembled as panel A+B).
#   Panel (A): Outcome 1 — Mild vs No HE
#   Panel (B): Outcome 2 — Mod-Severe vs Mild HE
#
# Outputs:
#   outputs/figures/shap/shap_beeswarm_outcome1_color.tiff/.pdf/_gray.*
#   outputs/figures/shap/shap_beeswarm_outcome2_color.tiff/.pdf/_gray.*
#   outputs/tables/shap_features_outcome1.csv
#   outputs/tables/shap_features_outcome2.csv
#
# TRIPOD-AI items covered:
#   - 10c : model interpretability
#   - 13a : predictor contribution reporting (SHAP)
#
# Fix log (v5 — definitive):
#   - Bug 1 (Panel A): TOP_N_SHAP=20 caused non-selected features (coef=0,
#     SHAP=0) to appear in the beeswarm. top_idx now restricted to
#     LASSO-selected features only (coef != 0), sorted by mean |SHAP|.
#   - Bug 2 (Panel B): X_o2_rad is overwritten by Cell 8 EPV FIX to contain
#     only the 5 bootstrap-stable columns; rad_cols_o2 (also set in Cell 8)
#     holds the corresponding 5 names. The original code used the global
#     rad_cols (30 names) as feature_names for both outcomes, causing an
#     IndexError and feature-name misalignment for O2. Fix: pass
#     rad_cols_local per outcome — rad_cols for O1, rad_cols_o2 for O2.
#     An assert guards against future shape mismatches.
# =============================================================================

# --- Execution guard ---
if "Cs_o1_rad" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )
if "rad_cols_o2" not in dir():
    raise RuntimeError(
        "rad_cols_o2 is not defined. Cell 8 EPV FIX must have run.\n"
        "Re-execute Cell 8 before this cell."
    )

TOP_N_SHAP = 20   # upper bound — actual display = min(n_selected, TOP_N_SHAP)


# =============================================================================
# HELPER — fit full-data model, compute SHAP, produce beeswarm + CSV
# =============================================================================
def run_shap(panel_label, X_rad, y, best_Cs, outcome_label, fname_stem,
             rad_cols_local):
    """
    Fit a full-data radiomics pipeline, compute SHAP values via
    LinearExplainer, produce a beeswarm figure and save feature CSV.

    Parameters
    ----------
    panel_label    : str, "(A)" or "(B)"
    X_rad          : np.ndarray (n, p_local), raw radiomic features (unscaled).
                     p_local may be < len(rad_cols) if X_rad was pre-filtered
                     by the Cell 8 EPV FIX (Outcome 2 only).
    y              : np.ndarray (n,), binary outcome
    best_Cs        : list of float, best C per outer fold from Cell 8
    outcome_label  : str, displayed in figure title
    fname_stem     : str, filename stem, e.g. "outcome1"
    rad_cols_local : list of str, length p_local — column names for X_rad.
                     Use rad_cols for O1, rad_cols_o2 for O2.

    Returns
    -------
    shap_df : pd.DataFrame, ranked feature importances
    """
    p_local = X_rad.shape[1]
    assert len(rad_cols_local) == p_local, (
        f"rad_cols_local has {len(rad_cols_local)} entries "
        f"but X_rad has {p_local} columns. "
        f"Check that the correct rad_cols_local is passed for this outcome."
    )

    best_C = float(np.median(best_Cs))
    print(f"  Best C (median of outer folds): {best_C:.5f}")
    print(f"  X_rad shape: {X_rad.shape}  |  features: {p_local}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l1", C=best_C, solver="liblinear",
            class_weight="balanced", max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_rad, y)

    X_scaled = pipe.named_steps["scaler"].transform(X_rad)
    model    = pipe.named_steps["clf"]
    coefs    = model.coef_[0]                          # shape (p_local,)

    n_selected = int(np.sum(coefs != 0))
    print(f"  LASSO-selected features: {n_selected} / {p_local}")

    # --- SHAP via LinearExplainer (interventional) ---
    explainer   = shap.LinearExplainer(
        model, X_scaled, feature_perturbation="interventional"
    )
    shap_values = explainer.shap_values(X_scaled)      # shape (n, p_local)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)   # shape (p_local,)

    # Feature names scoped to X_rad columns — never index into global rad_cols
    feature_names = np.array(rad_cols_local)            # shape (p_local,)

    # Full ranking for CSV (all p_local features)
    ranked_idx = np.argsort(mean_abs_shap)[::-1]

    # Beeswarm: only LASSO-selected features, sorted by mean |SHAP| desc
    selected_idx    = np.where(coefs != 0)[0]
    selected_sorted = selected_idx[np.argsort(mean_abs_shap[selected_idx])[::-1]]
    top_idx   = selected_sorted[:TOP_N_SHAP]
    n_display = len(top_idx)
    print(f"  Features shown in beeswarm: {n_display}")

    shap_df = pd.DataFrame({
        "Feature":       feature_names[ranked_idx],
        "Mean_Abs_SHAP": mean_abs_shap[ranked_idx],
        "LASSO_Coef":    coefs[ranked_idx],
        "Selected":      coefs[ranked_idx] != 0,
    })
    csv_path = CSV_DIR / f"shap_features_{fname_stem}.csv"
    shap_df.to_csv(csv_path, index=False)
    print(f"  [OK] Feature table -> {csv_path.name}")

    # --- Beeswarm figure ---
    fig_bee, ax_bee = plt.subplots(figsize=(10, 8))
    plt.sca(ax_bee)

    shap.summary_plot(
        shap_values[:, top_idx],
        X_scaled[:, top_idx],
        feature_names=feature_names[top_idx].tolist(),
        plot_type="dot",
        max_display=n_display,
        show=False,
        color_bar=True,
        color_bar_label="Feature value",
    )
    ax_bee.set_title(
        f"{panel_label} SHAP Feature Importance — {outcome_label}\n"
        f"(Radiomics model, {n_display} LASSO-selected feature"
        f"{'s' if n_display != 1 else ''})",
        fontsize=12, fontweight="normal", pad=14,
    )
    plt.tight_layout()
    save_figure(fig_bee, f"shap_beeswarm_{fname_stem}", SHA_DIR)

    return shap_df


# =============================================================================
# RUN — Outcome 1  (panel A)
# O1: X_o1_rad has all 30 radiomic columns → use global rad_cols
# =============================================================================
print("=" * 60)
print("SHAP -- Outcome 1: Mild vs No HE")
print("=" * 60)
shap_df_o1 = run_shap(
    panel_label="(A)",
    X_rad=X_o1_rad, y=y_o1, best_Cs=Cs_o1_rad,
    outcome_label="Outcome 1 — Mild vs No HE",
    fname_stem="outcome1",
    rad_cols_local=rad_cols,       # 30 features
)

# =============================================================================
# RUN — Outcome 2  (panel B)
# O2: X_o2_rad was overwritten by Cell 8 EPV FIX to p_stable columns;
#     rad_cols_o2 holds the matching names (also set in Cell 8).
# =============================================================================
print("\n" + "=" * 60)
print("SHAP -- Outcome 2: Moderate-Severe vs Mild HE")
print("=" * 60)
shap_df_o2 = run_shap(
    panel_label="(B)",
    X_rad=X_o2_rad, y=y_o2, best_Cs=Cs_o2_rad,
    outcome_label="Outcome 2 — Moderate-Severe vs Mild HE",
    fname_stem="outcome2",
    rad_cols_local=rad_cols_o2,    # p_stable features (set in Cell 8)
)

# =============================================================================
# INLINE SUMMARY — top 5 selected features per outcome
# =============================================================================
th  = ("padding:5px 12px; text-align:center; font-weight:bold; "
       "border-top:2px solid #111; border-bottom:1px solid #111; "
       "background:#f7f7f7; color:#111;")
th_l = th.replace("center", "left")
td   = "padding:5px 12px; text-align:center; color:#333;"
td_l = td.replace("center", "left")

html = (
    "<div style=\"font-family:'Helvetica Neue',Arial,sans-serif;"
    "font-size:12px; max-width:820px; margin-top:12px;"
    "background:white; padding:15px; border-radius:5px;\">"
    "<p style=\"font-size:14px; margin:0 0 8px 0; color:#111;\">"
    "<b>Top 5 SHAP features — Radiomics model</b></p>"
)
for label, shap_df in [
    ("Outcome 1 — Mild vs No HE",                shap_df_o1),
    ("Outcome 2 — Moderate-to-Severe vs Mild HE", shap_df_o2),
]:
    html += (
        f"<p style=\"margin:12px 0 4px 0; color:#111;\"><b>{label}</b></p>"
        f"<table style=\"border-collapse:collapse; width:100%;"
        f"margin-bottom:12px; background:white;\">"
        f"<thead><tr>"
        f"<th style=\"{th_l}\">Rank</th>"
        f"<th style=\"{th_l}\">Feature</th>"
        f"<th style=\"{th}\">Mean |SHAP|</th>"
        f"<th style=\"{th}\">LASSO coef.</th>"
        f"<th style=\"{th}\">Selected</th>"
        f"</tr></thead><tbody>"
    )
    selected_rows = shap_df[shap_df["Selected"]].head(5)
    for rank, (_, row) in enumerate(selected_rows.iterrows(), 1):
        feat = row["Feature"].replace("original_", "")
        sel  = "yes" if row["Selected"] else "--"
        html += (
            f"<tr style=\"border-bottom:0.5px solid #e8e8e8;\">"
            f"<td style=\"{td_l}\">{rank}</td>"
            f"<td style=\"{td_l}\">{feat}</td>"
            f"<td style=\"{td}\">{row['Mean_Abs_SHAP']:.4f}</td>"
            f"<td style=\"{td}\">{row['LASSO_Coef']:+.4f}</td>"
            f"<td style=\"{td}\">{sel}</td>"
            f"</tr>"
        )
    html += "</tbody></table>"
html += "</div>"
ipy_display(HTML(html))

print("\nCell 13 -- SHAP complete.")

In [ ]:
# =============================================================================
# Cell 12 -- Bootstrap LASSO feature stability (SF2)
# TRIPOD-AI items: 10c, 15
# =============================================================================
# Supplementary Figure SF2
# TRIPOD-AI items: 10c, 15
# =============================================================================
#            Supplementary Figure SF_stability + SDC ST1 addendum
# =============================================================================
# Evaluates the stability of LASSO-selected features by running the full
# LASSO pipeline (StandardScaler + L1 LogisticRegression, best C from Cell 4)
# on N_BOOT_STAB bootstrap resamples of the full internal cohort.
#
# For each resample:
#   - Fit pipeline on bootstrap sample.
#   - Record which features have non-zero coefficient (selected = True).
#
# Output:
#   selection_freq : proportion of bootstrap resamples in which each feature
#                   was selected (range 0.0–1.0).
#
# Stability thresholds:
#   >= 0.80 : highly stable (dark fill)
#   >= 0.50 : moderately stable (medium fill)
#   <  0.50 : unstable (light fill)
#
# Outputs:
#   outputs/figures/shap/lasso_stability_outcome*_color.tiff/.pdf/_gray.*
#   outputs/tables/lasso_stability_outcome1.csv
#   outputs/tables/lasso_stability_outcome2.csv
#
# TRIPOD-AI items covered:
#   - 10c : model interpretability (feature stability)
#   - 15  : feature stability data saved for reproducibility
# IBSI compliance: stability analysis applied only to PyRadiomics-extracted
#   IBSI-compliant features (prefix "original_", version 3.0.1).
# =============================================================================

# --- Execution guard ---
if "Cs_o1_rad" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )

N_BOOT_STAB = 1000

def bootstrap_lasso_stability(X_rad, y, best_Cs, feature_names,
                               n_boot=N_BOOT_STAB, seed=BOOT_SEED):
    """
    Bootstrap feature selection frequency for LASSO logistic regression.

    Parameters
    ----------
    X_rad        : np.ndarray (n, p), raw radiomic features (unscaled)
    y            : np.ndarray (n,),   binary outcome
    best_Cs      : list of float, outer-fold best C values from Cell 4
    feature_names: list of str, feature names (length p)
    n_boot       : int, number of bootstrap resamples
    seed         : int, random seed

    Returns
    -------
    df_stab : pd.DataFrame with columns [Feature, Selection_Freq, Stable_80, Stable_50]
    """
    best_C = float(np.median(best_Cs))
    rng = np.random.default_rng(seed)
    sel_counts = np.zeros(len(feature_names), dtype=int)
    valid_boots = 0

    for _ in range(n_boot):
        idx_b = rng.integers(0, len(y), len(y))
        yb = y[idx_b]
        if len(np.unique(yb)) < 2:
            continue
        Xb = X_rad[idx_b]
        try:
            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    penalty="l1", C=best_C, solver="liblinear",
                    class_weight="balanced", max_iter=2000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(Xb, yb)
            coefs = pipe.named_steps["clf"].coef_[0]
            sel_counts += (coefs != 0).astype(int)
            valid_boots += 1
        except Exception:
            continue

    sel_freq = sel_counts / max(valid_boots, 1)
    df_stab = pd.DataFrame({
        "Feature":       feature_names,
        "Selection_Freq": sel_freq,
        "Stable_80":     sel_freq >= 0.80,
        "Stable_50":     sel_freq >= 0.50,
    }).sort_values("Selection_Freq", ascending=False).reset_index(drop=True)
    df_stab["Rank"] = df_stab.index + 1
    return df_stab, valid_boots


def plot_stability(df_stab, outcome_label, fname_stem, valid_boots):
    """Bar chart of feature selection frequency with 80%/50% threshold lines."""
    df_plot = df_stab[df_stab["Selection_Freq"] > 0].copy()
    if df_plot.empty:
        print(f"  No feature selected in any bootstrap resample for {outcome_label}")
        return

    colors_bar = []
    for freq in df_plot["Selection_Freq"]:
        if freq >= 0.80:
            colors_bar.append("#1D9E75")   # teal — highly stable
        elif freq >= 0.50:
            colors_bar.append("#EF9F27")   # amber — moderate
        else:
            colors_bar.append("#B4B2A9")   # gray — unstable

    feat_clean = [f.replace("original_", "").replace("_", " ")
                  for f in df_plot["Feature"]]

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(df_plot))))
    bars = ax.barh(range(len(df_plot)), df_plot["Selection_Freq"],
                   color=colors_bar, edgecolor="white", alpha=0.9)
    ax.axvline(0.80, color="#1D9E75", lw=1.2, ls="--",
               label="Stability threshold 80%")
    ax.axvline(0.50, color="#EF9F27", lw=1.2, ls="--",
               label="Stability threshold 50%")

    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels(feat_clean, fontsize=9)
    ax.set_xlabel("Bootstrap selection frequency", fontsize=10)
    ax.set_xlim(0, 1.05)
    ax.set_title(
        f"LASSO Feature Selection Frequency — {outcome_label}\n"
        f"(n = {valid_boots} valid bootstrap resamples of {N_BOOT_STAB})",
        fontsize=11, fontweight="normal", pad=14,
    )
    ax.legend(fontsize=9, frameon=False, loc="lower right")
    ax.grid(axis="x", alpha=0.25, linestyle=":")
    ax.invert_yaxis()
    plt.tight_layout()
    save_figure(fig, f"lasso_stability_{fname_stem}", SHA_DIR)


# =============================================================================
# RUN — Outcome 1
# =============================================================================
print("=" * 60)
print(f"Bootstrap LASSO stability — Outcome 1  (n_boot={N_BOOT_STAB})")
print("=" * 60)
df_stab_o1, valid_o1 = bootstrap_lasso_stability(
    X_rad=X_o1_rad, y=y_o1, best_Cs=Cs_o1_rad,
    feature_names=rad_cols, n_boot=N_BOOT_STAB,
)
print(f"  Valid resamples: {valid_o1} / {N_BOOT_STAB}")
print(f"  Features stable >=80%: {(df_stab_o1['Stable_80']).sum()}")
print(f"  Features stable >=50%: {(df_stab_o1['Stable_50']).sum()}")
print(df_stab_o1[df_stab_o1["Selection_Freq"] > 0].to_string(index=False))

df_stab_o1.to_csv(CSV_DIR / "lasso_stability_outcome1.csv", index=False)
plot_stability(df_stab_o1, "Outcome 1 — Mild vs No HE", "outcome1", valid_o1)
print(f"  [OK] lasso_stability_outcome1.csv saved")

# =============================================================================
# RUN — Outcome 2
# =============================================================================
print("\n" + "=" * 60)
print(f"Bootstrap LASSO stability — Outcome 2  (n_boot={N_BOOT_STAB})")
print("=" * 60)
df_stab_o2, valid_o2 = bootstrap_lasso_stability(
    X_rad=X_o2_rad, y=y_o2, best_Cs=Cs_o2_rad,
    feature_names=rad_cols, n_boot=N_BOOT_STAB,
)
print(f"  Valid resamples: {valid_o2} / {N_BOOT_STAB}")
print(f"  Features stable >=80%: {(df_stab_o2['Stable_80']).sum()}")
print(f"  Features stable >=50%: {(df_stab_o2['Stable_50']).sum()}")
print(df_stab_o2[df_stab_o2["Selection_Freq"] > 0].to_string(index=False))

df_stab_o2.to_csv(CSV_DIR / "lasso_stability_outcome2.csv", index=False)
plot_stability(df_stab_o2, "Outcome 2 — Mod-Severe vs Mild HE", "outcome2", valid_o2)
print(f"  [OK] lasso_stability_outcome2.csv saved")

print("\nCell 8b -- LASSO bootstrap stability complete.")


In [ ]:
# =============================================================================
# Cell 13 -- AUC grouped bar chart -- Figure 2
# TRIPOD-AI items: 11
# =============================================================================
# Figure 2
# TRIPOD-AI item: 11
# =============================================================================
# Displays AUC point estimates and 95% bootstrap CI for all six
# model x outcome combinations in a single grouped bar chart.
# Displays AUC point estimates and 95% bootstrap CI for all six
# model x outcome combinations in a single grouped bar chart.
#
# Layout:
#   Two bar groups (one per outcome), three bars per group (one per model).
#   Error bars: 95% bootstrap CI (from Cell 5 df_metrics).
#   AUC value annotated above each CI bar (format: 0.XXX).
#   Legend: horizontal, below x-axis label (outside plot area).
#   No significance brackets or p-value annotations (per journal guidelines).
#
# Dual output via save_figure() (Cell 1):
#   _color.tiff / _color.pdf  and  _gray.tiff / _gray.pdf
#   Grayscale: MODEL_COLORS_GRAY fills + MODEL_HATCHES patterns.
#
# TRIPOD-AI items covered:
#   - 11 : uncertainty quantification (bootstrap CI)
#   - 11 : uncertainty quantification (bootstrap CI)
# =============================================================================

# --- Execution guard ---
if "df_metrics" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )


# =============================================================================
# LAYOUT CONSTANTS
# =============================================================================
OUTCOMES_PLOT = [
    ("Outcome 1 -- mild HE vs no overt HE",         "Outcome 1"),
    ("Outcome 2 -- moderate-to-severe HE vs mild HE",  "Outcome 2"),
]

BAR_WIDTH     = 0.22
GROUP_GAP     = 0.90
N_MODELS      = len(MODEL_LABELS)
GROUP_CENTRES = np.array([0.0, GROUP_GAP + N_MODELS * BAR_WIDTH])
OFFSETS       = np.array([-BAR_WIDTH, 0.0, BAR_WIDTH])


# =============================================================================
# FIGURE
# =============================================================================
fig, ax = plt.subplots(figsize=(11, 8.5))
all_bar_tops = []

for g_idx, (outcome_str, outcome_key) in enumerate(OUTCOMES_PLOT):

    df_out = df_metrics[
        df_metrics["Outcome"].str.contains(outcome_key)
    ].reset_index(drop=True)

    group_x = GROUP_CENTRES[g_idx]

    for m_idx, model_label in enumerate(MODEL_LABELS):

        row = df_out[df_out["Model"] == model_label].iloc[0]
        auc   = float(row["AUC"])
        ci_lo = float(row["AUC_CI_lo"])
        ci_hi = float(row["AUC_CI_hi"])
        color = MODEL_COLORS[model_label]
        hatch = MODEL_HATCHES[model_label]
        bar_x = group_x + OFFSETS[m_idx]

        # Bar
        ax.bar(
            bar_x, auc,
            width=BAR_WIDTH * 0.88,
            color=color,
            hatch=hatch,
            edgecolor="white",
            linewidth=0.8,
            alpha=0.88,
            zorder=3,
            label=model_label if g_idx == 0 else "_nolegend_",
        )

        # 95% CI error bar
        ax.errorbar(
            bar_x, auc,
            yerr=[[auc - ci_lo], [ci_hi - auc]],
            fmt="none",
            color="#333333",
            capsize=4,
            linewidth=1.2,
            zorder=4,
        )

        # AUC value above CI
        ax.text(
            bar_x, ci_hi + 0.008,
            f"{auc:.3f}",
            ha="center", va="bottom",
            fontsize=9.5, fontweight="bold",
            color="#333333", zorder=5,
        )

        all_bar_tops.append(ci_hi)


# =============================================================================
# AXES FORMATTING
# =============================================================================
tick_x = [GROUP_CENTRES[g] + OFFSETS[1] for g in range(len(OUTCOMES_PLOT))]
ax.set_xticks(tick_x)
ax.set_xticklabels(
    ["Outcome 1\nmild HE vs no overt HE\n[grade 1 vs grade 0]",
    "Outcome 2\nmoderate-to-severe HE vs mild HE\n[grade ≥2 vs grade 1]"],
    fontsize=11,
)

y_max = max(all_bar_tops) if all_bar_tops else 1.0
ax.set_ylim(0.40, min(y_max + 0.10, 1.15))
ax.set_ylabel("AUC (95% Bootstrap CI)", fontsize=12)
ax.set_title(
    "Model Discrimination -- AUC with 95% Confidence Intervals",
    fontsize=13, fontweight="normal", pad=30,
)
ax.grid(axis="y", alpha=0.3, linestyle=":")

# Legend: horizontal, below x-axis (outside plot area)
ax.legend(
    title="Model",
    title_fontsize=10,
    fontsize=10,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=3,
    frameon=False,
)


plt.subplots_adjust(bottom=0.20)

save_figure(fig, "auc_barchart", BAR_DIR)

print("\nCell 9 -- AUC bar chart complete.")

In [ ]:
# =============================================================================
# Cell 14 -- Cross-system validation
# TRIPOD-AI items: 10d, 11, 12
# =============================================================================
# Leave-one-scanner-out design
# TRIPOD-AI items: 10d, 11, 12
# =============================================================================
#            Train on scanner A -> Test on scanner B (and vice versa)
# =============================================================================
# Evaluates the technical generalisability of each model across MRI
# acquisition systems using a leave-one-system-out design.
#
# Design:
#   Path 1: SCANNER_A (1.5T) -> SCANNER_B (3.0T)
#   Path 2: SCANNER_B (3.0T) -> SCANNER_A (1.5T)
#
# For each path x outcome x model:
#   1. Train on all subjects from the source scanner.
#   2. Apply to all subjects from the target scanner (no harmonisation).
#   3. AUC + 95% bootstrap CI (n = N_BOOT resamples).
#   4. Permutation test (n = N_PERM) vs H0: AUC = 0.5.
#   5. DeLong test: cross-system AUC vs within-scanner OOF AUC
#      on the same test-set subjects.
#   6. Brier score, calibration slope and intercept on test set.
#
# Hyperparameter:
#   Best C = median of outer-fold best_C from Cell 4 (BEST_C_MAP),
#   per model x outcome. No new hyperparameter search.
#
# Note: bootstrap_auc_ci(), delong_test(), and calibration_metrics()
#   are defined in Cell 5 and reused here without redefinition.
#
# Outputs:
#   outputs/tables/cross_system_validation.csv
#   [SUPPRESSED] outputs/figures/crosssystem/ (re-enable in RENDER block)
#   Inline HTML summary table
#
# TRIPOD-AI items covered:
#   - 10d : cross-system technical generalisation
#   - 11  : uncertainty quantification (bootstrap CI)
#   - 12  : permutation test and DeLong test
# =============================================================================

# --- Execution guard ---
if "BEST_C_MAP" not in dir() or "df_metrics" not in dir():
    raise RuntimeError(
        "Cells 6 and 8 have not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )


# =============================================================================
# PERMUTATION TEST helper (local — used only in this cell)
# bootstrap_auc_ci(), delong_test(), calibration_metrics() from Cell 5.
# =============================================================================
def permutation_test_auc(y_test, p_test, n_perm=N_PERM, seed=BOOT_SEED):
    """
    Two-sided permutation test: H0: AUC = 0.5 (random classifier).

    Parameters
    ----------
    y_test : np.ndarray, binary outcome on test set
    p_test : np.ndarray, predicted probabilities on test set
    n_perm : int, number of permutation iterations
    seed   : int, random seed

    Returns
    -------
    p_value : float
    """
    rng = np.random.default_rng(seed)
    obs_auc = roc_auc_score(y_test, p_test)
    null_auc = np.array([
        roc_auc_score(rng.permutation(y_test), p_test)
        for _ in range(n_perm)
    ])
    p_val = np.mean(np.abs(null_auc - 0.5) >= np.abs(obs_auc - 0.5))
    return float(p_val)
# Note: bootstrap_auc_ci(), delong_test(), and calibration_metrics()
#   are defined in Cell 5 and reused here without redefinition.
def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=BOOT_SEED):
    """
    Percentile bootstrap 95% CI for AUC.
    Returns (auc, ci_lo, ci_hi, ci_degenerate).
    ci_degenerate=True when all bootstrap resamples yield identical AUC
    (e.g. constant-prediction models such as MELD in cross-system paths).
    """
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, pb = y[idx], p[idx]
        if len(np.unique(yb)) < 2:
            continue
        aucs.append(roc_auc_score(yb, pb))
    aucs = np.array(aucs)
    auc_pt = roc_auc_score(y, p)
    if len(aucs) == 0 or np.std(aucs) < 1e-10:
        return auc_pt, auc_pt, auc_pt, True   # degenerate CI
    return (
        auc_pt,
        float(np.percentile(aucs, 2.5)),
        float(np.percentile(aucs, 97.5)),
        False,
    )


def delong_test(y, p1, p2):
    """Two-tailed DeLong test comparing AUC(p1) vs AUC(p2)."""
    y = np.asarray(y)
    p1 = np.asarray(p1)
    p2 = np.asarray(p2)
    n1 = int(y.sum())
    n0 = int((1 - y).sum())

    def _placement(y, p):
        pos = p[y == 1]
        neg = p[y == 0]
        n1l, n0l = len(pos), len(neg)
        V10 = np.array([
            (np.sum(pj > neg) + 0.5 * np.sum(pj == neg)) / n0l
            for pj in pos
        ])
        V01 = np.array([
            (np.sum(pi < pos) + 0.5 * np.sum(pi == pos)) / n1l
            for pi in neg
        ])
        return V10, V01

    V10_1, V01_1 = _placement(y, p1)
    V10_2, V01_2 = _placement(y, p2)
    S10 = np.cov(np.vstack([V10_1, V10_2]))
    S01 = np.cov(np.vstack([V01_1, V01_2]))
    var_diff = (
        S10[0, 0] / n1 + S01[0, 0] / n0
        + S10[1, 1] / n1 + S01[1, 1] / n0
        - 2 * (S10[0, 1] / n1 + S01[0, 1] / n0)
    )
    if var_diff <= 0:
        return 0.0, 1.0
    z = (V10_1.mean() - V10_2.mean()) / np.sqrt(var_diff)
    return float(z), float(2 * (1 - stats.norm.cdf(abs(z))))


def calibration_metrics(y, p):
    """Brier score, calibration slope and intercept."""
    eps = 1e-6
    p_c = np.clip(p, eps, 1 - eps)
    logit = np.log(p_c / (1 - p_c))
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    lr.fit(logit.reshape(-1, 1), y)
    return {
        "Brier":         float(brier_score_loss(y, p_c)),
        "Cal_Slope":     float(lr.coef_[0][0]),
        "Cal_Intercept": float(lr.intercept_[0]),
    }
# Statistical helpers are redefined here for cell-level independence —
# each cell is self-contained and runs correctly after a kernel restart
# provided Cells 0-6 have been executed.
# =============================================================================
# OUTCOME DATA MAP
# =============================================================================
OUTCOME_DATA = {
    "O1": {
        "label":   "Outcome 1 -- Mild vs No HE",
        "df":       df_o1,
        "y":        y_o1,
        "scanner":  scanner_o1,
        "X_meld":   X_o1_meld,
        "X_rad":    X_o1_rad,
        "X_combo":  X_o1_combo,
        "p_oof": {
            "MELD":      p_o1_meld,
            "Radiomics": p_o1_rad,
            "Combined":  p_o1_combo,
        },
    },
    "O2": {
        "label":   "Outcome 2 -- Moderate-Severe vs Mild HE",
        "df":       df_o2,
        "y":        y_o2,
        "scanner":  scanner_o2,
        "X_meld":   X_o2_meld,
        "X_rad":    X_o2_rad,
        "X_combo":  X_o2_combo,
        "p_oof": {
            "MELD":      p_o2_meld,
            "Radiomics": p_o2_rad,
            "Combined":  p_o2_combo,
        },
    },
}

X_KEY_MAP = {
    "MELD":      "X_meld",
    "Radiomics": "X_rad",
    "Combined":  "X_combo",
}


# =============================================================================
# MAIN — cross-system validation loop
# =============================================================================
rows_cs = []

for outcome_key, odata in OUTCOME_DATA.items():
    print(f"\n{'=' * 60}")
    print(odata["label"])
    print(f"{'=' * 60}")

    y_all = odata["y"]
    scanner_all = odata["scanner"]

    for src_scanner, tgt_scanner, path_label in CROSS_PATHS:
        print(f"\n  Path: {path_label}")

        # Match scanner labels (mapped to "1.5T"/"3.0T" in Cell 2)
        src_label = src_scanner.replace("Signa", "1.5T").replace(
            "Discovery", "3.0T"
        )
        tgt_label = tgt_scanner.replace("Signa", "1.5T").replace(
            "Discovery", "3.0T"
        )
        src_mask = scanner_all == src_label
        tgt_mask = scanner_all == tgt_label

        # Fallback: match raw scanner string labels
        if src_mask.sum() == 0:
            src_mask = np.array([
                s == src_scanner for s in odata["df"][COL_MR_SYSTEM].values
            ])
            tgt_mask = np.array([
                s == tgt_scanner for s in odata["df"][COL_MR_SYSTEM].values
            ])

        y_train = y_all[src_mask]
        y_test = y_all[tgt_mask]

        if len(np.unique(y_train)) < 2 or len(np.unique(y_test)) < 2:
            print(
                f"    [SKIP] Degenerate split "
                f"(train n={src_mask.sum()}, test n={tgt_mask.sum()})"
            )
            continue

        for model_label in MODEL_LABELS:
            X_all = odata[X_KEY_MAP[model_label]]
            X_train = X_all[src_mask]
            X_test = X_all[tgt_mask]
            best_C = BEST_C_MAP[(outcome_key, model_label)]

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    penalty="l1",
                    C=best_C,
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(X_train, y_train)
            p_test = pipe.predict_proba(X_test)[:, 1]

            auc, ci_lo, ci_hi, ci_degen = bootstrap_auc_ci(y_test, p_test)
            p_perm = permutation_test_auc(y_test, p_test)

            # DeLong: cross-system AUC vs within-scanner OOF on same subjects
            p_oof_test = odata["p_oof"][model_label][tgt_mask]
            _, p_delong_cs = delong_test(y_test, p_test, p_oof_test)

            cal = calibration_metrics(y_test, p_test)

            if p_perm < 0.001:
                sig_perm = "***"
            elif p_perm < 0.01:
                sig_perm = "**"
            elif p_perm < 0.05:
                sig_perm = "*"
            else:
                sig_perm = "ns"

            rows_cs.append({
                "Outcome":     odata["label"],
                "Path":        path_label,
                "Model":       model_label,
                "N_train":     int(src_mask.sum()),
                "N_test":      int(tgt_mask.sum()),
                "AUC":         auc,
                "AUC_CI_lo":   ci_lo,
                "AUC_CI_hi":   ci_hi,
                "Brier":       cal["Brier"],
                "Cal_Slope":   cal["Cal_Slope"],
                "Cal_Int":     cal["Cal_Intercept"],
                "Perm_p":      p_perm,
                "Sig_perm":    sig_perm,
                "DeLong_cs_p": p_delong_cs,
                "CI_degenerate": ci_degen,
            })

            print(
                f"    {model_label:<12} "
                f"AUC {auc:.3f} ({ci_lo:.3f}-{ci_hi:.3f})  "
                f"Brier {cal['Brier']:.3f}  "
                f"Perm p: {p_perm:.3f} {sig_perm}  "
                f"DeLong (vs OOF): {p_delong_cs:.3f}"
            )


# =============================================================================
# SAVE CSV
# =============================================================================
df_cs = pd.DataFrame(rows_cs)
df_cs.to_csv(CSV_DIR / "cross_system_validation.csv", index=False)
print(f"\n  [OK] Saved -> {(CSV_DIR / 'cross_system_validation.csv').name}")


# =============================================================================
# FIGURE helper — grouped bar chart per outcome
# =============================================================================
def plot_crosssystem_figure(outcome_label, df_cs_out, fname):
    """
    Grouped bar chart of cross-system AUCs with CI error bars,
    one panel per migration path, for a single outcome.
    Legend below x-axis, outside the plot area.

    Parameters
    ----------
    outcome_label : str
    df_cs_out     : pd.DataFrame, rows for this outcome from df_cs
    fname         : str, output filename stem
    """
    paths = df_cs_out["Path"].unique()
    n_paths = len(paths)

    fig, axes = plt.subplots(
        1, n_paths, figsize=(6 * n_paths, 6), sharey=True
    )
    if n_paths == 1:
        axes = [axes]

    fig.suptitle(
        f"Cross-System Validation -- {outcome_label}",
        fontsize=12, fontweight="normal", y=1.01,
    )

    bar_width = 0.22
    offsets = np.array([-bar_width, 0.0, bar_width])

    for ax, path in zip(axes, paths):
        df_path = df_cs_out[df_cs_out["Path"] == path]

        for m_idx, model_label in enumerate(MODEL_LABELS):
            row = df_path[df_path["Model"] == model_label]
            if row.empty:
                continue
            row = row.iloc[0]
            auc = float(row["AUC"])
            ci_lo = float(row["AUC_CI_lo"])
            ci_hi = float(row["AUC_CI_hi"])
            color = MODEL_COLORS[model_label]
            hatch = MODEL_HATCHES[model_label]
            bar_x = offsets[m_idx]

            ax.bar(
                bar_x, auc,
                width=bar_width * 0.88,
                color=color,
                hatch=hatch,
                edgecolor="white",
                alpha=0.88,
                zorder=3,
                label=model_label if path == paths[0] else "_nolegend_",
            )
            ax.errorbar(
                bar_x, auc,
                yerr=[[auc - ci_lo], [ci_hi - auc]],
                fmt="none",
                color="#333333",
                capsize=4, capthick=1.2, linewidth=1.2,
                zorder=4,
            )
            ax.text(
                bar_x, ci_hi + 0.012,
                f"{auc:.3f}",
                ha="center", va="bottom",
                fontsize=8.5, fontweight="bold",
                color="#333333", zorder=5,
            )

            # Permutation significance above bar
            ax.text(
                bar_x, ci_hi + 0.032,
                row["Sig_perm"],
                ha="center", va="bottom",
                fontsize=10, color=color,
            )

        ax.set_xticks(offsets)
        ax.set_xticklabels(MODEL_LABELS, fontsize=10)
        ax.set_title(path, fontsize=11, fontweight="normal")
        ax.axhline(0.5, color="#aaaaaa", linewidth=0.8,
                   linestyle="--", zorder=1)
        ax.set_ylim(0.35, 1.05)
        ax.set_ylabel("AUC (95% Bootstrap CI)", fontsize=10)
        ax.grid(axis="y", alpha=0.3, linestyle=":")

        n_train = int(df_path["N_train"].iloc[0])
        n_test = int(df_path["N_test"].iloc[0])
        ax.text(
            0.98, 0.02,
            f"Train n={n_train}  |  Test n={n_test}",
            transform=ax.transAxes,
            ha="right", va="bottom",
            fontsize=8, color="#555555",
        )

    # Legend below x-axis of first panel (outside plot area)
    fig.legend(
        loc="lower center",
        bbox_to_anchor=(0.5, -0.06),
        ncol=3,
        frameon=False,
        fontsize=10,
    )

    plt.tight_layout()
    save_figure(fig, fname, CRS_DIR)


# =============================================================================
# RENDER FIGURES — suppressed (results reported in Table 3 only)
# =============================================================================
# Figures disabled per submission figure limit.
# Re-enable by uncommenting the block below.
# for outcome_key, odata in OUTCOME_DATA.items():
#     df_cs_out = df_cs[df_cs["Outcome"] == odata["label"]]
#     if df_cs_out.empty:
#         continue
#     fname = "crosssystem_outcome1" if outcome_key == "O1" else "crosssystem_outcome2"
#     plot_crosssystem_figure(odata["label"], df_cs_out, fname)


# =============================================================================
# INLINE HTML SUMMARY TABLE
# =============================================================================
th = (
    "padding:5px 12px; text-align:center; font-weight:bold; "
    "border-top:2px solid #111; border-bottom:1px solid #111; "
    "background:#f7f7f7; color:#111;"
)
th_l = th.replace("center", "left")
td = "padding:5px 12px; text-align:center; color:#333;"
td_l = td.replace("center", "left")


def _p_str(p):
    if np.isnan(float(p)):
        return "--"
    if float(p) < 0.001:
        return "< 0.001"
    if float(p) < 0.01:
        return "< 0.01"
    return f"{float(p):.3f}"


html = (
    "<div style=\"font-family:'Helvetica Neue',Arial,sans-serif;"
    "font-size:12px; max-width:980px; margin-top:12px;"
    "background:white; padding:20px; border-radius:5px; color:#111;\">"
    "<p style=\"font-size:13px; margin:0 0 8px 0; color:#111;\">"
    "<b>Cross-System Validation Summary</b></p>"
)

for outcome_label in df_cs["Outcome"].unique():
    df_out = df_cs[df_cs["Outcome"] == outcome_label]
    html += (
        f"<p style=\"margin:12px 0 4px 0; color:#111;\">"
        f"<b>{outcome_label}</b></p>"
        f"<table style=\"border-collapse:collapse; width:100%;"
        f"margin-bottom:16px; background:white;\">"
        f"<thead><tr>"
        f"<th style=\"{th_l}\">Path</th>"
        f"<th style=\"{th_l}\">Model</th>"
        f"<th style=\"{th}\">N train / test</th>"
        f"<th style=\"{th}\">AUC (95% CI)</th>"
        f"<th style=\"{th}\">Brier</th>"
        f"<th style=\"{th}\">Cal. Slope</th>"
        f"<th style=\"{th}\">Cal. Int.</th>"
        f"<th style=\"{th}\">Perm. p</th>"
        f"<th style=\"{th}\">DeLong vs OOF</th>"
        f"</tr></thead><tbody>"
    )

    for path in df_out["Path"].unique():
        df_path = df_out[df_out["Path"] == path]
        for m_idx, model_label in enumerate(MODEL_LABELS):
            row = df_path[df_path["Model"] == model_label]
            if row.empty:
                continue
            row = row.iloc[0]
            cs = f"color:{MODEL_COLORS[model_label]};"
            bo = "<b>" if model_label == "Combined" else ""
            bc = "</b>" if model_label == "Combined" else ""
            path_cell = (
                f"<td style=\"{td_l}\" rowspan=\"3\">{path}</td>"
                if m_idx == 0 else ""
            )
            html += (
                f"<tr style=\"border-bottom:0.5px solid #e8e8e8;\">"
                f"{path_cell}"
                f"<td style=\"{td_l}{cs}\">{bo}{model_label}{bc}</td>"
                f"<td style=\"{td}\">{int(row.N_train)} / {int(row.N_test)}</td>"
                f"<td style=\"{td}\">{bo}{row.AUC:.3f} "
                f"({row.AUC_CI_lo:.3f}-{row.AUC_CI_hi:.3f}){bc}</td>"
                f"<td style=\"{td}\">{row.Brier:.3f}</td>"
                f"<td style=\"{td}\">{row.Cal_Slope:.3f}</td>"
                f"<td style=\"{td}\">{row.Cal_Int:.3f}</td>"
                f"<td style=\"{td}\">{_p_str(row.Perm_p)} {row.Sig_perm}</td>"
                f"<td style=\"{td}\">{_p_str(row.DeLong_cs_p)}</td>"
                f"</tr>"
            )

    footnote = (
        f"AUC = area under ROC curve; CI = 95% bootstrap CI "
        f"(n = {N_BOOT} resamples); Perm. p = permutation test "
        f"vs H0: AUC = 0.5 (n = {N_PERM} iterations); "
        f"DeLong vs OOF = comparison of cross-system AUC with "
        f"within-scanner OOF AUC on identical test subjects. "
        f"* p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001, ns = not significant. CI degenerate = bootstrap CI collapsed to point estimate; occurs when all model predictions are constant (e.g., MELD in cross-system paths)."
    )
    html += (
        f"</tbody><tfoot>"
        f"<tr style=\"border-top:2px solid #111;\">"
        f"<td colspan=\"9\" style=\"padding:8px 12px; font-size:11px;"
        f"color:#555; line-height:1.4;\">{footnote}</td>"
        f"</tr></tfoot></table>"
    )

html += "</div>"
ipy_display(HTML(html))

print("\nCell 10 -- Cross-system validation complete.")

In [ ]:
# =============================================================================
# Cell 15 -- External validation
# TRIPOD-AI items: 10e, 11, 12
# =============================================================================
# Train on internal cohort, test on Ext-A and Ext-B
# TRIPOD-AI item: 10e
# =============================================================================
#            Train on full internal cohort -> Test on Ext-A and Ext-B
# =============================================================================
# Evaluates the generalisability of each model to two independent external
# cohorts (TRIPOD-AI item 10e).
#
# Cohort design:
#   Internal  : 168 patients, two scanners (development cohort)
#   Ext-A     : Independent centre, scanner-matched (same vendor/field strength)
#               -> isolates centre / population effect
#   Ext-B     : Independent centre, different vendor, same field strength
#               -> isolates vendor / platform effect
#
# ComBat harmonisation — reference-based approach:
#   ComBat is fitted on the full internal cohort (both scanners).
#   Each external cohort is harmonised in transform-only mode using the
#   batch correction parameters estimated from the internal data
#   (gamma_ and delta_sq_ for the assigned batch ID).
#   This avoids leakage: no external data is used during ComBat fitting.
#
#   Implementation: pycombat 0.20 transform() requires all fitted batch IDs
#   to be present. For single-scanner external cohorts this is not satisfied.
#   A manual reference-based transform is therefore applied, mathematically
#   equivalent to pycombat.transform() for the subset of batches present.
#   Verified to produce identical results to the official API when all batches
#   are present (max absolute difference = 0.0).
#
# Batch ID assignment for external cohorts:
#   Ext-A (same vendor, 1.5T) -> batch 0 (Signa 1.5T reference)
#   Ext-B (different vendor, 1.5T) -> batch 0 (closest match available)
#   Edit EXT_COHORTS[*]["batch_id"] if your scanner mapping differs.
#
# Analytic design:
#   1. Train on entire internal cohort using BEST_C_MAP from Cell 4.
#   2. Apply to harmonised external cohort without re-fitting.
#   3. Per cohort x outcome x model:
#      AUC + 95% bootstrap CI, permutation test, Brier score,
#      calibration slope/intercept, unpaired z-test vs internal OOF AUC.
#
# CSV format for external cohort files:
#   Same column names as internal dataset (same PyRadiomics prefix
#   "original_", same COL_MELD, COL_GRADE, COL_HE_MILD, COL_HE_MODSEV).
#   Decimal separator: period (European comma auto-corrected).
#
# Outputs:
#   outputs/tables/external_validation.csv
#   [SUPPRESSED] outputs/figures/external/ (re-enable in RENDER block)
#   Inline HTML summary table
#
# TRIPOD-AI items covered:
#   - 10e : external validation
#   - 11  : uncertainty quantification (bootstrap CI)
#   - 12  : permutation test
# =============================================================================

# --- Execution guard ---
if "BEST_C_MAP" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed in this session.\n"
        "Please run Cells 0-9 before this cell."
    )

# =============================================================================
# CONFIGURATION — edit paths and batch IDs before running
# =============================================================================
EXT_COHORTS = [
    {
        "key":      "ExtA",
        "label":    "Ext-A (scanner-matched, independent centre)",
        "csv":      DATA_DIR / "radiomic_HE_dataset_extA.csv",
        "short":    "exta",
        "batch_id": 0,   # 0 = Signa 1.5T reference batch
    },
    {
        "key":      "ExtB",
        "label":    "Ext-B (different vendor, 1.5T)",
        "csv":      DATA_DIR / "radiomic_HE_dataset_extB.csv",
        "short":    "extb",
        "batch_id": 0,   # map to batch 0 (1.5T reference); edit if needed
    },
]


# =============================================================================
# STATISTICAL HELPERS  (self-contained for cell-level independence)
# =============================================================================
def bootstrap_auc_ci(y, p, n_boot=N_BOOT, seed=BOOT_SEED):
    """Percentile bootstrap 95% CI for AUC."""
    rng = np.random.default_rng(seed)
    aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y), len(y))
        yb, pb = y[idx], p[idx]
        if len(np.unique(yb)) < 2:
            continue
        aucs.append(roc_auc_score(yb, pb))
    aucs = np.array(aucs)
    return (
        roc_auc_score(y, p),
        float(np.percentile(aucs, 2.5)),
        float(np.percentile(aucs, 97.5)),
    )


def permutation_test_auc(y_test, p_test, n_perm=N_PERM, seed=BOOT_SEED):
    """Two-sided permutation test: H0: AUC = 0.5."""
    rng = np.random.default_rng(seed)
    obs_auc = roc_auc_score(y_test, p_test)
    null_auc = np.array([
        roc_auc_score(rng.permutation(y_test), p_test)
        for _ in range(n_perm)
    ])
    return float(np.mean(np.abs(null_auc - 0.5) >= np.abs(obs_auc - 0.5)))


def calibration_metrics(y, p):
    """Brier score, calibration slope and intercept."""
    eps = 1e-6
    p_c = np.clip(p, eps, 1 - eps)
    logit = np.log(p_c / (1 - p_c))
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=500)
    lr.fit(logit.reshape(-1, 1), y)
    return {
        "Brier":         float(brier_score_loss(y, p_c)),
        "Cal_Slope":     float(lr.coef_[0][0]),
        "Cal_Intercept": float(lr.intercept_[0]),
    }


# =============================================================================
# COMBAT REFERENCE-BASED TRANSFORM
# =============================================================================
def combat_reference_transform(X_new_std, meld_new_std, combat_obj, batch_id):
    """
    Apply ComBat batch correction to new data using parameters estimated
    on the internal training cohort (reference-based harmonisation).

    Mathematically equivalent to combat_obj.transform() for the
    specified batch. Verified against pycombat 0.20 official API
    (max absolute difference = 0.0 when all batches are present).

    Parameters
    ----------
    X_new_std    : np.ndarray (n, p), standardised radiomic features
    meld_new_std : np.ndarray (n, 1), standardised MELD covariate
    combat_obj   : fitted Combat instance
    batch_id     : int, batch ID to apply (must be in combat_obj.batches_)

    Returns
    -------
    Y_harm : np.ndarray (n, p), harmonised features
    """
    Y = X_new_std.copy() - combat_obj.intercept_
    if combat_obj.coefs_x_.size > 0:
        Y -= np.matmul(meld_new_std, combat_obj.coefs_x_)
    Y /= np.sqrt(combat_obj.epsilon_)

    idx = int(np.where(combat_obj.batches_ == batch_id)[0][0])
    Y -= combat_obj.gamma_[idx]
    Y /= np.sqrt(combat_obj.delta_sq_[idx])

    Y *= np.sqrt(combat_obj.epsilon_)
    Y += combat_obj.intercept_
    if combat_obj.coefs_x_.size > 0:
        Y += np.matmul(meld_new_std, combat_obj.coefs_x_)
    return Y


# =============================================================================
# STEP 1 — Fit ComBat and scalers on full internal cohort
# =============================================================================
print("=" * 65)
print("STEP 1 -- Fitting ComBat on full internal cohort")
print("=" * 65)

X_rad_internal = df[rad_cols].values.astype(float)
scanner_internal = df[COL_MR_SYSTEM].values
unique_scanners_int = np.unique(scanner_internal)
scanner_to_id = {s: i for i, s in enumerate(unique_scanners_int)}
b_internal = np.array([scanner_to_id[s] for s in scanner_internal])

meld_internal = df[[COL_MELD]].values.astype(float)

scaler_rad = StandardScaler()
X_rad_int_std = scaler_rad.fit_transform(X_rad_internal)

scaler_meld = StandardScaler()
meld_int_std = scaler_meld.fit_transform(meld_internal)

combat_internal = Combat()
combat_internal.fit(X_rad_int_std, b_internal, X=meld_int_std)

print(f"  Internal cohort: {len(df)} patients")
print(f"  Scanners fitted: {dict(zip(unique_scanners_int, range(len(unique_scanners_int))))}")
print(f"  Batch IDs:       {np.unique(b_internal).tolist()}")
print("  [OK] ComBat fitted on internal cohort")


# =============================================================================
# STEP 2 — Build harmonised internal feature matrices for training
# =============================================================================
# Re-derive harmonised internal features using the fitted combat_internal
# (consistent with what external cohorts receive after transform)

def _harmonise_internal_subset(mask, X_meld_raw):
    """Return ComBat-harmonised radiomic features for an internal subset."""
    X_sub_std = X_rad_int_std[mask]
    b_sub = b_internal[mask]
    meld_sub_std = scaler_meld.transform(X_meld_raw)
    # Use official transform (all batches present in internal subsets)
    return combat_internal.transform(X_sub_std, b_sub, X=meld_sub_std)


mask_o1 = df[COL_GRADE].isin([0, 1]).values
mask_o2 = (df[COL_GRADE] >= 1).values

X_o1_rad_harm = _harmonise_internal_subset(mask_o1, X_o1_meld)
X_o2_rad_harm = _harmonise_internal_subset(mask_o2, X_o2_meld)
X_o1_combo_harm = np.hstack([X_o1_meld, X_o1_rad_harm])
X_o2_combo_harm = np.hstack([X_o2_meld, X_o2_rad_harm])

print("  [OK] Internal harmonised feature matrices ready")


# =============================================================================
# OUTCOME DEFINITIONS
# =============================================================================
EXT_OUTCOMES = [
    {
        "key":       "O1",
        "label":     "Outcome 1 -- Mild vs No HE",
        "grades":    [0, 1],
        "col_y":     COL_HE_MILD,
        "X_int": {
            "MELD":      X_o1_meld,
            "Radiomics": X_o1_rad_harm,
            "Combined":  X_o1_combo_harm,
        },
        "y_int":  y_o1,
        "p_oof": {
            "MELD":      p_o1_meld,
            "Radiomics": p_o1_rad,
            "Combined":  p_o1_combo,
        },
    },
    {
        "key":       "O2",
        "label":     "Outcome 2 -- Moderate-Severe vs Mild HE",
        "grades":    [1, 2],
        "col_y":     COL_HE_MODSEV,
        "X_int": {
            "MELD":      X_o2_meld,
            "Radiomics": X_o2_rad_harm,
            "Combined":  X_o2_combo_harm,
        },
        "y_int":  y_o2,
        "p_oof": {
            "MELD":      p_o2_meld,
            "Radiomics": p_o2_rad,
            "Combined":  p_o2_combo,
        },
    },
]


# =============================================================================
# STEP 3 — External validation loop
# =============================================================================
rows_ev = []

for cohort in EXT_COHORTS:
    print(f"\n{'=' * 65}")
    print(f"EXTERNAL COHORT: {cohort['label']}")
    print(f"{'=' * 65}")

    if not cohort["csv"].exists():
        print(f"  [SKIP] File not found: {cohort['csv']}")
        print(f"         Place CSV at: {cohort['csv'].resolve()}")
        continue

    # Load and decimal-correct
    df_ext_raw = pd.read_csv(
        cohort["csv"], sep=None, engine="python", dtype=str
    )
    for col in df_ext_raw.columns:
        converted = (
            df_ext_raw[col]
            .str.replace(",", ".", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
        if converted.notna().any():
            df_ext_raw[col] = converted

    print(f"  Loaded: {len(df_ext_raw)} patients from {cohort['csv'].name}")

    # Reference-based ComBat harmonisation
    X_rad_ext_raw = df_ext_raw[rad_cols].values.astype(float)
    meld_ext_raw = df_ext_raw[[COL_MELD]].values.astype(float)

    X_rad_ext_std = scaler_rad.transform(X_rad_ext_raw)
    meld_ext_std = scaler_meld.transform(meld_ext_raw)

    X_rad_ext_harm = combat_reference_transform(
        X_rad_ext_std, meld_ext_std,
        combat_internal, cohort["batch_id"],
    )
    print(
        f"  [OK] Reference-based ComBat applied "
        f"(batch_id={cohort['batch_id']})"
    )

    # Rebuild df_ext with harmonised features
    df_ext = df_ext_raw.copy()
    for i, col in enumerate(rad_cols):
        df_ext[col] = X_rad_ext_harm[:, i]

    for odata in EXT_OUTCOMES:
        mask_ext = df_ext[COL_GRADE].isin(odata["grades"])
        df_ext_o = df_ext[mask_ext].reset_index(drop=True)
        y_ext = df_ext_o[odata["col_y"]].values.astype(int)

        n_ext = len(df_ext_o)
        if n_ext == 0 or len(np.unique(y_ext)) < 2:
            print(f"\n  {odata['label']}: [SKIP] insufficient data")
            continue

        n_ext_pos = int(y_ext.sum())
        print(f"\n  {odata['label']}")
        print(f"    n={n_ext}  |  positive={n_ext_pos} ({100*n_ext_pos/n_ext:.1f}%)")

        X_ext_rad_o = df_ext_o[rad_cols].values.astype(float)
        X_ext_meld_o = df_ext_o[[COL_MELD]].values.astype(float)
        X_ext_dict = {
            "MELD":      X_ext_meld_o,
            "Radiomics": X_ext_rad_o,
            "Combined":  np.hstack([X_ext_meld_o, X_ext_rad_o]),
        }

        print(
            f"    {'Model':<12} {'AUC':>6}  {'95% CI':<17} "
            f"{'Brier':>6}  {'Perm p':>8}  Sig.  vs OOF"
        )
        print(f"    {'-' * 65}")

        for model_label in MODEL_LABELS:
            best_C = BEST_C_MAP[(odata["key"], model_label)]

            pipe = Pipeline([
                ("scaler", StandardScaler()),
                ("clf", LogisticRegression(
                    penalty="l1",
                    C=best_C,
                    solver="liblinear",
                    class_weight="balanced",
                    max_iter=2000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(odata["X_int"][model_label], odata["y_int"])
            p_ext = pipe.predict_proba(X_ext_dict[model_label])[:, 1]

            auc_ev, ci_lo, ci_hi = bootstrap_auc_ci(y_ext, p_ext)
            p_perm = permutation_test_auc(y_ext, p_ext)
            auc_oof = roc_auc_score(odata["y_int"], odata["p_oof"][model_label])
            cal = calibration_metrics(y_ext, p_ext)

            n_int = len(odata["y_int"])
            se_ext = np.sqrt(auc_ev * (1 - auc_ev) / n_ext)
            se_int = np.sqrt(auc_oof * (1 - auc_oof) / n_int)
            denom = np.sqrt(se_ext**2 + se_int**2 + 1e-8)
            z_diff = (auc_ev - auc_oof) / denom
            p_diff = float(2 * (1 - stats.norm.cdf(abs(z_diff))))

            if p_perm < 0.001:
                sig_perm = "***"
            elif p_perm < 0.01:
                sig_perm = "**"
            elif p_perm < 0.05:
                sig_perm = "*"
            else:
                sig_perm = "ns"

            rows_ev.append({
                "Cohort":         cohort["label"],
                "Outcome":        odata["label"],
                "Model":          model_label,
                "N_train":        n_int,
                "N_test":         n_ext,
                "AUC_ext":        auc_ev,
                "AUC_CI_lo":      ci_lo,
                "AUC_CI_hi":      ci_hi,
                "AUC_int_OOF":    auc_oof,
                "p_diff_ext_int": p_diff,
                "Brier":          cal["Brier"],
                "Cal_Slope":      cal["Cal_Slope"],
                "Cal_Int":        cal["Cal_Intercept"],
                "Perm_p":         p_perm,
                "Sig_perm":       sig_perm,
            })

            print(
                f"    {model_label:<12} "
                f"{auc_ev:>6.3f}  ({ci_lo:.3f}-{ci_hi:.3f})  "
                f"{cal['Brier']:>6.3f}  {p_perm:>8.3f}  {sig_perm:<4}  "
                f"OOF={auc_oof:.3f} p={p_diff:.3f}"
            )


# =============================================================================
# SAVE CSV
# =============================================================================
df_ev = pd.DataFrame(rows_ev) if rows_ev else pd.DataFrame()
if not df_ev.empty:
    df_ev.to_csv(CSV_DIR / "external_validation.csv", index=False)
    print(f"\n  [OK] Saved -> {(CSV_DIR / 'external_validation.csv').name}")
else:
    print("\n  [WARN] No external validation results — check CSV paths.")


# =============================================================================
# FIGURES — one panel per cohort x outcome
# =============================================================================
def plot_extval_figure(cohort_label, cohort_short, outcome_label,
                       outcome_short, df_ev_sub):
    """
    Grouped bar chart of external validation AUCs with CI error bars.
    Dashed horizontal tick per bar = internal OOF AUC reference.
    Legend below x-axis (outside plot area).
    """
    fig, ax = plt.subplots(figsize=(7, 5.5))
    bar_width = 0.22
    offsets = np.array([-bar_width, 0.0, bar_width])

    for m_idx, model_label in enumerate(MODEL_LABELS):
        row = df_ev_sub[df_ev_sub["Model"] == model_label]
        if row.empty:
            continue
        row = row.iloc[0]
        auc = float(row["AUC_ext"])
        ci_lo = float(row["AUC_CI_lo"])
        ci_hi = float(row["AUC_CI_hi"])
        auc_oof = float(row["AUC_int_OOF"])
        color = MODEL_COLORS[model_label]
        hatch = MODEL_HATCHES[model_label]
        bar_x = offsets[m_idx]

        ax.bar(
            bar_x, auc,
            width=bar_width * 0.88,
            color=color, hatch=hatch,
            edgecolor="white", alpha=0.88, zorder=3,
        )
        ax.errorbar(
            bar_x, auc,
            yerr=[[auc - ci_lo], [ci_hi - auc]],
            fmt="none", color="#333333",
            capsize=4, capthick=1.2, linewidth=1.2, zorder=4,
        )
        ax.text(
            bar_x, ci_hi + 0.012,
            f"{auc:.3f}",
            ha="center", va="bottom",
            fontsize=8.5, fontweight="bold",
            color="#333333", zorder=5,
        )
        # Internal OOF reference tick (dashed)
        ax.plot(
            [bar_x - bar_width * 0.44, bar_x + bar_width * 0.44],
            [auc_oof, auc_oof],
            color=color, linewidth=1.8, linestyle="--", zorder=6,
        )
        ax.text(
            bar_x, ci_hi + 0.032,
            row["Sig_perm"],
            ha="center", va="bottom",
            fontsize=10, color=color,
        )

    ax.set_xticks(offsets)
    ax.set_xticklabels(MODEL_LABELS, fontsize=10)
    ax.axhline(0.5, color="#aaaaaa", linewidth=0.8,
               linestyle="--", zorder=1)
    ax.set_ylim(0.35, 1.05)
    ax.set_ylabel("AUC (95% Bootstrap CI)", fontsize=10)
    ax.set_title(
        f"External Validation -- {outcome_label}\n{cohort_label}",
        fontsize=11, fontweight="normal",
    )
    ax.grid(axis="y", alpha=0.3, linestyle=":")

    n_train = int(df_ev_sub["N_train"].iloc[0])
    n_test = int(df_ev_sub["N_test"].iloc[0])
    ax.text(
        0.98, 0.02,
        f"Train n={n_train} (internal)  |  Test n={n_test}",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=8, color="#555555",
    )

    # Legend below x-axis (outside plot area)
    legend_handles = [
        Line2D(
            [0], [0], color="#555555", linestyle="--", linewidth=1.2,
            label="Internal OOF AUC (reference)",
        )
    ]
    ax.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.14),
        frameon=False, fontsize=8,
    )

    plt.tight_layout()
    fname = f"extval_{cohort_short}_{outcome_short}"
    save_figure(fig, fname, EXT_DIR)


# =============================================================================
# INLINE HTML SUMMARY TABLE
# =============================================================================
th = (
    "padding:5px 12px; text-align:center; font-weight:bold; "
    "border-top:2px solid #111; border-bottom:1px solid #111; "
    "background:#f7f7f7; color:#111;"
)
th_l = th.replace("center", "left")
td = "padding:5px 12px; text-align:center; color:#333;"
td_l = td.replace("center", "left")


def _p_str(p):
    v = float(p)
    if v != v:
        return "--"
    if v < 0.001:
        return "< 0.001"
    if v < 0.01:
        return "< 0.01"
    return f"{v:.3f}"


if not df_ev.empty:
    html = (
        "<div style=\"font-family:'Helvetica Neue',Arial,sans-serif;"
        "font-size:12px; max-width:1020px; margin-top:12px;"
        "background:white; padding:20px; border-radius:5px; color:#111;\">"
        "<p style=\"font-size:13px; margin:0 0 8px 0; color:#111;\">"
        "<b>External Validation Summary</b></p>"
    )

    for cohort in EXT_COHORTS:
        df_c = df_ev[df_ev["Cohort"] == cohort["label"]]
        if df_c.empty:
            continue
        html += (
            f"<p style=\"margin:12px 0 4px 0; color:#111;\">"
            f"<b>{cohort['label']}</b></p>"
        )
        for odata in EXT_OUTCOMES:
            df_co = df_c[df_c["Outcome"] == odata["label"]]
            if df_co.empty:
                continue
            html += (
                f"<p style=\"margin:8px 0 2px 16px; color:#555;\">"
                f"{odata['label']}</p>"
                f"<table style=\"border-collapse:collapse; width:100%;"
                f"margin-bottom:16px; background:white;\">"
                f"<thead><tr>"
                f"<th style=\"{th_l}\">Model</th>"
                f"<th style=\"{th}\">N train / test</th>"
                f"<th style=\"{th}\">AUC ext (95% CI)</th>"
                f"<th style=\"{th}\">AUC int OOF</th>"
                f"<th style=\"{th}\">Delta p</th>"
                f"<th style=\"{th}\">Brier</th>"
                f"<th style=\"{th}\">Cal. Slope</th>"
                f"<th style=\"{th}\">Perm. p</th>"
                f"</tr></thead><tbody>"
            )
            for model_label in MODEL_LABELS:
                row = df_co[df_co["Model"] == model_label]
                if row.empty:
                    continue
                row = row.iloc[0]
                cs = f"color:{MODEL_COLORS[model_label]};"
                bo = "<b>" if model_label == "Combined" else ""
                bc = "</b>" if model_label == "Combined" else ""
                html += (
                    f"<tr style=\"border-bottom:0.5px solid #e8e8e8;\">"
                    f"<td style=\"{td_l}{cs}\">{bo}{model_label}{bc}</td>"
                    f"<td style=\"{td}\">"
                    f"{int(row.N_train)} / {int(row.N_test)}</td>"
                    f"<td style=\"{td}\">{bo}{row.AUC_ext:.3f} "
                    f"({row.AUC_CI_lo:.3f}-{row.AUC_CI_hi:.3f}){bc}</td>"
                    f"<td style=\"{td}\">{row.AUC_int_OOF:.3f}</td>"
                    f"<td style=\"{td}\">{_p_str(row.p_diff_ext_int)}</td>"
                    f"<td style=\"{td}\">{row.Brier:.3f}</td>"
                    f"<td style=\"{td}\">{row.Cal_Slope:.3f}</td>"
                    f"<td style=\"{td}\">"
                    f"{_p_str(row.Perm_p)} {row.Sig_perm}</td>"
                    f"</tr>"
                )
            footnote = (
                f"AUC ext = AUC on external test cohort; "
                f"AUC int OOF = internal out-of-fold AUC (Cell 4); "
                f"Delta p = unpaired z-test, external vs internal AUC; "
                f"Perm. p = permutation test vs H0: AUC=0.5 "
                f"(n={N_PERM}). "
                f"Dashed ticks in figures = internal OOF reference. "
                f"* p&lt;0.05, ** p&lt;0.01, *** p&lt;0.001."
            )
            html += (
                f"</tbody><tfoot>"
                f"<tr style=\"border-top:2px solid #111;\">"
                f"<td colspan=\"8\" style=\"padding:8px 12px;"
                f"font-size:11px; color:#555; line-height:1.4;\">"
                f"{footnote}</td>"
                f"</tr></tfoot></table>"
            )

    html += "</div>"
    ipy_display(HTML(html))

print("\nCell 11 -- External validation complete.")

In [ ]:
# =============================================================================
# Cell 16 -- Domain shift analysis (SF4)
# TRIPOD-AI items: 10e, 12
# =============================================================================
# Supplementary Figure SF4 -- KS test + PCA biplot
# TRIPOD-AI items: 10e, 12
# =============================================================================
#             Supplementary Figure SF_domainshift
# =============================================================================
# Quantifies the feature-level distributional shift between the internal
# cohort and Ext-B (different vendor, 1.5T) that explains the collapse of
# radiomic model performance on Ext-B (AUC ≈ 0.50).
#
# Analysis:
#   1. Kolmogorov-Smirnov (KS) two-sample test for each of the 30 radiomic
#      features: internal (post-ComBat) vs Ext-B (post-reference-ComBat).
#      FDR correction (Benjamini-Hochberg) across features.
#   2. PCA biplot (PC1 vs PC2) coloured by cohort (internal / Ext-A / Ext-B)
#      to visualise the multivariate domain shift.
#   3. Comparison: Ext-A (scanner-matched) vs Ext-B (different vendor)
#      as quantitative evidence of vendor-specific shift.
#
# Requires:
#   combat_internal, scaler_rad, scaler_meld  (Cell 11)
#   EXT_COHORTS, combat_reference_transform   (Cell 11)
#
# Outputs:
#   outputs/figures/external/domain_shift_ks_color.tiff/.pdf/_gray.*
#   outputs/figures/external/domain_shift_pca_color.tiff/.pdf/_gray.*
#   outputs/tables/domain_shift_ks_results.csv
#
# TRIPOD-AI items covered:
#   - 10e : external validation — domain shift documentation
#   - 12  : statistical testing of distributional differences (KS + FDR)
# =============================================================================

# --- Execution guard ---
if "combat_internal" not in dir() or "scaler_rad" not in dir():
    raise RuntimeError(
        "Cell 15 has not been executed in this session.\n"
        "Please run Cells 0-15 before this cell."
    )

from scipy.stats import ks_2samp
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import PCA

print("=" * 65)
print("Domain shift analysis — Ext-B vs internal cohort")
print("=" * 65)

# =============================================================================
# LOAD AND HARMONISE ALL EXTERNAL COHORTS
# =============================================================================
ext_harmonised = {}
for cohort in EXT_COHORTS:
    if not cohort["csv"].exists():
        print(f"  [SKIP] {cohort['key']}: file not found")
        continue
    df_ext_raw = pd.read_csv(cohort["csv"], sep=None, engine="python", dtype=str)
    for col in df_ext_raw.columns:
        converted = (df_ext_raw[col]
                     .str.replace(",", ".", regex=False)
                     .pipe(pd.to_numeric, errors="coerce"))
        if converted.notna().any():
            df_ext_raw[col] = converted

    X_ext_raw  = df_ext_raw[rad_cols].values.astype(float)
    meld_ext   = df_ext_raw[[COL_MELD]].values.astype(float)
    X_ext_std  = scaler_rad.transform(X_ext_raw)
    meld_ext_s = scaler_meld.transform(meld_ext)
    X_ext_harm = combat_reference_transform(
        X_ext_std, meld_ext_s, combat_internal, cohort["batch_id"]
    )
    ext_harmonised[cohort["key"]] = {
        "label": cohort["label"],
        "short": cohort["short"],
        "X_harm": X_ext_harm,
        "n": len(X_ext_harm),
    }
    print(f"  Loaded {cohort['key']}: n={len(X_ext_harm)}")

# Internal post-ComBat
X_int_full  = df[rad_cols].values.astype(float)
scanner_int = df[COL_MR_SYSTEM].values
unique_sc   = np.unique(scanner_int)
sc_to_id    = {s: i for i, s in enumerate(unique_sc)}
b_int       = np.array([sc_to_id[s] for s in scanner_int])
X_int_std   = scaler_rad.transform(X_int_full)
meld_int_s  = scaler_meld.transform(df[[COL_MELD]].values.astype(float))
X_int_harm  = combat_internal.transform(X_int_std, b_int, X=meld_int_s)
print(f"  Internal (post-ComBat): n={len(X_int_harm)}")

# =============================================================================
# KS TEST — internal vs each external cohort
# =============================================================================
ks_rows = []
for ext_key, ext_data in ext_harmonised.items():
    X_ext = ext_data["X_harm"]
    ks_stats, ks_pvals = [], []
    for fi in range(len(rad_cols)):
        stat, pval = ks_2samp(X_int_harm[:, fi], X_ext[:, fi])
        ks_stats.append(stat)
        ks_pvals.append(pval)
    reject, pvals_fdr, _, _ = multipletests(ks_pvals, method="fdr_bh")
    n_sig = int(reject.sum())
    print(f"\n  KS test: Internal vs {ext_key}")
    print(f"    Features with KS p_FDR < 0.05: {n_sig} / {len(rad_cols)}")
    for fi, feat in enumerate(rad_cols):
        ks_rows.append({
            "Cohort":    ext_key,
            "Feature":   feat,
            "KS_stat":   round(ks_stats[fi], 4),
            "KS_p":      round(ks_pvals[fi], 4),
            "KS_p_FDR":  round(pvals_fdr[fi], 4),
            "Significant_FDR": bool(reject[fi]),
        })

df_ks = pd.DataFrame(ks_rows)
df_ks.to_csv(CSV_DIR / "domain_shift_ks_results.csv", index=False)
print(f"\n  [OK] domain_shift_ks_results.csv saved")

# =============================================================================
# FIGURE 1 — KS statistic bar chart: ExtA vs ExtB comparison
# =============================================================================
ext_keys = list(ext_harmonised.keys())
if len(ext_keys) >= 2:
    fig_ks, ax_ks = plt.subplots(figsize=(14, 5))
    x = np.arange(len(rad_cols))
    width = 0.35
    colors_ext = ["#1D9E75", "#D85A30"]
    for ki, (ext_key, color) in enumerate(zip(ext_keys, colors_ext)):
        df_k = df_ks[df_ks["Cohort"] == ext_key].sort_values("Feature")
        bars = ax_ks.bar(x + ki * width, df_k["KS_stat"].values,
                         width=width * 0.9, color=color, alpha=0.8,
                         label=ext_harmonised[ext_key]["label"])
    ax_ks.set_xticks(x + width / 2)
    ax_ks.set_xticklabels(
        [f.replace("original_", "")[:20] for f in sorted(rad_cols)],
        rotation=90, fontsize=7,
    )
    ax_ks.set_ylabel("KS statistic", fontsize=10)
    ax_ks.set_title(
        "Kolmogorov-Smirnov Statistics: Internal vs External Cohorts\n"
        "(post-ComBat harmonisation, FDR-corrected)",
        fontsize=11, fontweight="normal",
    )
    ax_ks.legend(fontsize=9, frameon=False)
    ax_ks.axhline(0.3, color="#aaaaaa", lw=0.8, ls="--",
                  label="Moderate shift threshold")
    ax_ks.grid(axis="y", alpha=0.25, linestyle=":")
    plt.tight_layout()
    save_figure(fig_ks, "domain_shift_ks", EXT_DIR)
    print("  [OK] domain_shift_ks figure saved")

# =============================================================================
# FIGURE 2 — PCA biplot coloured by cohort
# =============================================================================
X_all_list  = [X_int_harm]
labels_list = ["Internal"] * len(X_int_harm)

for k in ext_keys:
    X_all_list.append(ext_harmonised[k]["X_harm"])
    labels_list.extend([ext_harmonised[k]["label"]] * ext_harmonised[k]["n"])

X_all  = np.vstack(X_all_list)
labels = labels_list  # ora len(labels) == len(X_all) garantito

assert len(labels) == len(X_all), \
    f"Mismatch: X_all={len(X_all)}, labels={len(labels)}"

pca2 = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca2.fit_transform(X_all)
var_exp = pca2.explained_variance_ratio_ * 100

cohort_colors = {"Internal": "#888780",
                 **{ext_harmonised[k]["label"]: c
                    for k, c in zip(ext_keys, ["#1D9E75", "#D85A30"])}}

fig_pca, ax_pca = plt.subplots(figsize=(8, 6))
for cohort_label in dict.fromkeys(labels):
    mask = np.array([l == cohort_label for l in labels])
    ax_pca.scatter(X_pca[mask, 0], X_pca[mask, 1],
                   c=cohort_colors.get(cohort_label, "#888780"),
                   alpha=0.6, s=20, label=cohort_label)

ax_pca.set_xlabel(f"PC1 ({var_exp[0]:.1f}% variance)", fontsize=10)
ax_pca.set_ylabel(f"PC2 ({var_exp[1]:.1f}% variance)", fontsize=10)
ax_pca.set_title(
    "PCA of Harmonised Radiomic Features\n"
    "Internal cohort vs External validation cohorts",
    fontsize=11, fontweight="normal",
)
ax_pca.legend(fontsize=9, frameon=False, loc="upper right")
ax_pca.grid(alpha=0.2, linestyle=":")
plt.tight_layout()
save_figure(fig_pca, "domain_shift_pca", EXT_DIR)
print("  [OK] domain_shift_pca figure saved")

print("\nCell 11b -- Domain shift analysis complete.")


In [ ]:
# =============================================================================
# Cell 17 -- ComBat harmonisation diagnostics (SF3)
# TRIPOD-AI items: 9, 15
# =============================================================================
# Supplementary Figure SF3
# TRIPOD-AI items: 9, 15
# =============================================================================
#             Supplementary Figure SF_combat
# =============================================================================
# Evaluates the effectiveness of ComBat harmonisation by comparing
# radiomic feature distributions before and after batch correction.
#
# Analysis:
#   1. Pre/post-ComBat distribution comparison for the top-5 SHAP features
#      (by mean |SHAP| from Cell 8) — violin plots per scanner group.
#   2. Levene test for equality of variances across scanners,
#      pre- and post-ComBat (IBSI-compliant feature stability check).
#   3. Silhouette score (scanner batch labels) pre- and post-ComBat
#      on PCA-reduced features — lower score = better harmonisation.
#
# Requires: combat_internal (Cell 11), shap_df_o1/shap_df_o2 (Cell 8).
# Falls back to top-5 features by variance if Cell 8 was not executed.
#
#
# TRIPOD-AI items covered:
#   - 9  : preprocessing (scanner harmonisation documented)
#   - 15 : harmonisation diagnostics saved for reproducibility
# =============================================================================

# --- Execution guard ---
if "combat_internal" not in dir():
    raise RuntimeError(
        "Cell 15 has not been executed in this session.\n"
        "Please run Cells 0-15 before this cell."
    )

from scipy.stats import levene, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score as sil_score

# =============================================================================
# SELECT TOP-5 FEATURES
# =============================================================================
if "shap_df_o1" in dir():
    top5_feats = shap_df_o1.head(5)["Feature"].tolist()
    print("Using top-5 SHAP features from Cell 8 (Outcome 1):")
else:
    # Fallback: top-5 by variance in internal cohort
    vars_ = pd.Series(
        df[rad_cols].var().values, index=rad_cols
    ).nlargest(5)
    top5_feats = vars_.index.tolist()
    print("Fallback: top-5 features by variance:")
for f in top5_feats:
    print(f"  {f}")

# =============================================================================
# COMPUTE PRE- AND POST-COMBAT FEATURE MATRICES
# =============================================================================
X_rad_int  = df[rad_cols].values.astype(float)
scanner_int = df[COL_MR_SYSTEM].values
unique_sc   = np.unique(scanner_int)
sc_to_id    = {s: i for i, s in enumerate(unique_sc)}
b_int       = np.array([sc_to_id[s] for s in scanner_int])

scaler_diag   = StandardScaler()
X_rad_std     = scaler_diag.fit_transform(X_rad_int)
meld_int_std  = StandardScaler().fit_transform(df[[COL_MELD]].values.astype(float))

X_rad_harm = combat_internal.transform(X_rad_std, b_int, X=meld_int_std)
print(f"\nHarmonised feature matrix: {X_rad_harm.shape}")

# =============================================================================
# SILHOUETTE SCORE PRE/POST-COMBAT (batch = scanner)
# =============================================================================
pca = PCA(n_components=10, random_state=RANDOM_STATE)
X_pre_pca  = pca.fit_transform(X_rad_std)
X_post_pca = pca.fit_transform(X_rad_harm)

sil_pre  = sil_score(X_pre_pca,  b_int, metric="euclidean")
sil_post = sil_score(X_post_pca, b_int, metric="euclidean")
print(f"\nSilhouette score (scanner batch):")
print(f"  Pre-ComBat  : {sil_pre:.4f}  (higher = more batch separation)")
print(f"  Post-ComBat : {sil_post:.4f}  (closer to 0 = better harmonisation)")
delta = sil_pre - sil_post
print(f"  Delta       : {delta:.4f}  ({'reduction' if delta > 0 else 'increase'} in batch separation)")

# =============================================================================
# LEVENE TEST PRE/POST-COMBAT
# =============================================================================
levene_rows = []
feat_idx = [rad_cols.index(f) for f in top5_feats]
for fi, feat in zip(feat_idx, top5_feats):
    groups_pre  = [X_rad_std[b_int == i, fi] for i in range(len(unique_sc))]
    groups_post = [X_rad_harm[b_int == i, fi] for i in range(len(unique_sc))]
    _, p_pre  = levene(*groups_pre,  center="median")
    _, p_post = levene(*groups_post, center="median")
    levene_rows.append({
        "Feature":      feat,
        "Levene_p_pre":  round(p_pre,  4),
        "Levene_p_post": round(p_post, 4),
        "Harmonised":   "Yes" if p_post > 0.05 else "No",
    })
    print(f"  {feat[-35:]:35s}  pre p={p_pre:.4f}  post p={p_post:.4f}")

df_levene = pd.DataFrame(levene_rows)
df_levene.to_csv(CSV_DIR / "combat_levene_results.csv", index=False)
print(f"\n  [OK] combat_levene_results.csv -> {CSV_DIR / 'combat_levene_results.csv'}")

# =============================================================================
# FIGURE — violin plots pre/post-ComBat for top-5 features
# =============================================================================
SCANNER_LABELS = {i: s for s, i in sc_to_id.items()}
n_feat = len(top5_feats)
fig, axes = plt.subplots(2, n_feat, figsize=(4 * n_feat, 8),
                         sharey=False, constrained_layout=True)
fig.suptitle(
    "ComBat Harmonisation Diagnostics\n"
    "Pre- and Post-ComBat Feature Distributions by Scanner",
    fontsize=12, fontweight="normal",
)

colors_sc = [MODEL_COLORS[m] for m in MODEL_LABELS[:len(unique_sc)]]

for col_i, (fi, feat) in enumerate(zip(feat_idx, top5_feats)):
    for row_i, (X_mat, phase) in enumerate([
        (X_rad_std, "Pre-ComBat"),
        (X_rad_harm, "Post-ComBat"),
    ]):
        ax = axes[row_i, col_i]
        data_by_sc = [X_mat[b_int == i, fi] for i in range(len(unique_sc))]
        parts = ax.violinplot(data_by_sc, positions=range(len(unique_sc)),
                              showmedians=True, showextrema=False)
        for pc, c in zip(parts["bodies"], colors_sc):
            pc.set_facecolor(c)
            pc.set_alpha(0.65)
        parts["cmedians"].set_color("#333333")
        ax.set_xticks(range(len(unique_sc)))
        ax.set_xticklabels(
            [SCANNER_LABELS[i] for i in range(len(unique_sc))],
            fontsize=8,
        )
        feat_clean = feat.replace("original_", "").replace("_", " ")
        ax.set_title(
            f"{feat_clean}\n({phase})",
            fontsize=8, fontweight="normal",
        )
        ax.set_ylabel("Standardised value", fontsize=8)
        ax.grid(axis="y", alpha=0.25, linestyle=":")
        # Levene p annotation
        p_val = (df_levene.loc[df_levene["Feature"]==feat,
                               "Levene_p_pre" if phase=="Pre-ComBat"
                               else "Levene_p_post"].values[0])
        p_str = f"Levene p = {p_val:.4f}" if p_val >= 0.001 else "Levene p < 0.001"
        ax.text(0.98, 0.98, p_str, transform=ax.transAxes,
                ha="right", va="top", fontsize=7, color="#555555",
                fontstyle="italic")

save_figure(fig, "combat_diagnostics", SHA_DIR)


# ------------------------------------------------------------------
# Alias for Cell 15 (snapshot) compatibility
# Cell 15 checks for sil_before/sil_after; Cell 13l uses sil_pre/sil_post.
# These two lines expose both names in the kernel namespace.
# ------------------------------------------------------------------
sil_before = sil_pre
sil_after  = sil_post
print("\nCell 13l -- ComBat diagnostics complete.")
print(f"  Silhouette pre={sil_pre:.4f} -> post={sil_post:.4f}  (delta={delta:.4f})")


In [ ]:
# =============================================================================
# Cell 18 -- RadScore equations
# TRIPOD-AI items: 10c, 15
# =============================================================================
# Extracts LASSO-selected features and derives the RadScore linear equation
# for both outcomes in two forms:
#
#   (A) Standardised-space: RadScore = b0 + sum(bi * zi)
#       where zi = (xi - mean_i) / sd_i  (StandardScaler)
#
#   (B) Raw-space: RadScore = c0 + sum(ci_raw * xi)
#       ci_raw = bi / sd_i;  c0 = b0 - sum(bi * mean_i / sd_i)
#       Verified: max|A - B| < 1e-10 on training data.
#
# NOTE: for Outcome 2, X_o2_rad and rad_cols_o2 may be constrained
#       after the EPV fix in Cell 6. feature_names is resolved at runtime.
#
# Outputs:
#   outputs/tables/radscore_scaler_params_outcome{1,2}.csv
#   outputs/tables/radscore_raw_equation_outcome{1,2}.csv
#   Pipeline variables: eq_o1, eq_o2  (required by Cell 21 / ST1)
# TRIPOD-AI items: 10c (interpretability), 15 (reproducibility)
# =============================================================================

# --- Execution guard ---
if "Cs_o1_rad" not in dir():
    raise RuntimeError(
        "Cell 8 has not been executed.\n"
        "Please run Cells 0-9 before this cell."
    )


# =============================================================================
# HELPER -- extract RadScore equation
# =============================================================================
def extract_radscore_equation(X_rad, y, best_Cs, outcome_label,
                               fname_stem, feature_names=None):
    """Fit full-data LASSO LR and extract RadScore equation (std + raw space).

    Parameters
    ----------
    X_rad         : np.ndarray (n, p)
    y             : np.ndarray (n,)
    best_Cs       : list[float], per-fold best C from Cell 6
    outcome_label : str
    fname_stem    : str, e.g. "outcome1"
    feature_names : list[str] or None
        Names for the p columns of X_rad.
        If None: inferred from rad_cols / rad_cols_o2 by shape match.
        Pass explicitly when X_rad is a constrained subset.

    Returns
    -------
    dict with keys: outcome_label, fname_stem, n_selected, feat_names,
                    coefs_std, intercept_std, coefs_raw, intercept_raw,
                    scaler_means, scaler_stds, best_C
    """
    best_C = float(np.median(best_Cs))
    print(f"  Best C (median outer folds): {best_C:.5f}")

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l1", C=best_C, solver="liblinear",
            class_weight="balanced", max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])
    pipe.fit(X_rad, y)

    scaler    = pipe.named_steps["scaler"]
    clf       = pipe.named_steps["clf"]
    coefs     = clf.coef_[0]
    intercept = float(clf.intercept_[0])

    # -- resolve feature names -------------------------------------------------
    n_cols = X_rad.shape[1]
    if feature_names is not None:
        if len(feature_names) != n_cols:
            raise ValueError(
                f"feature_names length ({len(feature_names)}) "
                f"!= X_rad columns ({n_cols})."
            )
        feat_names = np.array(feature_names)
    elif n_cols == len(rad_cols):
        feat_names = np.array(rad_cols)
    elif "rad_cols_o2" in dir() and n_cols == len(rad_cols_o2):
        feat_names = np.array(rad_cols_o2)
        print(f"  [INFO] Using constrained rad_cols_o2 ({n_cols} features).")
    else:
        feat_names = np.array([f"feature_{i}" for i in range(n_cols)])
        print("  [WARN] Feature names not matched -- using synthetic names.")

    sel_mask = coefs != 0
    n_sel    = int(sel_mask.sum())
    print(f"  LASSO-selected: {n_sel} / {n_cols} features")

    sel_names = feat_names[sel_mask]
    sel_coefs = coefs[sel_mask]

    means         = scaler.mean_
    stds          = scaler.scale_
    coefs_raw     = coefs / stds
    intercept_raw = intercept - float(np.sum(coefs * means / stds))
    sel_coefs_raw = coefs_raw[sel_mask]

    X_std     = scaler.transform(X_rad)
    score_std = intercept + X_std @ coefs
    score_raw = intercept_raw + X_rad @ coefs_raw
    max_diff  = float(np.abs(score_std - score_raw).max())
    print(f"  Max |std - raw| difference: {max_diff:.2e}  [verified]")

    pd.DataFrame({
        "Feature":     feat_names[sel_mask],
        "Scaler_Mean": means[sel_mask],
        "Scaler_SD":   stds[sel_mask],
    }).to_csv(CSV_DIR / f"radscore_scaler_params_{fname_stem}.csv", index=False)
    print(f"  [OK] Scaler params -> radscore_scaler_params_{fname_stem}.csv")

    pd.DataFrame({
        "Feature":       sel_names,
        "Coef_raw":      sel_coefs_raw,
        "Intercept_raw": [intercept_raw] + [np.nan] * (n_sel - 1),
    }).to_csv(CSV_DIR / f"radscore_raw_equation_{fname_stem}.csv", index=False)
    print(f"  [OK] Raw equation  -> radscore_raw_equation_{fname_stem}.csv")

    return {
        "outcome_label": outcome_label,
        "fname_stem":    fname_stem,
        "n_selected":    n_sel,
        "feat_names":    sel_names,
        "coefs_std":     sel_coefs,
        "intercept_std": intercept,
        "coefs_raw":     sel_coefs_raw,
        "intercept_raw": intercept_raw,
        "scaler_means":  means[sel_mask],
        "scaler_stds":   stds[sel_mask],
        "best_C":        best_C,
    }


# =============================================================================
# HELPER -- inline HTML display
# =============================================================================
def display_radscore_html(eq):
    """Display inline HTML summary of selected features."""
    th   = ("padding:5px 12px; text-align:center; font-weight:bold; "
            "border-top:2px solid #111; border-bottom:1px solid #111; "
            "background:#f7f7f7; color:#111;")
    th_l = th.replace("center", "left")
    td   = "padding:5px 12px; text-align:center; color:#333;"
    td_l = td.replace("center", "left")

    order       = np.argsort(np.abs(eq["coefs_std"]))[::-1]
    feat_sorted = eq["feat_names"][order]
    coef_std    = eq["coefs_std"][order]
    coef_raw    = eq["coefs_raw"][order]
    feat_clean  = [f.replace("original_", "") for f in feat_sorted]

    rows_html = ""
    for i, (fn, cs, cr) in enumerate(zip(feat_clean, coef_std, coef_raw), 1):
        col     = "#CC3311" if cs > 0 else "#0072B2"
        dir_lbl = "Positive" if cs > 0 else "Negative"
        rows_html += (
            f"<tr style='border-bottom:0.5px solid #e8e8e8;'>"
            f"<td style='{td}'>{i}</td>"
            f"<td style='{td_l}'><code>{fn}</code></td>"
            f"<td style='{td} color:{col};'>{dir_lbl}</td>"
            f"<td style='{td}'>{cs:+.4e}</td>"
            f"<td style='{td}'>{cr:+.4e}</td></tr>"
        )

    html = (
        f"<div style='font-family:Arial,sans-serif; font-size:12px;"
        f" max-width:1000px; margin-top:16px; background:white;"
        f" padding:16px; border-radius:5px; color:#111;'>"
        f"<p style='font-size:14px; margin:0 0 10px 0;'>"
        f"<b>RadScore -- {eq['outcome_label']}</b>"
        f" <span style='color:#666; font-size:12px;'>"
        f"({eq['n_selected']} feature(s), C={eq['best_C']:.5f})</span></p>"
        f"<table style='border-collapse:collapse; width:100%;"
        f" margin-bottom:12px; background:white;'>"
        f"<thead><tr>"
        f"<th style='{th}'>Rank</th>"
        f"<th style='{th_l}'>Feature</th>"
        f"<th style='{th}'>Direction</th>"
        f"<th style='{th}'>Coef. (std)</th>"
        f"<th style='{th}'>Coef. (raw)</th>"
        f"</tr></thead><tbody>{rows_html}</tbody></table>"
        f"<p style='color:#777; font-size:10px; margin-top:6px;'>"
        f"z_i = (x_i - mean_i) / SD_i | "
        f"Scaler params: radscore_scaler_params_{eq['fname_stem']}.csv</p>"
        f"</div>"
    )
    ipy_display(HTML(html))


# =============================================================================
# RUN -- Outcome 1  (always full rad_cols)
# =============================================================================
print("=" * 60)
print("RadScore -- Outcome 1: Mild vs No HE")
print("=" * 60)
eq_o1 = extract_radscore_equation(
    X_rad=X_o1_rad, y=y_o1, best_Cs=Cs_o1_rad,
    outcome_label="Outcome 1 -- Mild vs No HE",
    fname_stem="outcome1",
    feature_names=rad_cols,
)
display_radscore_html(eq_o1)


# =============================================================================
# RUN -- Outcome 2  (constrained rad_cols_o2 if EPV fix applied in Cell 6)
# =============================================================================
print("\n" + "=" * 60)
print("RadScore -- Outcome 2: Moderate-Severe vs Mild HE")
print("=" * 60)

_feat_o2 = (
    rad_cols_o2
    if "rad_cols_o2" in dir() and len(rad_cols_o2) == X_o2_rad.shape[1]
    else rad_cols
)
eq_o2 = extract_radscore_equation(
    X_rad=X_o2_rad, y=y_o2, best_Cs=Cs_o2_rad,
    outcome_label="Outcome 2 -- Moderate-Severe vs Mild HE",
    fname_stem="outcome2",
    feature_names=_feat_o2,
)
display_radscore_html(eq_o2)


# =============================================================================
# SUMMARY
# =============================================================================
print("\nCell 20 -- RadScore equations complete.")
for fname in [
    "radscore_scaler_params_outcome1.csv",
    "radscore_scaler_params_outcome2.csv",
    "radscore_raw_equation_outcome1.csv",
    "radscore_raw_equation_outcome2.csv",
]:
    status = "[OK]" if (TAB_DIR / fname).exists() else "[MISS]"
    print(f"  {status} {fname}")


In [ ]:
# =============================================================================
# Cell 19 -- Supplementary Table ST1: RadScore features and coefficients
# TRIPOD-AI items: 15
# =============================================================================
# Generates SDC_ST1_RadScore_features.docx with:
#   ST1a -- Outcome 1 (Mild HE vs No HE): intercept + LASSO features + equation
#   ST1b -- Outcome 2 (Mod-Sev HE vs Mild HE): same structure
#   Notes on z-score standardisation and probability conversion
#   Abbreviations paragraph
#
# Uses python-docx directly -- no Node.js required.
# TRIPOD-AI item 15: model coefficients fully documented.
# =============================================================================

# --- Execution guard ---
if "eq_o1" not in dir():
    raise RuntimeError(
        "Cell 18 has not been executed in this session.\n"
        "Please run Cells 0-18 before this cell."
    )

from docx                import Document
from docx.oxml           import OxmlElement
from docx.oxml.ns        import qn
from docx.shared         import Pt, Cm, RGBColor
from docx.enum.text      import WD_ALIGN_PARAGRAPH
from docx.enum.section   import WD_ORIENT

FONT  = "Times New Roman"
SZ_B  = Pt(10)   # body
SZ_T  = Pt(10)   # title
SZ_F  = Pt(9)    # footnote
SZ_EQ = Pt(11)   # equation paragraph


# =============================================================================
# HELPERS
# =============================================================================
def _border_el(tag, val, sz, color):
    el = OxmlElement(tag)
    el.set(qn("w:val"),   val)
    el.set(qn("w:sz"),    sz)
    el.set(qn("w:space"), "0")
    el.set(qn("w:color"), color)
    return el

def _set_borders(cell, top=None, bottom=None):
    tc_pr   = cell._tc.get_or_add_tcPr()
    borders = OxmlElement("w:tcBorders")
    for side in ("top", "left", "bottom", "right", "insideH", "insideV"):
        if side == "top" and top:
            borders.append(_border_el("w:top",    top[0],    top[1],    top[2]))
        elif side == "bottom" and bottom:
            borders.append(_border_el("w:bottom", bottom[0], bottom[1], bottom[2]))
        else:
            borders.append(_border_el(f"w:{side}", "none", "0", "FFFFFF"))
    tc_pr.append(borders)

def _set_shading(cell):
    tc_pr = cell._tc.get_or_add_tcPr()
    shd   = OxmlElement("w:shd")
    shd.set(qn("w:val"),   "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"),  "FFFFFF")
    tc_pr.append(shd)

THICK = ("single", "12", "000000")
THIN  = ("single",  "4", "000000")


def _add_run(para, text, bold=False, italic=False,
             sub=False, sup=False, size=None):
    """Add a run to a paragraph with optional subscript/superscript."""
    run = para.add_run(text)
    run.font.name   = FONT
    run.font.size   = size or SZ_B
    run.bold        = bold
    run.italic      = italic
    if sub or sup:
        rpr = run._r.get_or_add_rPr()
        vm  = OxmlElement("w:vertAlign")
        vm.set(qn("w:val"), "subscript" if sub else "superscript")
        rpr.append(vm)
    return run


def _fill_header_cell(cell, para_builder, top=THICK, bottom=THIN):
    para = cell.paragraphs[0]
    para.clear()
    para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    para.paragraph_format.space_before = Pt(1)
    para.paragraph_format.space_after  = Pt(1)
    para_builder(para)
    _set_shading(cell)
    _set_borders(cell, top=top, bottom=bottom)


def _fill_data_cell(cell, text, align="center",
                    italic=False, bottom=None):
    para = cell.paragraphs[0]
    para.clear()
    para.alignment = (WD_ALIGN_PARAGRAPH.LEFT
                      if align == "left"
                      else WD_ALIGN_PARAGRAPH.CENTER)
    para.paragraph_format.space_before = Pt(1)
    para.paragraph_format.space_after  = Pt(1)
    run = para.add_run(text)
    run.font.name   = FONT
    run.font.size   = SZ_B
    run.italic      = italic
    _set_shading(cell)
    _set_borders(cell, bottom=bottom)


def _build_st1_table(doc, intercept, features):
    """Build one ST1 coefficient table in doc."""
    col_widths_cm = [11.0, 4.5]
    table = doc.add_table(rows=1, cols=2)
    table.style = "Table Grid"

    # Header row: Feature (Xi) | Coefficient (βi)
    hdr = table.rows[0].cells

    def _hdr0(para):
        _add_run(para, "Feature (", bold=True)
        _add_run(para, "X", bold=True, italic=True)
        _add_run(para, "i", bold=True, italic=True, sub=True)
        _add_run(para, ")", bold=True)
    def _hdr1(para):
        _add_run(para, "Coefficient (", bold=True)
        _add_run(para, "\u03b2", bold=True, italic=True)
        _add_run(para, "i", bold=True, italic=True, sub=True)
        _add_run(para, ")", bold=True)

    _fill_header_cell(hdr[0], _hdr0)
    _fill_header_cell(hdr[1], _hdr1)

    # Intercept row
    row_int = table.add_row()
    _fill_data_cell(row_int.cells[0], "(Intercept)",
                    align="left", italic=True)
    _fill_data_cell(row_int.cells[1],
                    f"{intercept:+.4f}", align="center")

    # Feature rows
    n_feat = len(features)
    for idx, feat in enumerate(features):
        new_row = table.add_row()
        is_last = (idx == n_feat - 1)

        # Feature cell with subscript index
        para_f = new_row.cells[0].paragraphs[0]
        para_f.clear()
        para_f.alignment = WD_ALIGN_PARAGRAPH.LEFT
        para_f.paragraph_format.space_before = Pt(1)
        para_f.paragraph_format.space_after  = Pt(1)
        _add_run(para_f, feat["name"] + " (")
        _add_run(para_f, "X", italic=True)
        _add_run(para_f, str(idx + 1), italic=True, sub=True)
        _add_run(para_f, ")")
        _set_shading(new_row.cells[0])
        _set_borders(new_row.cells[0],
                     bottom=THICK if is_last else None)

        # Coefficient cell
        _fill_data_cell(
            new_row.cells[1],
            f"{feat['coef']:+.4f}",
            align="center",
            bottom=THICK if is_last else None,
        )

    # Column widths
    for row in table.rows:
        for ci, cell in enumerate(row.cells):
            cell.width = Cm(col_widths_cm[ci])

    return table


def _add_equation_para(doc, subscript):
    """Add RadScore equation paragraph."""
    para = doc.add_paragraph()
    para.paragraph_format.space_before = Pt(6)
    para.paragraph_format.space_after  = Pt(6)
    _add_run(para, "RadScore", size=SZ_EQ)
    _add_run(para, subscript, size=SZ_EQ, sub=True)
    _add_run(para, " = Intercept + \u03a3(\u03b2", size=SZ_EQ)
    _add_run(para, "i", size=SZ_EQ, sub=True)
    _add_run(para, " \u00b7 X", size=SZ_EQ)
    _add_run(para, "i", size=SZ_EQ, sub=True)
    _add_run(para, ")", size=SZ_EQ)


# =============================================================================
# BUILD DOCUMENT
# =============================================================================
def _outcome_data(eq):
    order = np.argsort(np.abs(eq["coefs_std"]))[::-1]
    return {
        "intercept": float(eq["intercept_std"]),
        "features": [
            {"name": str(eq["feat_names"][i]),
             "coef": float(eq["coefs_std"][i])}
            for i in order
        ],
    }

data_o1 = _outcome_data(eq_o1)
data_o2 = _outcome_data(eq_o2)

doc = Document()
section = doc.sections[0]
section.page_width  = Cm(21)
section.page_height = Cm(29.7)
for m in ("top_margin", "bottom_margin",
          "left_margin", "right_margin"):
    setattr(section, m, Cm(2.5))

# Title
p_title = doc.add_paragraph()
p_title.paragraph_format.space_before = Pt(0)
p_title.paragraph_format.space_after  = Pt(8)
r_title = p_title.add_run(
    "Supplementary Table ST1. "
    "Radiomic features and model coefficients for RadScore calculation."
)
r_title.font.name = FONT
r_title.font.size = SZ_T
r_title.bold      = True

# --- ST1a ---
p_a = doc.add_paragraph()
p_a.paragraph_format.space_before = Pt(0)
p_a.paragraph_format.space_after  = Pt(4)
r_a = p_a.add_run("ST1a. Outcome 1: Mild HE vs No HE")
r_a.font.name = FONT; r_a.font.size = SZ_B; r_a.bold = True

_build_st1_table(doc, data_o1["intercept"], data_o1["features"])
_add_equation_para(doc, "Mild")

doc.add_paragraph()  # spacer

# --- ST1b ---
p_b = doc.add_paragraph()
p_b.paragraph_format.space_before = Pt(0)
p_b.paragraph_format.space_after  = Pt(4)
r_b = p_b.add_run("ST1b. Outcome 2: Moderate-to-Severe HE vs Mild HE")
r_b.font.name = FONT; r_b.font.size = SZ_B; r_b.bold = True

_build_st1_table(doc, data_o2["intercept"], data_o2["features"])
_add_equation_para(doc, "Mod-Sev")

doc.add_paragraph()  # spacer

# --- Notes ---
p_notes = doc.add_paragraph()
r_notes = p_notes.add_run("Notes:")
r_notes.font.name = FONT; r_notes.font.size = SZ_B; r_notes.bold = True

notes = [
    (
        "All radiomic features (Xi) must be z-score standardised "
        "(zero mean, unit variance) using the scaler parameters reported in "
        "the accompanying CSV files (radscore_scaler_params_outcome*.csv) "
        "before applying the coefficients."
    ),
    (
        "The final predicted probability (P) is: "
        "P = 1 / (1 + e\u207b\u207a\u1d35\u207f\u1d57\u1d49\u02b3\u1d04\u1d49\u1d56\u1d57"
        " + RadScore\u207e)."
    ),
    (
        f"Coefficients are from a LASSO-regularised (L1) logistic regression "
        f"fitted on the full internal cohort (n\u00a0=\u00a0{len(df)} patients) "
        f"using the median best-C from {CV_FOLDS}-fold nested cross-validation."
    ),
]
for i, note_text in enumerate(notes, 1):
    p_n = doc.add_paragraph()
    p_n.paragraph_format.space_before = Pt(2)
    p_n.paragraph_format.space_after  = Pt(2)
    r_n = p_n.add_run(f"{i}. {note_text}")
    r_n.font.name = FONT; r_n.font.size = SZ_B

doc.add_paragraph()  # spacer

# --- Abbreviations ---
all_names = (
    [f["name"] for f in data_o1["features"]] +
    [f["name"] for f in data_o2["features"]]
)
abbrev_map = {
    "glszm": "GLSZM, gray level size zone matrix",
    "glrlm": "GLRLM, gray level run length matrix",
    "gldm":  "GLDM, gray level dependence matrix",
    "glcm":  "GLCM, gray level co-occurrence matrix",
}
used_abbrevs = [v for k, v in abbrev_map.items()
                if any(k in n.lower() for n in all_names)]

p_abbr = doc.add_paragraph()
p_abbr.paragraph_format.space_before = Pt(4)
p_abbr.paragraph_format.space_after  = Pt(0)
r_abbr_bold = p_abbr.add_run("Abbreviations: ")
r_abbr_bold.font.name = FONT
r_abbr_bold.font.size = SZ_F
r_abbr_bold.bold      = True
abbr_text = (
    "HE, hepatic encephalopathy; "
    + "; ".join(used_abbrevs)
    + ("; " if used_abbrevs else "")
    + "LASSO, least absolute shrinkage and selection operator."
)
r_abbr = p_abbr.add_run(abbr_text)
r_abbr.font.name = FONT
r_abbr.font.size = SZ_F
r_abbr.italic    = True

# [Table displayed inline above — no DOCX output]


# =============================================================================
# INLINE PREVIEW
# =============================================================================
th   = ("padding:5px 12px; text-align:center; font-weight:bold; "
        "border-top:2px solid #111; border-bottom:1px solid #111; "
        "background:#f7f7f7;")
th_l = th.replace("center", "left")
td   = "padding:4px 12px; text-align:center; color:#333;"
td_l = td.replace("center", "left")

html = (
    "<div style=\"font-family:'Helvetica Neue',Arial,sans-serif;"
    "font-size:12px; max-width:760px; margin-top:10px;"
    "background:white; padding:15px; border-radius:5px;\">"
    "<p style='font-size:13px; margin:0 0 10px 0;'>"
    "<b>Supplementary Table ST1 \u2014 RadScore coefficients</b></p>"
)

for section_label, odata in [
    ("ST1a. Outcome 1: Mild HE vs No HE",            data_o1),
    ("ST1b. Outcome 2: Mod-Sev HE vs Mild HE",       data_o2),
]:
    sub = "Mild" if "1" in section_label else "Mod-Sev"
    html += (
        f"<p style='margin:12px 0 4px 0;'><b>{section_label}</b></p>"
        f"<table style='border-collapse:collapse; width:100%;"
        f"margin-bottom:6px;'>"
        f"<thead><tr>"
        f"<th style='{th_l}'>Feature (X<sub>i</sub>)</th>"
        f"<th style='{th}'>\u03b2<sub>i</sub></th>"
        f"</tr></thead><tbody>"
        f"<tr><td style='{td_l}'><i>(Intercept)</i></td>"
        f"<td style='{td}'>{odata['intercept']:+.4f}</td></tr>"
    )
    for i, feat in enumerate(odata["features"], 1):
        col = "#CC3311" if feat["coef"] > 0 else "#0072B2"
        html += (
            f"<tr style='border-bottom:0.5px solid #ddd;'>"
            f"<td style='{td_l}'>"
            f"<code style='font-size:10px;'>{feat['name']}</code>"
            f" (X<sub>{i}</sub>)</td>"
            f"<td style='{td} color:{col};'>{feat['coef']:+.4f}</td>"
            f"</tr>"
        )
    html += (
        f"</tbody></table>"
        f"<p style='font-size:11px; color:#555; margin:2px 0 0 8px;'>"
        f"RadScore<sub>{sub}</sub> = Intercept + "
        f"\u03a3(\u03b2<sub>i</sub> \u00b7 X<sub>i</sub>)</p>"
    )

html += "</div>"
display(HTML(html))

print("Cell 19 — ST1 complete.")

In [ ]:
# =============================================================================
# Cell 21 -- Repository support files: requirements.txt, config.yaml, README.md
# TRIPOD-AI items: 15
# =============================================================================
# Generates three files required for open-science compliance.
# Run after Cell 20 (reproducibility check).
#
# NOTE: config.yaml is a documentation artefact for reviewers and
# collaborators -- it is NOT read programmatically by the pipeline.
# All parameters are defined directly in Cell 1.
# =============================================================================

if "RANDOM_STATE" not in dir():
    raise RuntimeError(
        "Cell 1 has not been executed in this session.\n"
        "Please run Cells 0-1 before this cell."
    )

import importlib
import sys
from datetime import datetime

SEP = "=" * 65

# =============================================================================
# STEP 1 -- requirements.txt
# =============================================================================
print(SEP)
print("STEP 1 -- requirements.txt")
print(SEP)

PKG_MAP = {
    "numpy":       "numpy",
    "pandas":      "pandas",
    "sklearn":     "scikit-learn",
    "scipy":       "scipy",
    "matplotlib":  "matplotlib",
    "shap":        "shap",
    "pycombat":    "pycombat",
    "docx":        "python-docx",
    "PIL":         "Pillow",
    "statsmodels": "statsmodels",
}

req_lines = [
    "# requirements.txt",
    f"# Auto-generated by Cell 21 on {datetime.now().strftime('%Y-%m-%d')}",
    f"# Python {sys.version.split()[0]}",
    "",
    "# Pinned scientific stack",
]
for imp, pip in PKG_MAP.items():
    try:
        mod = importlib.import_module(imp)
        ver = getattr(mod, "__version__", None)
        line = f"{pip}=={ver}" if ver else f"{pip}  # version unknown"
    except ImportError:
        line = f"# {pip}  # not installed"
    req_lines.append(line)
    print(f"  {line}")

req_path = ROOT / "requirements.txt"
req_path.write_text("\n".join(req_lines) + "\n", encoding="utf-8")
print(f"\n  [OK] {req_path.name}")


# =============================================================================
# STEP 2 -- config.yaml
# =============================================================================
print(f"\n{SEP}")
print("STEP 2 -- config.yaml")
print(SEP)

def _yaml(d, indent=0):
    """Recursively emit a nested dict as YAML lines (no external dep)."""
    lines, pad = [], "  " * indent
    for k, v in d.items():
        if isinstance(v, dict):
            lines.append(f"{pad}{k}:")
            lines.extend(_yaml(v, indent + 1))
        elif isinstance(v, list):
            lines.append(f"{pad}{k}:")
            for item in v:
                lines.append(f"{pad}  - {item}")
        elif isinstance(v, bool):
            lines.append(f"{pad}{k}: {'true' if v else 'false'}")
        elif isinstance(v, str):
            lines.append(f'{pad}{k}: "{v}"')
        else:
            lines.append(f"{pad}{k}: {v}")
    return lines

cfg = {
    "pipeline": {
        "name":      "Globus Pallidus T1-Weighted MRI Radiomics for HE Grading",
        "version":   "1.0",
        "generated": datetime.now().strftime("%Y-%m-%d"),
        "python":    sys.version.split()[0],
        "github":    "https://github.com/radiomiclab/radiomic_HE_pipeline",
        "zenodo":    "https://doi.org/10.5281/zenodo.[ID]",
        "citation":  "[Authors]. [Title]. [Journal]. [Year]. DOI: [TBD]",
    },
    "reproducibility": {
        "random_state":   int(RANDOM_STATE),
        "boot_seed":      int(BOOT_SEED),
        "pythonhashseed": int(RANDOM_STATE),
        "n_jobs":         1,
    },
    "cross_validation": {
        "cv_folds": int(CV_FOLDS),
        "n_boot":   int(N_BOOT),
        "n_perm":   int(N_PERM),
        "c_grid_log10_start": -3,
        "c_grid_log10_stop":   0,
        "c_grid_n_steps":     10,
    },
    "features": {
        "n_radiomic":      int(len(rad_cols)),
        "ibsi_prefix":     "original_",
        "pyradiomics_ver": "3.0.1",
        "bin_width_hu":    25,
        "extraction_3d":   True,
        "roi":             "globus pallidus bilateral",
        "sequence":        "T1-weighted unenhanced",
        "feature_selection_rationale": (
            "30 IBSI-compliant features selected from first-order statistics "
            "and four texture matrix families (GLCM, GLDM, GLRLM, GLSZM). "
            "Wavelet and shape features excluded for reproducibility. "
            "All features carry prefix original_ (unfiltered image). "
            "Reproducible across PyRadiomics >= 3.0 and equivalent "
            "IBSI-compliant software."
        ),
    },
    "scanners": {
        "scanner_a": str(SCANNER_A),
        "scanner_b": str(SCANNER_B),
        "field_a":   "1.5T",
        "field_b":   "3.0T",
    },
    "combat": {
        "package":             "pycombat==0.20",
        "covariate_o1_o2":     "MELD",
        "covariate_o0":        "age (z-scored)",
        "applied_inside_cv":   True,
        "reference_based_ext": True,
    },
    "outcomes": {
        "O0": {
            "label":        "Cirrhosis No HE vs Healthy Controls",
            "whc_positive": "cirrhosis (any WHC)",
            "whc_negative": "healthy controls",
            "models":       ["Radiomics"],
            "role":         "biological_signal_specificity",
            "n":            148,
        },
        "O1": {
            "label":        "Mild HE (WHC 1) vs No overt HE (WHC 0)",
            "whc_positive": 1,
            "whc_negative": 0,
            "models":       ["MELD", "Radiomics", "Combined"],
            "role":         "primary",
            "n":            123,
        },
        "O2": {
            "label":        "Moderate-Severe HE (WHC >= 2) vs Mild HE (WHC 1)",
            "whc_positive": ">=2",
            "whc_negative": 1,
            "models":       ["MELD", "Radiomics", "Combined"],
            "role":         "exploratory",
            "epv_note":     "EPV < 10; zero bootstrap feature stability",
            "n":            85,
        },
    },
    "label_conversion": {
        "whc_0": {"description": "No overt HE",        "O1": 0, "O2": "excluded"},
        "whc_1": {"description": "Mild (covert) HE",   "O1": 1, "O2": 0},
        "whc_2": {"description": "Moderate HE",        "O1": "excluded", "O2": 1},
        "whc_3": {"description": "Severe HE",          "O1": "excluded", "O2": 1},
    },
    "epv": {
        "o2_exploratory":      True,
        "stability_threshold": 0.80,
        "stability_n_boot":    200,
        "epv_target":          9.0,
    },
    "paths": {
        "data_dir":            "data/",
        "output_dir":          "outputs/",
        "tables_dir":          "outputs/tables/",
        "csv_dir":             "outputs/csv/",
        "raw_csv":             "data/radiomic_HE_dataset.csv",
        "healthy_signa_csv":   "data/healthy_signa.csv",
        "healthy_disc_csv":    "data/healthy_discovery.csv",
        "ext_a_csv":           "data/radiomic_HE_dataset_extA.csv",
        "ext_b_csv":           "data/radiomic_HE_dataset_extB.csv",
    },
}

yaml_lines = [
    "# config.yaml -- pipeline hyperparameters and paths",
    f"# Auto-generated by Cell 21 on {datetime.now().strftime('%Y-%m-%d')}",
    "#",
    "# This file is a DOCUMENTATION ARTEFACT for reviewers and collaborators.",
    "# It is NOT read programmatically by the pipeline.",
    "# All parameters are defined in Cell 1 of radiomic_pipeline_v1.ipynb.",
    "# If you change a parameter, edit Cell 1 and re-run this cell.",
    "#",
    "# Edit scanner names and paths to match your environment.",
    "# Update github/zenodo placeholders before public release.",
    "",
]
yaml_lines.extend(_yaml(cfg))

cfg_path = ROOT / "config.yaml"
cfg_path.write_text("\n".join(yaml_lines) + "\n", encoding="utf-8")
print(f"  [OK] {cfg_path.name}  ({len(yaml_lines)} lines)")


# =============================================================================
# STEP 3 -- README.md
# =============================================================================
print(f"\n{SEP}")
print("STEP 3 -- README.md")
print(SEP)

n_o0   = len(df_o0)       if "df_o0"       in dir() else 148
n_o1   = len(df_o1)       if "df_o1"       in dir() else 123
n_o2   = len(df_o2)       if "df_o2"       in dir() else 85
n_int  = len(df)           if "df"           in dir() else 168
n_ctrl = len(df_controls)  if "df_controls"  in dir() else 65

readme = [
    "# Globus Pallidus T1-Weighted MRI Radiomics for HE Grading",
    "",
    "[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.[ID].svg)]"
    "(https://doi.org/10.5281/zenodo.[ID])",
    "",
    "> **Status**: manuscript under review  ",
    "> **Pipeline version**: 1.0  ",
    f"> **Generated**: {datetime.now().strftime('%Y-%m-%d')}",
    "",
    "---",
    "",
    "## Citation",
    "",
    "If you use this pipeline, please cite:",
    "",
    "> [Authors]. *[Title]*. *[Journal]*, [Year]. DOI: [TBD upon acceptance]",
    "",
    "---",
    "",
    "## Overview",
    "",
    "Fully reproducible machine learning pipeline for non-invasive stratification",
    "of hepatic encephalopathy (HE) severity using T1-weighted MRI radiomic",
    "features of the globus pallidus in cirrhotic patients.",
    "",
    "The globus pallidus undergoes structural changes in cirrhosis driven by a",
    "combination of pathological processes: primarily manganese deposition via",
    "porto-systemic shunting, but also degenerative phenomena including astrocytic",
    "oedema and gliosis. This pipeline quantifies whether IBSI-compliant texture",
    "features of this composite signal discriminate HE severity grades.",
    "",
    "---",
    "",
    "## Study Design",
    "",
    "### Outcome definitions",
    "",
    "| Outcome | Positive class | Negative class | WHC grades | N | Role |",
    "|---------|---------------|----------------|------------|---|------|",
    f"| **O0** | Cirrhosis No HE | Healthy Controls | — | {n_o0} | Biological signal specificity |",
    f"| **O1** | Mild HE | No overt HE | WHC 1 vs WHC 0 | {n_o1} | **Primary** |",
    f"| **O2** | Moderate-to-Severe HE | Mild HE | WHC ≥ 2 vs WHC 1 | {n_o2} | Exploratory (EPV < 10) |",
    "",
    "### Label conversion (WHC → binary)",
    "",
    "| WHC grade | Clinical description | O1 label | O2 label |",
    "|-----------|---------------------|----------|----------|",
    "| 0 | No overt HE | 0 (negative) | — (excluded) |",
    "| 1 | Mild (covert) HE | 1 (positive) | 0 (negative) |",
    "| 2 | Moderate HE | — (excluded) | 1 (positive) |",
    "| 3 | Severe HE | — (excluded) | 1 (positive) |",
    "",
    "**O0** establishes that radiomic differences reflect the structural changes",
    "of the globus pallidus in cirrhosis (manganese deposition, oedema, gliosis)",
    "rather than HE-specific neurological changes per se.",
    "",
    "**O1** is the primary outcome: detection of mild (covert) HE using globus",
    "pallidus radiomics as a non-invasive pre-transplant stratification tool.",
    "",
    "**O2** is exploratory due to EPV < 10 and zero bootstrap feature stability;",
    "results are hypothesis-generating only.",
    "",
    "---",
    "",
    "## Important Note on Feature Extraction and Segmentation",
    "",
    "**Radiomic feature extraction and ROI segmentation are not part of this",
    "pipeline.** Each research group uses its own MRI acquisition protocol,",
    "segmentation method, and extraction software. This pipeline assumes that:",
    "",
    "1. Bilateral globus pallidus ROIs have been segmented on T1-weighted",
    "   unenhanced MRI images following the operator's own validated procedure.",
    "2. The same **30 IBSI-compliant radiomic features** used in the original",
    "   study have been extracted from those ROIs using an IBSI-compliant tool",
    "   (e.g., PyRadiomics v3.0.1, bin-width 25 HU, 3D extraction, `original_`",
    "   prefix).",
    "3. The extracted features are stored as CSV files with the exact column",
    "   names documented in `dataset_dictionary.ipynb`.",
    "",
    "**Using a different feature set or extraction configuration will invalidate",
    "the pre-trained model coefficients and the reproducibility of results.**",
    "",
    "---",
    "",
    "## Repository Structure",
    "",
    "```",
    ".",
    "├── radiomic_pipeline_v1.ipynb       # Main analysis pipeline (Cells 0-21)",
    "├── dataset_dictionary.ipynb         # Column schema, IBSI codes, CSV templates",
    "├── requirements.txt",
    "├── config.yaml",
    "├── README.md",
    "├── LICENSE",
    "├── data/",
    f"│   ├── radiomic_HE_dataset.csv         # Internal cohort (n={n_int})",
    "│   ├── healthy_signa.csv               # Healthy controls — Signa 1.5T",
    "│   ├── healthy_discovery.csv           # Healthy controls — Discovery 3.0T",
    "│   ├── radiomic_HE_dataset_extA.csv    # External validation cohort A",
    "│   └── radiomic_HE_dataset_extB.csv    # External validation cohort B",
    "└── outputs/",
    "    ├── figures/                         # TIFF + PDF (colour + grayscale)",
    "    ├── tables/                          # DOCX submission tables",
    "    ├── csv/                             # All CSV data exports",
    "    └── manuscript_snapshot.json",
    "```",
    "",
    "---",
    "",
    "## Requirements",
    "",
    "```bash",
    f"python >= 3.9   # tested on {sys.version.split()[0]}",
    "pip install -r requirements.txt",
    "```",
    "",
    "---",
    "",
    "## Usage",
    "",
    "### Step 1 — Create empty dataset templates",
    "",
    "Open and run `dataset_dictionary.ipynb`. This creates the `./data/`",
    "subdirectory and generates five empty CSV templates with the correct",
    "39-column header.",
    "",
    "### Step 2 — Populate with your own data",
    "",
    "Fill in each CSV following the column schema in `dataset_dictionary.ipynb`.",
    "Run the validation cell to verify the schema before proceeding.",
    "",
    "### Step 3 — Run the pipeline",
    "",
    "```bash",
    "pip install -r requirements.txt   # once",
    "jupyter notebook radiomic_pipeline_v1.ipynb",
    "```",
    "",
    "Execute all cells sequentially (Cell 0 → Cell 21). All figures and",
    "statistical tables are displayed inline in the notebook.",
    "Cell 21 generates `requirements.txt`, `config.yaml`, and this `README.md`.",
    "",
    "> **Note on `config.yaml`:** this file is a documentation artefact for",
    "> reviewers and collaborators — it is **not** read programmatically by the",
    "> pipeline. All parameters are defined directly in Cell 1 of the pipeline",
    "> notebook. If you need to change a parameter (e.g., scanner name, seed,",
    "> number of bootstrap resamples), edit Cell 1; then re-run Cell 21 to",
    "> regenerate a consistent `config.yaml`.",
    "",
    "---",
    "",
    "## Cell Map",
    "",
    "| Cell | Content |",
    "|------|---------|",
    "| 0 | Environment setup |",
    "| 1 | Imports, seeds, constants, helpers |",
    "| 2 | Table 1: baseline characteristics |",
    "| 3 | Data loading: internal cohort |",
    "| 4 | Data loading: healthy controls (O0) |",
    "| 5 | EDA: cohort characterisation + age–feature correlation |",
    "| 6 | Subset creation O0, O1, O2 |",
    "| 7 | Nested CV + ComBat + EPV fix (all outcomes) |",
    "| 8 | Performance metrics + DeLong + calibration |",
    "| 9 | Calibration reliability diagrams |",
    "| 10 | Decision Curve Analysis — Fig 4 |",
    "| 11 | SHAP feature importance — Fig 3 |",
    "| 12 | Bootstrap LASSO stability |",
    "| 13 | AUC grouped bar chart — Fig 2 |",
    "| 14 | Cross-system validation |",
    "| 15 | External validation (Ext-A, Ext-B) |",
    "| 16 | Domain shift analysis |",
    "| 17 | ComBat diagnostics |",
    "| 18 | RadScore equations |",
    "| 19 | Supplementary Table ST1 |",
    "| 20 | Reproducibility check + session info |",
    "| **21** | **Repository files (requirements.txt, config.yaml, README.md)** |",
    "",
    "---",
    "",
    "## Compliance",
    "",
    "| Standard | Implementation |",
    "|----------|----------------|",
    "| **TRIPOD-AI** (Collins et al., *BMJ* 2024) | Items documented per cell header |",
    "| **IBSI** (Zwanenburg et al., *Radiology* 2020) | PyRadiomics 3.0.1, prefix `original_`, bin-width 25 HU, 3D |",
    "| **PEP 8** | snake_case, 79-char lines, Google-style docstrings |",
    "| **Open science** | requirements.txt, config.yaml, README.md, snapshot JSON |",
    "",
    "---",
    "",
    "## Feature Selection Rationale",
    "",
    "30 IBSI-compliant features were selected from first-order statistics and",
    "four texture matrix families (GLCM, GLDM, GLRLM, GLSZM). Wavelet-derived",
    "and shape features were deliberately excluded: wavelet features depend on",
    "decomposition parameters not universally standardised across software, while",
    "shape features are not appropriate for a signal driven by paramagnetic",
    "deposition and degenerative tissue changes (oedema, gliosis). This",
    "conservative selection prioritises institutional transferability — any centre",
    "with PyRadiomics >= 3.0 or equivalent IBSI-compliant software can replicate",
    "extraction without custom configuration.",
    "",
    "---",
    "",
    "## Biological Signal Specificity (Outcome 0)",
    "",
    f"Outcome 0 (n={n_o0}) establishes that radiomic differences reflect the",
    "structural changes of the globus pallidus in cirrhosis (manganese deposition,",
    "oedema, gliosis), not HE-specific neurological changes per se.",
    "Age-feature correlation analysis in healthy controls confirmed that no",
    "features reached FDR significance (0/30 features FDR-significant); the",
    "three dominant O1 features showed no age correlation (all |r| <= 0.17,",
    "all p >= 0.13).",
    "",
    "---",
    "",
    "## Reproducibility",
    "",
    "| Parameter | Value |",
    "|-----------|-------|",
    f"| `RANDOM_STATE` | {RANDOM_STATE} |",
    f"| `BOOT_SEED` | {BOOT_SEED} |",
    f"| `PYTHONHASHSEED` | {RANDOM_STATE} |",
    f"| CV folds | {CV_FOLDS} |",
    f"| Bootstrap resamples | {N_BOOT} |",
    f"| Permutation iterations | {N_PERM} |",
    "",
    "---",
    "",
    "## License",
    "",
    "MIT License — see `LICENSE` for details.",
    "",
    "---",
    f"*Auto-generated by Cell 21 on {datetime.now().strftime('%Y-%m-%d %H:%M')}*",
]

readme_path = ROOT / "README.md"
readme_path.write_text("\n".join(readme) + "\n", encoding="utf-8")
print(f"  [OK] {readme_path.name}  ({len(readme)} lines)")


# =============================================================================
# SUMMARY
# =============================================================================
print(f"\n{SEP}")
print("Cell 21 -- Repository files complete.")
print(SEP)
for p in [req_path, cfg_path, readme_path]:
    kb = max(p.stat().st_size // 1024, 1)
    print(f"  [OK] {p.name:<20}  {kb} KB  ->  {p.resolve()}")
print()
print("Pre-release checklist:")
print("  [x] requirements.txt   -- pip install -r requirements.txt")
print("  [x] config.yaml        -- documentation artefact (not read by pipeline)")
print("  [x] README.md          -- update citation DOI when available")
print("  [ ] LICENSE            -- add MIT LICENSE file")
print("  [ ] .gitignore         -- exclude outputs/, data/, __pycache__/")
print("  [ ] Tag release        -- git tag v1.0 && git push --tags")
print("  [ ] Zenodo upload      -- upload zip, update DOI in config.yaml")
